# Verified Negatives and False Alarms in Fire and Smoke Detection

Reproduction notebook for *Verified Negatives and False Alarms in Fire and Smoke Detection: A Cost-Sensitive Re-Evaluation on the D-Fire Benchmark*, submitted to Pattern Analysis and Applications (Springer). Citation: [ISI: citation once published].

The notebook rebuilds every computed table and figure of the paper from three inputs:

1. the frozen train/validation/test manifests of the curated D-Fire split (shipped in `data/manifests/`);
2. the detections that the eight trained models stored on the validation and test splits (download, see `README.md`);
3. for Figure 7 only, the D-Fire images themselves.

No detector is run here. Each model wrote its detections down to a confidence of 0.05, well below any threshold a deployment would use, so precision, recall, the false-alarm rate and the expected cost can be recomputed at any operating point from the stored files. That is also the practical point of the paper: the quantities it asks authors to report cost about a minute of computation over predictions they already hold.

**Runtime.** About 2 to 4 minutes on a laptop CPU (2.4 minutes when this notebook was validated), most of it spent parsing about 660 MB of JSON lines and running the two bootstrap procedures. Peak memory stays below 1 GB.

**How the notebook is organised.** Sections 1 to 3 load the data, check its integrity and define the metrics. Sections 4 to 14 each reproduce one part of the Results. Section 15 compares every computed number with the value printed in the manuscript and with the output files of the original analysis, and ends with a PASS/FAIL summary. Two appendices contain the training code for both detectors; they are switched off by default and are not needed for any table.

| Paper item | Content | Notebook section |
|---|---|---|
| Table 1 | Literature survey | not computed (a reading of nine papers) |
| Table 2 | Audited dataset composition per split | 4 |
| Table 3 | Object scale by class | 4 |
| Table 4 | Training configuration | not computed; the same values drive Appendices A and B |
| Table 5 | The three training conditions of the ablation | 4 |
| Table 6 | Test results at the validation-selected threshold | 5 |
| Table 7 | Per-seed mAP50, false-alarm rate and threshold | 5 |
| Table 8 | False alarms at fixed recall, with paired intervals | 7 (point estimates), 9 (intervals) |
| Table 9 | Precision against the alarm rate | 5 |
| Table 10 | Expected cost under the per-class policy, with paired intervals | 8 (point estimates), 9 (intervals) |
| Table 11 | Bootstrap intervals for the alarm rate | 9 |
| Table 12 | Size of missed versus caught objects | 10 |
| Table 13 | Three-condition negative-image ablation | 13 |
| Table 14 | The double dissociation | 13 |
| Table 15 | Computational cost on the GPU | 14 (recorded measurements, cannot be recomputed on a CPU) |
| Figure 1 | Fire versus smoke AP, and object scale | 6 |
| Figure 2 | False alarms against recall | 7 |
| Figure 3 | Expected cost against the cost ratio | 8 |
| Figure 4 | Per-seed mAP50 and false-alarm rate | 5 |
| Figure 5 | Median size of missed and caught objects | 10 |
| Figure 6 | Confidence calibration | 11 |
| Figure 7 | Qualitative examples (needs the D-Fire images) | 12 |
| Figure 8 | The ablation: mAP50 and false-alarm rate | 13 |
| Equation 1 | False-alarm rate per 1,000 verified negatives | 3 |
| Equation 2 | Expected cost | 8 |

Tables are written to `outputs/tables/`; each figure is written twice, as `outputs/figures/FigN.png` for the screen and as a vector `outputs/figures/FigN.pdf` for print.

## 1. Configuration

All paths come from environment variables, with defaults relative to this notebook:

| Variable | Default | Meaning |
|---|---|---|
| `FIRE_DATA_DIR` | `data` | folder holding `manifests/`, `recorded/` and, by default, `predictions/` |
| `FIRE_PRED_DIR` | `<FIRE_DATA_DIR>/predictions` | stored detections, one folder per run (`dfire_stored_predictions.zip` unzipped inside `data/` gives `data/predictions/<run>/<split>_dfire.jsonl`, e.g. `data/predictions/tanpa_negatif_seed42/test_dfire.jsonl`) |
| `FIRE_DFIRE_IMAGES` | unset | root of the D-Fire release (the folder containing `train/images` and `test/images`); only Figure 7 uses it |
| `FIRE_OUTPUT_DIR` | `outputs` | where tables and figures are written |

The settings below are the locked values of the original analysis (`config/pengaturan.yaml` and `config/biaya.yaml` in the authors' project). Changing any of them changes the paper's numbers, so they are constants here rather than parameters.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_START = time.time()

# --------------------------------------------------------------------------- paths
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path(os.environ.get("FIRE_DATA_DIR", str(NOTEBOOK_DIR / "data"))).expanduser().resolve()
MANIFEST_DIR = DATA_DIR / "manifests"
RECORDED_DIR = DATA_DIR / "recorded"
PRED_DIR = Path(os.environ.get("FIRE_PRED_DIR", str(DATA_DIR / "predictions"))).expanduser().resolve()
_image_root = os.environ.get("FIRE_DFIRE_IMAGES", "").strip()
DFIRE_IMAGE_DIR = Path(_image_root).expanduser().resolve() if _image_root else None
OUTPUT_DIR = Path(os.environ.get("FIRE_OUTPUT_DIR", str(NOTEBOOK_DIR / "outputs"))).expanduser().resolve()
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"


# --------------------------------------------------------------------------- runs
@dataclass(frozen=True)
class Run:
    run_id: str   # identifier used in this notebook and in its output tables
    folder: str   # folder name of the same run in the published prediction archive
    model: str    # label printed in the paper's tables
    seed: int
    role: str     # "main": the two compared detectors; "ablation": the negative-image ablation


RUNS: list[Run] = [
    Run("yolo11n_seed42", "yolo11n_seed42", "YOLO11n", 42, "main"),
    Run("yolo11n_seed1337", "yolo11n_seed1337", "YOLO11n", 1337, "main"),
    Run("yolo11n_seed2024", "yolo11n_seed2024", "YOLO11n", 2024, "main"),
    Run("resnet18fpn_seed42", "resnet18fpn_seed42", "ResNet18+FPN+FCOS", 42, "main"),
    Run("resnet18fpn_seed1337", "resnet18fpn_seed1337", "ResNet18+FPN+FCOS", 1337, "main"),
    Run("resnet18fpn_seed2024", "resnet18fpn_seed2024", "ResNet18+FPN+FCOS", 2024, "main"),
    # The two ablation arms reuse the ResNet18+FPN+FCOS architecture at seed 42 and differ from
    # resnet18fpn_seed42 only in the list of training images (Section 4, Table 5).
    Run("no_negatives_seed42", "tanpa_negatif_seed42", "No-negatives", 42, "ablation"),
    Run("size_matched_seed42", "kontrol_ukuran_seed42", "Size-matched", 42, "ablation"),
]
RUN_FOLDERS = {r.run_id: r.folder for r in RUNS}
RUN_BY_ID = {r.run_id: r for r in RUNS}
MAIN_RUNS = [r for r in RUNS if r.role == "main"]
ARCHITECTURES = ("YOLO11n", "ResNet18+FPN+FCOS")  # row order of the paper's tables

# --------------------------------------------------------------------------- locked settings
# Class order is fixed: index 0 = fire, 1 = smoke. The D-Fire release numbers smoke as 0; the
# manifests already store class names, so no index translation happens in this notebook.
CLASSES = ("fire", "smoke")
SELECTION_SPLIT = "val"          # thresholds are chosen here, never on the test split
TEST_SPLIT = "test"              # frozen; scored once at the validation-chosen threshold
IOU_MAIN = 0.5
IOU_RANGE = np.round(np.arange(0.50, 1.0, 0.05), 2)                  # 0.50:0.05:0.95 (COCO)
THRESHOLD_SWEEP = np.round(np.arange(0.05, 0.95 + 1e-9, 0.05), 2)    # 19 candidate thresholds
LITERATURE_THRESHOLD = 0.25      # the default most papers use without stating a reason
FIXED_RECALLS = (0.5, 0.6, 0.7, 0.8)
BOOTSTRAP_RESAMPLES = 2000
CI_LEVEL = 0.95
RNG_SEED = 42
COST_RATIOS = (1, 2, 5, 10, 25, 50)   # C_miss : C_false-alarm, swept rather than asserted
MAIN_COST_RATIO = 10
STORED_CONFIDENCE_FLOOR = 0.05
CALIBRATION_BINS = 10
EXPECTED_IMAGES = {"train": 14852, "val": 3418, "test": 3257}

# The test manifest was frozen on 22 August 2026; this content hash appears in every run record.
FROZEN_TEST_MANIFEST_SHA256 = "6246e1bcca7947392097d8460a33e4e5e5823c8ff3c20ebc6d852d3c1af56ae9"
# Byte-level hashes of the shipped parquet files. They are a fallback check: the content hash
# above is computed from the parsed rows and could in principle depend on library versions.
MANIFEST_FILE_SHA256 = {
    "train": "60ce3ea7720d0adcdf1de0da68ab329b90ff6f6115ee584fdf85583c66587852",
    "val": "1137b331d5842bc0a7752f19b1fde373516647de6fc46f63a51a967c4c3a5ce6",
    "test": "62290c8832be70437dff09af323deba6dd13af804403a20cd086765e98212aee",
}

# Reproduction check (Section 3): the rewritten analysis code must reproduce the original
# project's evaluation of ResNet18+FPN+FCOS seed 42 within these tolerances.
REPRODUCTION_REFERENCE = {"map50": 0.66107, "ap50_fire": 0.57554, "ap50_smoke": 0.74659,
                          "far_per_1000": 11.863, "threshold": 0.45}
REPRODUCTION_TOLERANCE = {"map50": 0.0005, "ap50_fire": 0.0005, "ap50_smoke": 0.0005,
                          "far_per_1000": 0.05, "threshold": 0.001}


# --------------------------------------------------------------------------- input checks
def prediction_file(run: Run, split: str) -> Path | None:
    """Locate the stored detections of one run. Three layouts are accepted:

    <pred_dir>/<folder>/<split>_dfire.jsonl            the Zenodo archive dfire_stored_predictions.zip,
                                                       unzipped inside data/
    <pred_dir>/<run_id>/<split>_dfire.jsonl            the same, with the run ids of this notebook
    <pred_dir>/<folder>/prediksi/<split>_dfire.jsonl   the working layout of the original project
    """
    candidates = [
        PRED_DIR / run.folder / f"{split}_dfire.jsonl",
        PRED_DIR / run.run_id / f"{split}_dfire.jsonl",
        PRED_DIR / run.folder / "prediksi" / f"{split}_dfire.jsonl",
    ]
    return next((c for c in candidates if c.is_file()), None)


def check_inputs() -> None:
    problems = []
    for name in ("dfire_train.parquet", "dfire_val.parquet", "dfire_test.parquet",
                 "MANIFEST_HASHES.json"):
        if not (MANIFEST_DIR / name).is_file():
            problems.append(f"manifest file missing: {MANIFEST_DIR / name}")
    for name in ("gpu_efficiency.csv", "training_runs.csv"):
        if not (RECORDED_DIR / name).is_file():
            problems.append(f"recorded file missing: {RECORDED_DIR / name}")
    for run in RUNS:
        for split in (SELECTION_SPLIT, TEST_SPLIT):
            if prediction_file(run, split) is None:
                problems.append(f"predictions missing: {run.run_id} / {split} "
                                f"(looked for {PRED_DIR / run.folder / f'{split}_dfire.jsonl'})")
    if problems:
        raise FileNotFoundError(
            "Some inputs are missing, so the analysis cannot start:\n  - "
            + "\n  - ".join(problems)
            + "\n\nThe manifests and recorded files ship with the repository under data/. "
            "The stored predictions are a separate download (see README.md, 'Getting the data'); "
            "unzip dfire_stored_predictions.zip inside data/, which creates data/predictions/<run>/, or "
            "point FIRE_PRED_DIR at the folder that holds the run folders."
        )


check_inputs()
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"data:        {DATA_DIR}")
print(f"predictions: {PRED_DIR}")
print(f"images:      {DFIRE_IMAGE_DIR if DFIRE_IMAGE_DIR else 'not set (Figure 7 will be skipped)'}")
print(f"outputs:     {OUTPUT_DIR}")
print(f"numpy {np.__version__}, pandas {pd.__version__}, matplotlib {matplotlib.__version__}")

In [ ]:
def save_table(df: pd.DataFrame, name: str) -> Path:
    """Write a table to outputs/tables/ and say so."""
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"  wrote tables/{path.name} ({len(df)} rows)")
    return path


def save_figure(fig: plt.Figure, name: str, formats: tuple[str, ...] = ("png", "pdf"),
                **savefig_kwargs: Any) -> None:
    """Write a figure to outputs/figures/ as PNG (raster, for screens) and PDF (vector, for print),
    with the same bounding box, then show it when running inside Jupyter."""
    stem = Path(name).stem
    for ext in formats:
        # A fixed creation date keeps the PDF bytes identical from one run to the next.
        extra = {"metadata": {"CreationDate": None}} if ext == "pdf" else {}
        fig.savefig(FIGURE_DIR / f"{stem}.{ext}", **savefig_kwargs, **extra)
    print(f"  wrote figures/{stem}.{' and .'.join(formats)}")
    if "inline" in matplotlib.get_backend().lower():
        plt.show()
    else:
        plt.close(fig)


# Embed fonts in PDF/PS output as TrueType (Type 42) rather than matplotlib's default Type 3, which
# publishers' production checks often reject. This affects only vector output, not the PNGs.
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

# Every figure is drawn at its final printed size: the manuscript places each one at the full text
# width of 372 pt (5.17 in), so no figure is scaled in print and the font sizes below are the sizes
# on the page. One scheme for all eight figures: panel letters and axis labels 9 pt, tick labels,
# legends and annotations 8 pt. Constrained layout keeps labels inside the figure without a tight
# bounding box, so the saved files are exactly FIGURE_WIDTH wide. Lines and bars differ by marker,
# dash pattern or grey level as well as by colour, so the figures survive greyscale printing.
FIGURE_WIDTH = 5.17
PAPER_STYLE = {
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.size": 8, "axes.titlesize": 9, "axes.titlelocation": "left", "axes.labelsize": 9,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.bbox": "standard", "figure.constrained_layout.use": True,
}
# Log axes are labelled 1, 10, 100 rather than 10^0, 10^1, 10^2: superscripts would fall below
# the 8 pt minimum.
PLAIN_NUMBER = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}")

# Photographs (Figure 7) keep a full frame and no grid.
PHOTO_STYLE = {**PAPER_STYLE, "axes.grid": False, "axes.spines.top": True, "axes.spines.right": True}
LINE_STYLES = ["-o", "--s", "-.^", ":d", "-v"]
GREYS = ["0.15", "0.45", "0.70", "0.85"]

## 2. Data loading and integrity checks

Two things can silently invalidate every number downstream, so both are checked before any metric is computed.

**The split.** The test manifest is frozen, and its content hash must equal the one recorded at freezing time and in every training run. The hash is order-independent: each manifest row is serialised as canonical JSON and hashed, the row hashes are sorted, and the sorted list is hashed again. A harmless re-sort of the file therefore leaves the hash unchanged, while any change to a row does not.

**The predictions.** A JSON-lines file truncated during a transfer still parses. The missing images would then look like images on which the model detected nothing, recall would drop for no visible reason, and nothing would raise an error. Each prediction file must therefore contain exactly one record per manifest image.

In [ ]:
def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while chunk := fh.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()


def sha256_json(obj: Any) -> str:
    """SHA-256 of canonical JSON (sorted keys, no whitespace), as the original pipeline computed it."""
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def manifest_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    """Rows as plain dicts, with missing values as None (the form the hash was computed on)."""
    return df.astype(object).where(df.notna(), None).to_dict(orient="records")


def manifest_content_sha256(records: list[dict[str, Any]]) -> str:
    return sha256_json(sorted(sha256_json(r) for r in records))


MANIFESTS = {split: pd.read_parquet(MANIFEST_DIR / f"dfire_{split}.parquet")
             for split in ("train", "val", "test")}
HASH_REGISTRY = json.loads((MANIFEST_DIR / "MANIFEST_HASHES.json").read_text(encoding="utf-8"))["manifests"]

integrity_rows = []
for split, df in MANIFESTS.items():
    entry = HASH_REGISTRY[f"dfire_{split}"]
    content_hash = manifest_content_sha256(manifest_records(df))
    file_hash = file_sha256(MANIFEST_DIR / f"dfire_{split}.parquet")
    content_ok = content_hash == entry["content_sha256"]
    file_ok = file_hash == MANIFEST_FILE_SHA256[split]
    if not content_ok and not file_ok:
        raise RuntimeError(
            f"The {split} manifest differs from the one the models were trained and evaluated on:\n"
            f"  registered content hash {entry['content_sha256']}\n"
            f"  recomputed content hash {content_hash}\n"
            "Restore data/manifests/ from the repository before running the analysis."
        )
    if not content_ok:
        print(f"  note: the {split} manifest file is byte-identical to the published one, but this "
              "pandas/pyarrow version serialises its rows differently, so the content hash cannot be "
              "recomputed here. The data are unchanged; continuing.")
    if len(df) != EXPECTED_IMAGES[split]:
        raise RuntimeError(f"{split}: {len(df)} images, expected {EXPECTED_IMAGES[split]}")
    integrity_rows.append({
        "split": split, "images": len(df), "expected_images": EXPECTED_IMAGES[split],
        "content_sha256": content_hash, "matches_registry": content_ok, "file_sha256_matches": file_ok,
        "frozen": bool(entry.get("immutable", False)),
    })

registered_test = HASH_REGISTRY["dfire_test"]["content_sha256"]
if registered_test != FROZEN_TEST_MANIFEST_SHA256 or not HASH_REGISTRY["dfire_test"].get("immutable"):
    raise RuntimeError("The registry does not list the frozen test manifest 6246e1bcca79...")
print(f"test manifest frozen since {HASH_REGISTRY['dfire_test'].get('frozen_utc')}, "
      f"hash {registered_test[:12]}...")
pd.DataFrame(integrity_rows)

### Ground truth

Each manifest row stores its boxes as normalised centre/size coordinates. They are converted to corner coordinates in pixels of the curated image (the size recorded in the manifest), which is the coordinate system the stored detections use.

An image whose annotation file exists but is empty is a **verified negative**: a person has confirmed that it contains neither fire nor smoke. That is different from an image whose annotation status is unknown, which must be left out of every false-alarm denominator, because a detection on an unchecked image is not evidence of a false alarm. In D-Fire every image has an annotation file, so the unknown case never occurs; the `known` flag is kept so the distinction stays in the code.

In [ ]:
@dataclass
class GroundTruth:
    split: str
    ids: list[str]                       # sample ids in manifest order
    index: dict[str, int]
    image_path: list[str]                # path relative to the D-Fire root
    width: np.ndarray                    # int, curated image width in pixels
    height: np.ndarray
    objects: list[list[tuple[list[float], str]]]   # per image: ([x1, y1, x2, y2], class)
    known: np.ndarray                    # bool, annotation exists
    per_class: dict[str, list[np.ndarray]] = field(default_factory=dict)  # float32 (G, 4) per image

    @property
    def negative(self) -> np.ndarray:
        """Verified negatives: annotation known and empty."""
        return self.known & np.array([len(o) == 0 for o in self.objects])

    def n_gt_per_image(self, cls: str) -> np.ndarray:
        return np.array([len(b) for b in self.per_class[cls]], dtype=np.int64) * self.known


def parse_boxes(cell: Any) -> list[dict[str, Any]]:
    if cell is None:
        return []
    return json.loads(cell) if isinstance(cell, str) else list(cell)


def load_ground_truth(split: str) -> GroundTruth:
    df = MANIFESTS[split]
    ids, paths, widths, heights, objects = [], [], [], [], []
    for row in df.itertuples(index=False):
        w, h = int(row.width or 1), int(row.height or 1)
        objs = []
        for column in ("fire_boxes", "smoke_boxes"):
            for b in parse_boxes(getattr(row, column)):
                x1 = (b["cx"] - b["w"] / 2) * w
                y1 = (b["cy"] - b["h"] / 2) * h
                x2 = (b["cx"] + b["w"] / 2) * w
                y2 = (b["cy"] + b["h"] / 2) * h
                objs.append(([max(0.0, x1), max(0.0, y1), min(float(w), x2), min(float(h), y2)],
                             b["cls"]))
        ids.append(row.sample_id)
        paths.append(row.image_path)
        widths.append(w)
        heights.append(h)
        objects.append(objs)
    gt = GroundTruth(
        split=split, ids=ids, index={s: i for i, s in enumerate(ids)}, image_path=paths,
        width=np.array(widths), height=np.array(heights), objects=objects,
        known=np.ones(len(ids), dtype=bool),   # D-Fire: every image has a label file
    )
    for cls in CLASSES:
        gt.per_class[cls] = [
            np.asarray([bb for bb, c in objs if c == cls], dtype=np.float32).reshape(-1, 4)
            for objs in objects
        ]
    return gt


GT = {split: load_ground_truth(split) for split in (SELECTION_SPLIT, TEST_SPLIT)}
for split, gt in GT.items():
    print(f"{split:5s}: {len(gt.ids)} images, {int(gt.negative.sum())} verified negatives, "
          + ", ".join(f"{int(gt.n_gt_per_image(c).sum())} {c} boxes" for c in CLASSES))

### Stored predictions

Each prediction file has one JSON record per image (`sample_id`, `detections` with `bbox` in pixels, `score`, `class`), optionally preceded by a `_meta` record that states the confidence floor used when the file was written. The detections are held as flat arrays sorted by manifest position; within an image the file order is kept, because tied scores are ranked by that order and the original evaluation did the same.

Scores are kept exactly as stored (five decimals) for the false-alarm test, and converted to float32 for ranking and matching, as the original code did. No stored score lies between a float32 rounding of a sweep threshold and the threshold itself, so the two precisions give identical counts on these files.

In [ ]:
@dataclass
class Predictions:
    run_id: str
    split: str
    path: Path
    meta: dict[str, Any]
    n_records: int
    img: np.ndarray      # int32, manifest position of the image
    cls: np.ndarray      # int8, index into CLASSES
    score: np.ndarray    # float64, confidence as stored
    box: np.ndarray      # float32 (N, 4), x1 y1 x2 y2 in pixels

    def image_slice(self, i: int) -> slice:
        return slice(int(np.searchsorted(self.img, i, "left")), int(np.searchsorted(self.img, i, "right")))


def load_predictions(run: Run, split: str, gt: GroundTruth) -> Predictions:
    path = prediction_file(run, split)
    assert path is not None
    class_index = {c: i for i, c in enumerate(CLASSES)}
    meta: dict[str, Any] = {}
    seen: set[str] = set()
    img_l: list[int] = []
    cls_l: list[int] = []
    score_l: list[float] = []
    box_l: list[list[float]] = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "_meta" in rec:
                meta = rec["_meta"]
                continue
            sid = rec["sample_id"]
            i = gt.index.get(sid)
            if i is None:
                raise ValueError(f"{path}: {sid} is not in the {split} manifest")
            if sid in seen:
                raise ValueError(f"{path}: {sid} appears twice")
            seen.add(sid)
            for d in rec.get("detections", []):
                img_l.append(i)
                cls_l.append(class_index[d["class"]])
                score_l.append(float(d["score"]))
                box_l.append(d["bbox"])
    if len(seen) != len(gt.ids):
        raise RuntimeError(
            f"{path}: {len(seen)} image records, but the {split} manifest has {len(gt.ids)} images. "
            "The file is incomplete (for example truncated during download); fetch it again."
        )
    img = np.asarray(img_l, dtype=np.int32)
    order = np.argsort(img, kind="stable")
    return Predictions(
        run_id=run.run_id, split=split, path=path, meta=meta, n_records=len(seen),
        img=img[order], cls=np.asarray(cls_l, dtype=np.int8)[order],
        score=np.asarray(score_l, dtype=np.float64)[order],
        box=np.asarray(box_l, dtype=np.float32).reshape(-1, 4)[order],
    )


PRED: dict[tuple[str, str], Predictions] = {}
t0 = time.time()
for run in RUNS:
    for split in (SELECTION_SPLIT, TEST_SPLIT):
        PRED[(run.run_id, split)] = load_predictions(run, split, GT[split])
    print(f"  loaded {run.run_id:22s} ({time.time() - t0:5.1f} s)")

prediction_rows = []
for run in RUNS:
    for split in (SELECTION_SPLIT, TEST_SPLIT):
        p = PRED[(run.run_id, split)]
        prediction_rows.append({
            "run_id": run.run_id, "model": run.model, "seed": run.seed, "split": split,
            "images_in_file": p.n_records, "images_expected": EXPECTED_IMAGES[split],
            "complete": p.n_records == EXPECTED_IMAGES[split], "boxes": len(p.score),
            "stored_confidence_floor": p.meta.get("conf_threshold_used"),
            "min_stored_score": float(p.score.min()) if len(p.score) else None,
            "inference_seconds": p.meta.get("inference_seconds"),
        })
PREDICTIONS_TABLE = pd.DataFrame(prediction_rows)
save_table(PREDICTIONS_TABLE, "predictions_available")
assert PREDICTIONS_TABLE.complete.all()
assert (PREDICTIONS_TABLE.min_stored_score >= STORED_CONFIDENCE_FLOOR).all()
PREDICTIONS_TABLE

## 3. Metrics and the reproduction check

**Average precision** follows the pycocotools convention: IoU thresholds 0.50:0.05:0.95, 101-point interpolated precision, and greedy matching in which detections are taken highest score first and each ground-truth box can be matched once. A detection claims the unused ground-truth box it overlaps most, provided the overlap reaches the IoU threshold. AP for a class without ground-truth boxes is undefined, not zero.

One detail of the interpolation is easy to get wrong. Recall points must be located with `searchsorted(..., side="left")`; using `np.interp` instead gives a perfect detector an AP of 0.9901 rather than 1.0 and pulls every AP down slightly.

**Precision, recall and F1** are computed at one confidence threshold, with IoU fixed at 0.50, and averaged over the two classes.

**False-alarm rate (Equation 1).** Only verified negatives enter the denominator:

$$\mathrm{FAR}_{\text{per }1{,}000} = 1000 \times \frac{N_{\mathrm{alarm}}}{N_{\mathrm{neg}}}$$

A negative image counts as one alarm if at least one retained detection of either class reaches the threshold, however many boxes it carries. The operator hears one alarm per image, not one per rectangle.

Matching does not depend on the confidence threshold, so it is computed once per run and split; every later threshold sweep only filters the matched detections with `>=`.

In [ ]:
RECALL_POINTS = np.linspace(0.0, 1.0, 101)


def iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """IoU of every box in `boxes_a` (N, 4) against every box in `boxes_b` (M, 4), xyxy pixels."""
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    a = boxes_a[:, None, :]
    b = boxes_b[None, :, :]
    x1 = np.maximum(a[..., 0], b[..., 0])
    y1 = np.maximum(a[..., 1], b[..., 1])
    x2 = np.minimum(a[..., 2], b[..., 2])
    y2 = np.minimum(a[..., 3], b[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = np.clip(a[..., 2] - a[..., 0], 0, None) * np.clip(a[..., 3] - a[..., 1], 0, None)
    area_b = np.clip(b[..., 2] - b[..., 0], 0, None) * np.clip(b[..., 3] - b[..., 1], 0, None)
    union = area_a + area_b - inter
    return (inter / np.clip(union, 1e-9, None)).astype(np.float32)


def greedy_match(scores: np.ndarray, iou: np.ndarray, iou_thresholds: np.ndarray) -> np.ndarray:
    """True-positive flags (N, T) for the detections of one image and one class.

    Detections are visited highest score first (stable for ties). Each one walks the ground-truth
    boxes in order of decreasing IoU, stops at the first box below the threshold, and claims the
    first box not yet used. A detection whose best IoU is below the threshold cannot match, so it
    is skipped without walking the list; the outcome is the same, only faster.
    """
    tp = np.zeros((len(scores), len(iou_thresholds)), dtype=np.float32)
    if iou.size == 0:
        return tp
    best = iou.max(axis=1)
    order = np.argsort(-scores, kind="stable")
    ranked: dict[int, np.ndarray] = {}
    for t, thr in enumerate(iou_thresholds):
        used: set[int] = set()
        for k in order[best[order] >= thr]:
            k = int(k)
            if k not in ranked:
                ranked[k] = np.argsort(-iou[k])
            for g in ranked[k]:
                if iou[k, g] < thr:
                    break
                if int(g) not in used:
                    used.add(int(g))
                    tp[k, t] = 1.0
                    break
    return tp


@dataclass
class ClassMatch:
    """Matching result for one class on one split, detections in canonical order."""
    img: np.ndarray            # int32 image position per detection
    score: np.ndarray          # float32
    tp: np.ndarray             # float32 (N, T): matched at IoU_RANGE[t] (or IoU 0.5 when T = 1)
    n_gt: int
    n_gt_per_image: np.ndarray

    def at_iou50(self) -> "ClassMatch":
        return ClassMatch(self.img, self.score, self.tp[:, :1].copy(), self.n_gt, self.n_gt_per_image)


def match_detections(gt: GroundTruth, pred: Predictions,
                     iou_thresholds: np.ndarray | list[float]) -> dict[str, ClassMatch]:
    thresholds = np.asarray(iou_thresholds, dtype=np.float64)
    out: dict[str, ClassMatch] = {}
    for c_idx, cls in enumerate(CLASSES):
        sel = (pred.cls == c_idx) & gt.known[pred.img]
        img = pred.img[sel]
        score = pred.score[sel].astype(np.float32)
        boxes = pred.box[sel]
        tp = np.zeros((len(img), len(thresholds)), dtype=np.float32)
        if len(img):
            starts = np.flatnonzero(np.r_[True, img[1:] != img[:-1]])
            ends = np.r_[starts[1:], len(img)]
            for s, e in zip(starts, ends):
                gt_boxes = gt.per_class[cls][img[s]]
                if len(gt_boxes):
                    tp[s:e] = greedy_match(score[s:e], iou_matrix(boxes[s:e], gt_boxes), thresholds)
        n_img = gt.n_gt_per_image(cls)
        out[cls] = ClassMatch(img, score, tp, int(n_img.sum()), n_img)
    return out


def average_precision(tp: np.ndarray, scores: np.ndarray, n_gt: int) -> float:
    """101-point interpolated AP from true-positive flags and scores (pycocotools convention)."""
    if n_gt == 0:
        return float("nan")
    order = np.argsort(-scores, kind="stable")
    tp = tp[order]
    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(1 - tp)
    recall = tp_cum / n_gt
    precision = tp_cum / np.clip(tp_cum + fp_cum, 1e-9, None)
    # Monotone envelope: a momentary dip in precision must not lower the AP.
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    interp = np.zeros_like(RECALL_POINTS)
    idx = np.searchsorted(recall, RECALL_POINTS, side="left")
    valid = idx < len(precision)
    interp[valid] = precision[idx[valid]]
    return float(interp.mean())


def image_max_score(gt: GroundTruth, pred: Predictions) -> np.ndarray:
    """Highest stored score of any class on each image (-inf when the image has no detection)."""
    out = np.full(len(gt.ids), -np.inf)
    np.maximum.at(out, pred.img, pred.score)
    return out


def false_alarm_rate(gt: GroundTruth, max_score: np.ndarray, threshold: float) -> dict[str, Any]:
    """Equation 1: alarms per 1,000 verified negatives; one image raises at most one alarm."""
    neg = gt.negative
    n_neg = int(neg.sum())
    if n_neg == 0:
        return {"per_1000": None, "n_negative": 0, "n_fired": 0}
    fired = int((max_score[neg] >= threshold).sum())
    return {"per_1000": round(1000.0 * fired / n_neg, 3), "n_negative": n_neg, "n_fired": fired}


def negative_class_max(gt: GroundTruth, pred: Predictions) -> np.ndarray:
    """(n_negatives, n_classes) float32: highest score per class on each verified negative, 0 if none.

    With this matrix the alarm rate can be evaluated for any pair of per-class thresholds without
    touching the prediction file again: an image fires if any class reaches its own threshold.
    """
    neg_idx = np.flatnonzero(gt.negative)
    position = np.full(len(gt.ids), -1)
    position[neg_idx] = np.arange(len(neg_idx))
    out = np.zeros((len(neg_idx), len(CLASSES)), dtype=np.float32)
    on_negative = position[pred.img] >= 0
    for j in range(len(CLASSES)):
        sel = on_negative & (pred.cls == j)
        best = np.full(len(neg_idx), -np.inf)
        np.maximum.at(best, position[pred.img[sel]], pred.score[sel])
        best[np.isinf(best)] = 0.0
        out[:, j] = best.astype(np.float32)
    return out


def class_mean(values: dict[str, float | None]) -> float | None:
    vals = [v for v in values.values() if v is not None]
    return round(float(np.mean(vals)), 5) if vals else None


def evaluate_detection(gt: GroundTruth, match: dict[str, ClassMatch], max_score: np.ndarray,
                       threshold: float) -> dict[str, Any]:
    """AP/mAP (threshold-free) plus precision, recall, F1 and FAR at one confidence threshold."""
    res: dict[str, Any] = {k: {} for k in ("ap50", "ap50_95", "precision", "recall", "f1", "n_gt", "n_pred")}
    for cls in CLASSES:
        m = match[cls]
        res["n_gt"][cls], res["n_pred"][cls] = m.n_gt, len(m.score)
        if m.n_gt == 0:          # AP undefined, not zero
            for k in ("ap50", "ap50_95", "precision", "recall", "f1"):
                res[k][cls] = None
            continue
        if len(m.score) == 0:
            for k in ("ap50", "ap50_95", "precision", "recall", "f1"):
                res[k][cls] = 0.0
            continue
        aps = [average_precision(m.tp[:, t], m.score, m.n_gt) for t in range(m.tp.shape[1])]
        res["ap50"][cls] = round(aps[0], 5)
        valid = [a for a in aps if not np.isnan(a)]
        res["ap50_95"][cls] = round(float(np.mean(valid)), 5) if valid else None
        used = m.score >= threshold
        tp_n = float(m.tp[used, 0].sum())
        fp_n = float(used.sum() - tp_n)
        p = tp_n / max(1e-9, tp_n + fp_n)
        r = tp_n / max(1e-9, m.n_gt)
        res["precision"][cls] = round(p, 5)
        res["recall"][cls] = round(r, 5)
        res["f1"][cls] = round(2 * p * r / max(1e-9, p + r), 5)
    for k in ("ap50", "ap50_95", "precision", "recall", "f1"):
        res[f"mean_{k}"] = class_mean(res[k])
    res["far"] = false_alarm_rate(gt, max_score, threshold)
    return res


def select_threshold(match50: dict[str, ClassMatch], candidates: np.ndarray) -> tuple[dict, list[dict]]:
    """Threshold with the highest class-mean F1 on the validation split (ties: lowest threshold)."""
    trace = []
    for t in candidates:
        f1s, ps, rs = [], [], []
        for cls in CLASSES:
            m = match50[cls]
            if m.n_gt == 0:
                continue
            used = m.score >= t
            tp = float(m.tp[used, 0].sum())
            fp = float(used.sum() - tp)
            p = tp / max(1e-9, tp + fp)
            r = tp / max(1e-9, m.n_gt)
            ps.append(p)
            rs.append(r)
            f1s.append(2 * p * r / max(1e-9, p + r))
        trace.append({
            "threshold": round(float(t), 2),
            "f1": round(float(np.mean(f1s)), 5) if f1s else None,
            "precision": round(float(np.mean(ps)), 5) if ps else None,
            "recall": round(float(np.mean(rs)), 5) if rs else None,
        })
    best = max((r for r in trace if r["f1"] is not None), key=lambda r: r["f1"])
    return best, trace

### Threshold selection on validation, one scoring of the test split

For every run the reporting threshold is the one that maximises class-mean F1 on the validation split, chosen among 0.05, 0.10, ..., 0.95. The test split is then scored once at that threshold. No test image takes part in the choice.

In [ ]:
MATCH50: dict[tuple[str, str], dict[str, ClassMatch]] = {}   # IoU-0.5 matching per run and split
MAX_SCORE: dict[tuple[str, str], np.ndarray] = {}
NEG_MAX: dict[tuple[str, str], np.ndarray] = {}
metric_rows, selection_rows = [], []

t0 = time.time()
for run in RUNS:
    pv, pt = PRED[(run.run_id, SELECTION_SPLIT)], PRED[(run.run_id, TEST_SPLIT)]
    m_val = match_detections(GT[SELECTION_SPLIT], pv, [IOU_MAIN])
    best, trace = select_threshold(m_val, THRESHOLD_SWEEP)
    threshold = best["threshold"]

    m_test = match_detections(GT[TEST_SPLIT], pt, IOU_RANGE)
    ms_test = image_max_score(GT[TEST_SPLIT], pt)
    res = evaluate_detection(GT[TEST_SPLIT], m_test, ms_test, threshold)

    MATCH50[(run.run_id, SELECTION_SPLIT)] = m_val
    MATCH50[(run.run_id, TEST_SPLIT)] = {c: m.at_iou50() for c, m in m_test.items()}
    MAX_SCORE[(run.run_id, TEST_SPLIT)] = ms_test
    MAX_SCORE[(run.run_id, SELECTION_SPLIT)] = image_max_score(GT[SELECTION_SPLIT], pv)
    for split in (SELECTION_SPLIT, TEST_SPLIT):
        NEG_MAX[(run.run_id, split)] = negative_class_max(GT[split], PRED[(run.run_id, split)])

    selection_rows.append({
        "run_id": run.run_id, "model": run.model, "seed": run.seed, "threshold": threshold,
        "selected_on": SELECTION_SPLIT, "val_f1": best["f1"], "val_recall": best["recall"],
        "val_precision": best["precision"], "thresholds_tried": len(trace),
        "criterion": "highest class-mean F1",
    })
    metric_rows.append({
        "run_id": run.run_id, "model": run.model, "seed": run.seed, "role": run.role,
        "threshold": threshold,
        "map50": res["mean_ap50"], "map50_95": res["mean_ap50_95"],
        "ap50_fire": res["ap50"]["fire"], "ap50_smoke": res["ap50"]["smoke"],
        "ap50_95_fire": res["ap50_95"]["fire"], "ap50_95_smoke": res["ap50_95"]["smoke"],
        "precision": res["mean_precision"], "recall": res["mean_recall"], "f1": res["mean_f1"],
        "recall_fire": res["recall"]["fire"], "recall_smoke": res["recall"]["smoke"],
        "far_per_1000": res["far"]["per_1000"], "negatives": res["far"]["n_negative"],
        "negatives_fired": res["far"]["n_fired"],
        "gt_fire": res["n_gt"]["fire"], "gt_smoke": res["n_gt"]["smoke"],
        "detections_fire": res["n_pred"]["fire"], "detections_smoke": res["n_pred"]["smoke"],
    })
    print(f"  {run.run_id:22s} thr {threshold:.2f} | mAP50 {res['mean_ap50']:.5f} | "
          f"AP fire {res['ap50']['fire']:.5f} | AP smoke {res['ap50']['smoke']:.5f} | "
          f"FAR {res['far']['per_1000']:7.3f}/1000 ({time.time() - t0:5.1f} s)")

RUN_METRICS = pd.DataFrame(metric_rows)
VAL_SELECTION = pd.DataFrame(selection_rows)
save_table(RUN_METRICS, "run_metrics")
save_table(VAL_SELECTION, "thresholds_selected_on_validation")

### Reproduction check

The analysis code was rewritten independently of the training pipeline, and a rewrite of metric code can produce plausible but wrong numbers. As a guard, it must reproduce the original project's evaluation of ResNet18+FPN+FCOS seed 42 within the locked tolerances (0.0005 for AP values, 0.05 per 1,000 for the alarm rate, exact for the threshold). If it does not, the notebook stops here rather than produce tables from a faulty metric. Section III-H of the paper lists all eight quantities; section 15 checks each of them to five decimals.

In [ ]:
ref_row = RUN_METRICS.set_index("run_id").loc["resnet18fpn_seed42"]
repro = []
for key, reference in REPRODUCTION_REFERENCE.items():
    value = float(ref_row[key])
    diff = abs(value - reference)
    repro.append({"quantity": key, "original_project": reference, "this_notebook": value,
                  "difference": round(diff, 6), "tolerance": REPRODUCTION_TOLERANCE[key],
                  "status": "PASS" if diff <= REPRODUCTION_TOLERANCE[key] else "FAIL"})
REPRODUCTION = pd.DataFrame(repro)
print(REPRODUCTION.to_string(index=False))
print(f"\n  ({int(ref_row.negatives_fired)} of {int(ref_row.negatives)} verified negatives raised an alarm)")
if (REPRODUCTION.status != "PASS").any():
    raise RuntimeError("Reproduction check failed: the metric code does not reproduce the original "
                       "evaluation. Stop and compare the implementation before using any table.")
print("Reproduction check passed.")

## 4. Dataset composition, object scale and the ablation splits (Tables 2, 3 and 5)

Table 2 counts images, verified negatives and boxes per split; every false-alarm rate in the paper uses the 1,433 verified negatives of the test split as its denominator. Table 3 measures object size as the fraction of the image a box covers. The difference between the classes is large enough to matter later: a median smoke box covers about 29 times the area of a median fire box in the training split.

Table 5 needs the training manifests of the two ablation arms. They are not shipped separately because they can be rebuilt exactly from the full training manifest with the recipe the authors used: the No-negatives arm keeps every image with at least one box; the Size-matched arm draws the same number of images with the original positive-to-negative ratio, using Python's `random.Random(42)`. Their content hashes must equal the hashes recorded in the training runs.

In [ ]:
def count_boxes(cell: Any) -> int:
    return len(parse_boxes(cell))


dataset_rows = []
for split in ("train", "val", "test"):
    d = MANIFESTS[split]
    n_fire = d.fire_boxes.map(count_boxes)
    n_smoke = d.smoke_boxes.map(count_boxes)
    has_object = (n_fire + n_smoke) > 0
    entry = HASH_REGISTRY[f"dfire_{split}"]
    dataset_rows.append({
        "split": split,
        "images": len(d),
        "with_objects": int(has_object.sum()),
        "verified_negative": int((~has_object).sum()),
        "pct_negative": round(100 * float((~has_object).mean()), 1),
        "fire_boxes": int(n_fire.sum()),
        "smoke_boxes": int(n_smoke.sum()),
        "images_with_fire": int((n_fire > 0).sum()),
        "images_with_smoke": int((n_smoke > 0).sum()),
        "median_width": int(d.width.median()),
        "median_height": int(d.height.median()),
        "manifest_hash": entry["content_sha256"][:12],
        "frozen": bool(entry.get("immutable", False)),
    })
TABLE_2 = pd.DataFrame(dataset_rows)
save_table(TABLE_2, "table02_dataset_composition")

scale_rows = []
for split in ("train", "test"):
    d = MANIFESTS[split]
    for column, cls in (("fire_boxes", "fire"), ("smoke_boxes", "smoke")):
        areas = pd.Series([b["w"] * b["h"] for cell in d[column] for b in parse_boxes(cell)])
        scale_rows.append({
            "split": split, "class": cls, "boxes": len(areas),
            "median_area_fraction": round(float(areas.median()), 5),
            "mean_area_fraction": round(float(areas.mean()), 5),
            "pct_boxes_below_1pct_of_image": round(100 * float((areas < 0.01).mean()), 1),
        })
TABLE_3 = pd.DataFrame(scale_rows)
save_table(TABLE_3, "table03_object_scale")
display_cols = ["split", "images", "with_objects", "verified_negative", "pct_negative",
                "fire_boxes", "smoke_boxes", "manifest_hash", "frozen"]
print(TABLE_2[display_cols].to_string(index=False))
print()
print(TABLE_3.to_string(index=False))

In [ ]:
# Published description of D-Fire (release page, https://github.com/gaiasd/DFireDataset):
# 21,527 images, 9,838 of them without fire or smoke; 14,692 fire and 11,865 smoke boxes.
DFIRE_PUBLISHED = {"images": 21527, "negative_images": 9838, "only_fire": 1164, "only_smoke": 5867,
                   "fire_and_smoke": 4658, "fire_boxes": 14692, "smoke_boxes": 11865}

all_rows = pd.concat(MANIFESTS.values(), ignore_index=True)
nf_all = all_rows.fire_boxes.map(count_boxes)
ns_all = all_rows.smoke_boxes.map(count_boxes)
DATASET_AUDIT = {
    "images": len(all_rows),
    "negative_images": int(((nf_all + ns_all) == 0).sum()),
    "only_fire": int(((nf_all > 0) & (ns_all == 0)).sum()),
    "only_smoke": int(((nf_all == 0) & (ns_all > 0)).sum()),
    "fire_and_smoke": int(((nf_all > 0) & (ns_all > 0)).sum()),
    "fire_boxes": int(nf_all.sum()),
    "smoke_boxes": int(ns_all.sum()),
}
# Images from D-Fire's official test folder that the rebuilt split places in training (Section V).
DATASET_AUDIT["official_test_images_in_train"] = int(
    MANIFESTS["train"].sample_id.str.startswith("dfire/test/").sum())
# Exact duplicates across splits, by file content hash and by perceptual hash.
cross_split = 0
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    for column in ("content_hash", "perceptual_hash"):
        cross_split += len(set(MANIFESTS[a][column]) & set(MANIFESTS[b][column]))
DATASET_AUDIT["cross_split_identical_hashes"] = cross_split

audit = pd.DataFrame([{"quantity": k, "this_split": DATASET_AUDIT[k],
                       "published_D-Fire": DFIRE_PUBLISHED.get(k, "")} for k in DATASET_AUDIT])
print(audit.to_string(index=False))
print(f"\nBox shortfall against the published counts (degenerate annotations dropped by the parser): "
      f"{DFIRE_PUBLISHED['fire_boxes'] - DATASET_AUDIT['fire_boxes']} fire, "
      f"{DFIRE_PUBLISHED['smoke_boxes'] - DATASET_AUDIT['smoke_boxes']} smoke")

In [ ]:
TRAINING_RUNS = pd.read_csv(RECORDED_DIR / "training_runs.csv").set_index("run_id")
ABLATION_SAMPLING_SEED = 42   # fixed by the authors so the Size-matched subset can be rebuilt exactly

train_records = manifest_records(MANIFESTS["train"])


def boxes_in_record(rec: dict[str, Any]) -> int:
    return count_boxes(rec["fire_boxes"]) + count_boxes(rec["smoke_boxes"])


positives = [r for r in train_records if boxes_in_record(r) > 0]
negatives = [r for r in train_records if boxes_in_record(r) == 0]
target_size = len(positives)
n_neg_matched = round(target_size * len(negatives) / len(train_records))
n_pos_matched = target_size - n_neg_matched
sampler = random.Random(ABLATION_SAMPLING_SEED)
size_matched = sampler.sample(positives, n_pos_matched) + sampler.sample(negatives, n_neg_matched)
sampler.shuffle(size_matched)   # order does not enter the hash; kept to mirror the original recipe

ABLATION_MANIFESTS = {
    "Full": (train_records, "resnet18fpn_seed42"),
    "No-negatives": (positives, "no_negatives_seed42"),
    "Size-matched": (size_matched, "size_matched_seed42"),
}
ablation_rows = []
for condition, (records, run_id) in ABLATION_MANIFESTS.items():
    n_pos = sum(boxes_in_record(r) > 0 for r in records)
    content_hash = manifest_content_sha256(records)
    recorded = TRAINING_RUNS.loc[run_id, "train_manifest_sha256"]
    ablation_rows.append({
        "condition": condition, "run_id": run_id, "training_images": len(records),
        "positives": n_pos, "negatives": len(records) - n_pos,
        "composition": "positives only" if n_pos == len(records)
        else f"{n_pos:,} positive + {len(records) - n_pos:,} negative",
        "train_manifest_hash": content_hash[:12],
        "matches_training_run_record": content_hash == recorded,
    })
TABLE_5 = pd.DataFrame(ablation_rows)
save_table(TABLE_5, "table05_ablation_conditions")
assert TABLE_5.matches_training_run_record.all(), "an ablation manifest could not be rebuilt"
# The three conditions share the validation manifest (and the frozen test manifest).
assert TRAINING_RUNS.val_manifest_sha256.nunique() == 1
assert TRAINING_RUNS.val_manifest_sha256.iloc[0] == HASH_REGISTRY["dfire_val"]["content_sha256"]
TABLE_5

## 5. Main results at the validation-selected threshold (Tables 6, 7 and 9; Figure 4)

Means and standard deviations are taken over the three seeds of each architecture, using the per-run values rounded to five decimals (three for the alarm rate), as the original tables did. The seed spread is the noise floor of this study: an mAP50 difference smaller than about twice the larger standard deviation cannot be told apart from a lucky seed.

In [ ]:
MAIN_METRICS = RUN_METRICS[RUN_METRICS.role == "main"].copy()
TABLE_6_METRICS = [("map50", "mAP50", 4), ("map50_95", "mAP50-95", 4), ("ap50_fire", "AP_fire", 4),
                   ("ap50_smoke", "AP_smoke", 4), ("recall", "Recall", 4),
                   ("precision", "Precision", 4), ("far_per_1000", "FAR / 1000", 2)]


def seed_summary(df: pd.DataFrame, column: str) -> tuple[float, float]:
    x = df[column].astype(float).to_numpy()
    return float(x.mean()), (float(x.std(ddof=1)) if len(x) > 1 else 0.0)


def threshold_label(values: pd.Series) -> str:
    lo, hi = float(values.min()), float(values.max())
    return f"{lo:.2f}" if lo == hi else f"{lo:.2f}–{hi:.2f}"


SEED_STATS: dict[tuple[str, str], tuple[float, float]] = {}
t6_numeric, t6_display = [], {"quantity": ["Seeds", "Thr."] + [n for _, n, _ in TABLE_6_METRICS]}
for arch in ARCHITECTURES:
    g = MAIN_METRICS[MAIN_METRICS.model == arch]
    column = [str(len(g)), threshold_label(g.threshold)]
    for key, name, dec in TABLE_6_METRICS:
        mean, sd = seed_summary(g, key)
        SEED_STATS[(arch, key)] = (mean, sd)
        t6_numeric.append({"model": arch, "quantity": name, "mean": mean, "sd": sd, "seeds": len(g)})
        column.append(f"{mean:.{dec}f} ± {sd:.{dec}f}")
    t6_display[arch] = column
TABLE_6 = pd.DataFrame(t6_display)
save_table(TABLE_6, "table06_main_results")
save_table(pd.DataFrame(t6_numeric), "table06_main_results_numeric")

TABLE_7 = MAIN_METRICS[["model", "seed", "map50", "far_per_1000", "threshold"]].copy()
TABLE_7["as_printed"] = [f"{r.map50:.4f} / {r.far_per_1000:.2f} / {r.threshold:.2f}"
                         for r in TABLE_7.itertuples(index=False)]
save_table(TABLE_7, "table07_per_seed")

TABLE_9 = pd.DataFrame([{
    "model": arch,
    "precision": f"{SEED_STATS[(arch, 'precision')][0]:.4f} ± {SEED_STATS[(arch, 'precision')][1]:.4f}",
    "far_per_1000": f"{SEED_STATS[(arch, 'far_per_1000')][0]:.2f} ± {SEED_STATS[(arch, 'far_per_1000')][1]:.2f}",
} for arch in ARCHITECTURES])
save_table(TABLE_9, "table09_precision_vs_far")

print(TABLE_6.to_string(index=False))
print()
print(TABLE_7.to_string(index=False))
print()
print(TABLE_9.to_string(index=False))

# Quantities quoted in the text of Section IV-A.
y_map, r_map = SEED_STATS[("YOLO11n", "map50")], SEED_STATS[("ResNet18+FPN+FCOS", "map50")]
y_far, r_far = SEED_STATS[("YOLO11n", "far_per_1000")], SEED_STATS[("ResNet18+FPN+FCOS", "far_per_1000")]
TEXT_VALUES: dict[str, float] = {
    "map50_gain_points": 100 * (y_map[0] - r_map[0]),
    "far_ratio_yolo_over_resnet": y_far[0] / r_far[0],
    "cv_far_yolo_pct": 100 * y_far[1] / y_far[0],
    "cv_far_resnet_pct": 100 * r_far[1] / r_far[0],
    "cv_map50_yolo_pct": 100 * y_map[1] / y_map[0],
    "cv_map50_resnet_pct": 100 * r_map[1] / r_map[0],
}
print(f"\nYOLO11n gains {TEXT_VALUES['map50_gain_points']:.1f} mAP50 points at "
      f"{TEXT_VALUES['far_ratio_yolo_over_resnet']:.1f} times the alarm rate.")
print(f"Coefficient of variation, FAR: {TEXT_VALUES['cv_far_yolo_pct']:.2f}% (YOLO11n), "
      f"{TEXT_VALUES['cv_far_resnet_pct']:.2f}% (ResNet); mAP50: {TEXT_VALUES['cv_map50_yolo_pct']:.2f}% "
      f"and {TEXT_VALUES['cv_map50_resnet_pct']:.2f}%")

In [ ]:
def figure_seed_spread() -> None:
    from matplotlib.legend_handler import HandlerTuple
    from matplotlib.lines import Line2D

    d = MAIN_METRICS
    labels = sorted(d.model.unique())
    markers = ["o", "s"]
    with plt.rc_context(PAPER_STYLE):
        fig, (left, right) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, 2.6))
        for i, lab in enumerate(labels):
            g = d[d.model == lab]
            for ax, column in ((left, "map50"), (right, "far_per_1000")):
                ax.scatter([i] * len(g), g[column], s=30, marker=markers[i], facecolors="none",
                           edgecolors="black", linewidths=1.0)
                ax.hlines(g[column].mean(), i - 0.22, i + 0.22, linewidth=2, color="black")
        for ax, ylabel, title in ((left, "mAP50", "(a) mAP50"),
                                  (right, "False alarms per 1,000", "(b) False-alarm rate")):
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels)
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            ax.set_xlim(-0.5, len(labels) - 0.5)
        # The low-alarm detector sits on the left, so the upper left of panel (b) is empty.
        right.set_ylim(0, d.far_per_1000.max() * 1.45)
        seed_marks = tuple(Line2D([], [], marker=m, ls="none", mfc="none", mec="black", ms=5) for m in markers)
        mean_bar = Line2D([], [], color="black", lw=2)
        right.legend([seed_marks, mean_bar], ["one seed", "mean of three seeds"], loc="upper left",
                     frameon=False, handler_map={tuple: HandlerTuple(ndivide=None)})
        save_figure(fig, "Fig4.png")


figure_seed_spread()

## 6. Class asymmetry: fire is the harder class (Figure 1)

Fire AP is below smoke AP in all six runs. Panel (b) of Figure 1 points to object scale as the likely mechanism: in the test split the median fire box is about 24 times smaller than the median smoke box.

In [ ]:
FIRE_SMOKE = MAIN_METRICS.groupby("model", as_index=False).agg(
    ap_fire=("ap50_fire", "mean"), ap_smoke=("ap50_smoke", "mean"))
FIRE_SMOKE["gap_smoke_minus_fire"] = FIRE_SMOKE.ap_smoke - FIRE_SMOKE.ap_fire
FIRE_SMOKE["gap_over_noise_floor"] = [
    row.gap_smoke_minus_fire / SEED_STATS[(row.model, "map50")][1] for row in FIRE_SMOKE.itertuples()]
print(FIRE_SMOKE.to_string(index=False))
assert (MAIN_METRICS.ap50_fire < MAIN_METRICS.ap50_smoke).all(), "fire AP is not below smoke AP in every run"

test_scale = TABLE_3[TABLE_3.split == "test"].set_index("class")
train_scale = TABLE_3[TABLE_3.split == "train"].set_index("class")
TEXT_VALUES["smoke_over_fire_median_area_train"] = (train_scale.loc["smoke", "median_area_fraction"]
                                                    / train_scale.loc["fire", "median_area_fraction"])
TEXT_VALUES["smoke_over_fire_median_area_test"] = (test_scale.loc["smoke", "median_area_fraction"]
                                                   / test_scale.loc["fire", "median_area_fraction"])
print(f"median smoke box / median fire box: {TEXT_VALUES['smoke_over_fire_median_area_train']:.1f}x (train), "
      f"{TEXT_VALUES['smoke_over_fire_median_area_test']:.1f}x (test)")


def figure_fire_vs_smoke() -> None:
    means = FIRE_SMOKE
    test = TABLE_3[TABLE_3.split == "test"]
    with plt.rc_context(PAPER_STYLE):
        fig, (left, right) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, 2.6))
        x = range(len(means))
        width = 0.36
        left.bar([i - width / 2 for i in x], means.ap_fire, width, label="AP fire",
                 color="0.30", edgecolor="black", linewidth=0.7)
        left.bar([i + width / 2 for i in x], means.ap_smoke, width, label="AP smoke",
                 color="0.78", edgecolor="black", linewidth=0.7)
        left.set_xticks(list(x))
        left.set_xticklabels(list(means.model))
        left.set_ylabel("AP at IoU 0.50")
        # Head room above the tallest bar (0.78) keeps the legend clear of the data.
        left.set_ylim(0, 1.0)
        left.legend(loc="upper center", ncol=2, frameon=False)
        left.set_title("(a) AP by class")

        shade = {"fire": "0.30", "smoke": "0.78"}
        right.bar(test["class"], test.median_area_fraction * 100, width=0.5,
                  color=[shade.get(k, "0.5") for k in test["class"]], edgecolor="black", linewidth=0.7)
        right.set_ylabel("Median box area (% of image)")
        right.set_xticks(range(len(test)))
        right.set_xticklabels(list(test["class"]))
        right.set_ylim(0, float(test.median_area_fraction.max()) * 100 * 1.2)
        for i, r in enumerate(test.itertuples(index=False)):
            right.text(i, r.median_area_fraction * 100, f"{r.median_area_fraction * 100:.2f}%",
                       ha="center", va="bottom")
        right.set_title("(b) Object scale, test split")
        save_figure(fig, "Fig1.png")


figure_fire_vs_smoke()

## 7. False alarms against recall (Figure 2 and the point estimates of Table 8)

Comparing alarm rates at each model's own threshold invites an objection: a model running at higher recall should be expected to raise more alarms. So the alarm rate is also read off at fixed recall. For each run, recall and the alarm rate are computed at the 19 thresholds of the sweep, and the alarm rate at recall 0.5, 0.6, 0.7 and 0.8 is linearly interpolated along that curve. A recall outside the reachable range gives no value rather than an extrapolation.

The curve is computed on the test split. That is description, not tuning: nothing is chosen from it, and the reporting thresholds of Section 5 stay those selected on validation.

In [ ]:
def alarm_curve(match50: dict[str, ClassMatch], neg_max: np.ndarray, candidates: np.ndarray) -> pd.DataFrame:
    """Class-mean recall and precision, and the alarm rate, at each candidate threshold."""
    n_neg = len(neg_max)
    rows = []
    for t in candidates:
        per_class = {}
        for cls in CLASSES:
            m = match50[cls]
            used = m.score >= t
            tp = float(m.tp[used, 0].sum())
            fp = float(used.sum() - tp)
            per_class[cls] = {"recall": tp / max(1e-9, m.n_gt), "precision": tp / max(1e-9, tp + fp)}
        # A negative image fires when any class reaches the threshold.
        fired = int((neg_max >= t).any(axis=1).sum()) if n_neg else 0
        recall = float(np.mean([v["recall"] for v in per_class.values()]))
        precision = float(np.mean([v["precision"] for v in per_class.values()]))
        rows.append({
            "threshold": round(float(t), 2),
            "recall": round(recall, 5),
            "precision": round(precision, 5),
            "f1": round(2 * precision * recall / max(1e-9, precision + recall), 5),
            "far_per_1000": round(1000.0 * fired / max(1, n_neg), 3),
            "negatives_fired": fired,
            "recall_fire": round(per_class["fire"]["recall"], 5),
            "recall_smoke": round(per_class["smoke"]["recall"], 5),
        })
    return pd.DataFrame(rows)


def far_at_recall(curve: pd.DataFrame, target: float) -> float | None:
    """Alarm rate at a given recall, by linear interpolation along the threshold sweep."""
    d = curve.sort_values("recall")
    if target < d.recall.min() or target > d.recall.max():
        return None
    return round(float(np.interp(target, d.recall.values, d.far_per_1000.values)), 3)


CURVES: dict[str, pd.DataFrame] = {}
far_rows = []
for run in RUNS:
    curve = alarm_curve(MATCH50[(run.run_id, TEST_SPLIT)], NEG_MAX[(run.run_id, TEST_SPLIT)], THRESHOLD_SWEEP)
    CURVES[run.run_id] = curve
    row = {"run_id": run.run_id, "model": run.model, "seed": run.seed}
    for rt in FIXED_RECALLS:
        row[f"far_at_recall_{rt}"] = far_at_recall(curve, rt)
    row["max_reachable_recall"] = round(float(curve.recall.max()), 5)
    far_rows.append(row)
CURVE_TABLE = pd.concat([c.assign(run_id=r, model=RUN_BY_ID[r].model, seed=RUN_BY_ID[r].seed)
                         for r, c in CURVES.items()], ignore_index=True)
FAR_AT_RECALL = pd.DataFrame(far_rows)
save_table(CURVE_TABLE, "far_recall_curves")
save_table(FAR_AT_RECALL, "far_at_fixed_recall_per_run")

FIXED_RECALL_COLUMNS = [f"far_at_recall_{rt}" for rt in FIXED_RECALLS]
far_main = FAR_AT_RECALL[FAR_AT_RECALL.model.isin(ARCHITECTURES)]
FAR_AT_RECALL_MEAN = far_main.groupby("model")[FIXED_RECALL_COLUMNS].mean()
table8_points = []
for rt, column in zip(FIXED_RECALLS, FIXED_RECALL_COLUMNS):
    y = float(FAR_AT_RECALL_MEAN.loc["YOLO11n", column])
    r = float(FAR_AT_RECALL_MEAN.loc["ResNet18+FPN+FCOS", column])
    table8_points.append({"recall": rt, "yolo11n_mean": round(y, 2), "resnet_mean": round(r, 2),
                          "ratio": f"{max(y, r) / min(y, r):.1f}x"})
print(pd.DataFrame(table8_points).to_string(index=False))


def figure_far_vs_recall() -> None:
    d = CURVE_TABLE[CURVE_TABLE.model.isin(ARCHITECTURES)]
    means = d.groupby(["model", "threshold"], as_index=False).agg(
        recall=("recall", "mean"), far=("far_per_1000", "mean"))
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, 3.0))
        for i, (lab, g) in enumerate(means.groupby("model")):
            g = g.sort_values("recall")
            ax.plot(g.recall, g.far, LINE_STYLES[i % len(LINE_STYLES)], color=GREYS[i % len(GREYS)],
                    label=lab, markersize=3.4, linewidth=1.4, markevery=2)
        # Log scale on the vertical axis: on a linear axis the region the paper is about
        # (recall 0.5-0.8, 4-100 alarms) is flattened against the floor by the climb to 1,000
        # at the right-hand end.
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(PLAIN_NUMBER)
        ax.set_xlim(0.25, None)
        ax.set_xlabel("Recall")
        ax.set_ylabel("False alarms per 1,000 verified\nnegatives (log scale)")
        ax.legend(frameon=False, loc="upper left")
        ax.grid(True, which="both", alpha=0.2, linewidth=0.5)
        save_figure(fig, "Fig2.png")
    return means


FIG2_DATA = figure_far_vs_recall()

## 8. Expected cost and a reversal of ranking (Equation 2, Figure 3 and the point estimates of Table 10)

An operating point is scored by

$$C_{\mathrm{exp}} = r \times (1 - \mathrm{Recall}) + \mathrm{FAR}, \qquad r = \frac{C_{\mathrm{miss}}}{C_{\mathrm{false\text{-}alarm}}},$$

where FAR is the per-image alarm fraction (Equation 1 divided by 1,000) and Recall the class-mean recall. The ratio $r$ is swept over 1, 2, 5, 10, 25 and 50; these are parameters, not costs attached to any real agency or currency.

Three threshold policies are compared, each selected on the validation split and applied unchanged to the test split:

1. the fixed 0.25 that many papers use without comment;
2. one cost-optimal threshold shared by both classes;
3. separate cost-optimal thresholds for fire and smoke (361 combinations), the policy Table 10 reports.

Ties are broken towards the lowest threshold, the first one met in the sweep. Recall per class depends only on that class's threshold, so it is tabulated once per candidate threshold; the alarm indicator of each negative image is tabulated per class in the same way, and the 361 combinations then only combine table entries.

In [ ]:
def recall_and_far(match50: dict[str, ClassMatch], neg_max: np.ndarray,
                   thresholds: dict[str, float]) -> tuple[float, float]:
    """Class-mean recall and alarm fraction (0..1) for one pair of per-class thresholds."""
    recalls = []
    for cls in CLASSES:
        m = match50[cls]
        if m.n_gt == 0:
            continue
        used = m.score >= thresholds[cls]
        recalls.append(float(m.tp[used, 0].sum()) / m.n_gt)
    if len(neg_max) == 0:
        far = 0.0
    else:
        limits = np.array([thresholds[c] for c in CLASSES], dtype=np.float32)
        far = float((neg_max >= limits).any(axis=1).mean())
    return (float(np.mean(recalls)) if recalls else 0.0), far


def expected_cost(recall: float, far: float, ratio: float) -> float:
    """Equation 2 with C_false-alarm = 1, so `ratio` is C_miss / C_false-alarm."""
    return ratio * (1.0 - recall) + far


@dataclass
class CostLookup:
    candidates: list[float]
    recall: dict[str, list[float]]        # per class, per candidate threshold
    fired: dict[str, list[np.ndarray]]    # per class, per candidate: bool per negative image

    def recall_far(self, i_fire: int, i_smoke: int) -> tuple[float, float]:
        recall = float(np.mean([self.recall["fire"][i_fire], self.recall["smoke"][i_smoke]]))
        far = float((self.fired["fire"][i_fire] | self.fired["smoke"][i_smoke]).mean())
        return recall, far


def build_cost_lookup(match50: dict[str, ClassMatch], neg_max: np.ndarray,
                      candidates: np.ndarray) -> CostLookup:
    cands = [float(t) for t in candidates]
    recall = {cls: [float(match50[cls].tp[match50[cls].score >= t, 0].sum()) / match50[cls].n_gt
                    for t in cands] for cls in CLASSES}
    fired = {cls: [neg_max[:, j] >= np.float32(t) for t in cands] for j, cls in enumerate(CLASSES)}
    return CostLookup(cands, recall, fired)


cost_rows = []
for run in RUNS:
    m_val, m_test = MATCH50[(run.run_id, SELECTION_SPLIT)], MATCH50[(run.run_id, TEST_SPLIT)]
    neg_val, neg_test = NEG_MAX[(run.run_id, SELECTION_SPLIT)], NEG_MAX[(run.run_id, TEST_SPLIT)]
    lookup = build_cost_lookup(m_val, neg_val, THRESHOLD_SWEEP)
    n_cand = len(lookup.candidates)
    for ratio in COST_RATIOS:
        # Policy 1: the literature default, no selection.
        r_lit, f_lit = recall_and_far(m_test, neg_test, {c: LITERATURE_THRESHOLD for c in CLASSES})
        # Policy 2: one threshold for both classes, chosen on validation by cost.
        best_single, best_single_cost = None, float("inf")
        for i in range(n_cand):
            c = expected_cost(*lookup.recall_far(i, i), ratio)
            if c < best_single_cost:
                best_single_cost, best_single = c, lookup.candidates[i]
        r_single, f_single = recall_and_far(m_test, neg_test, {c: best_single for c in CLASSES})
        # Policy 3: separate thresholds for fire and smoke, chosen on validation by cost.
        best_pair, best_pair_cost = None, float("inf")
        for i in range(n_cand):
            for j in range(n_cand):
                c = expected_cost(*lookup.recall_far(i, j), ratio)
                if c < best_pair_cost:
                    best_pair_cost = c
                    best_pair = {"fire": lookup.candidates[i], "smoke": lookup.candidates[j]}
        r_pair, f_pair = recall_and_far(m_test, neg_test, best_pair)
        cost_lit = expected_cost(r_lit, f_lit, ratio)
        cost_pair = expected_cost(r_pair, f_pair, ratio)
        cost_rows.append({
            "run_id": run.run_id, "model": run.model, "seed": run.seed, "ratio": ratio,
            "threshold_literature": LITERATURE_THRESHOLD,
            "recall_literature": round(r_lit, 5), "far_per_1000_literature": round(1000 * f_lit, 3),
            "cost_literature": round(cost_lit, 5),
            "threshold_single": best_single,
            "recall_single": round(r_single, 5), "far_per_1000_single": round(1000 * f_single, 3),
            "cost_single": round(expected_cost(r_single, f_single, ratio), 5),
            "threshold_fire": best_pair["fire"], "threshold_smoke": best_pair["smoke"],
            "recall_per_class": round(r_pair, 5), "far_per_1000_per_class": round(1000 * f_pair, 3),
            "cost_per_class": round(cost_pair, 5),
            "saving_pct_vs_literature": round(100 * (cost_lit - cost_pair) / max(1e-9, cost_lit), 2),
        })
COSTS = pd.DataFrame(cost_rows)
save_table(COSTS, "expected_cost_all_policies")

cost_main = COSTS[COSTS.model.isin(ARCHITECTURES)]
COST_MEAN = cost_main.groupby(["ratio", "model"]).cost_per_class.mean().unstack("model")
map50_winner = max(ARCHITECTURES, key=lambda a: SEED_STATS[(a, "map50")][0])
table10_points = pd.DataFrame({
    "r": COST_MEAN.index,
    "yolo11n_mean": COST_MEAN["YOLO11n"].round(4).values,
    "resnet_mean": COST_MEAN["ResNet18+FPN+FCOS"].round(4).values,
    "lower_cost": [min(ARCHITECTURES, key=lambda a: COST_MEAN.loc[r, a]) for r in COST_MEAN.index],
    "map50_winner": map50_winner,
})
print(table10_points.to_string(index=False))
policy_means = cost_main.groupby(["model", "ratio"])[["cost_literature", "cost_single", "cost_per_class"]].mean()
print("\nMean expected cost per policy (fixed 0.25 / one threshold / per class):")
print(policy_means.round(4).to_string())

In [ ]:
# Boundary solutions: the stored confidence floor (0.05) is the lowest threshold the sweep can
# reach. Where the optimum sits on it, the true optimum may be lower, and the cost is an upper bound.
floor = STORED_CONFIDENCE_FLOOR
yolo_costs = COSTS[COSTS.model == "YOLO11n"]
resnet_costs = COSTS[COSTS.model == "ResNet18+FPN+FCOS"]
FLOOR_FACTS = {
    "yolo_fire_at_floor_everywhere": bool((yolo_costs.threshold_fire == floor).all()),
    "yolo_smoke_at_floor_for_r_ge_5": bool((yolo_costs[yolo_costs.ratio >= 5].threshold_smoke == floor).all()),
    "resnet_floor_cases": [
        (int(r.seed), int(r.ratio)) for r in resnet_costs.itertuples()
        if r.threshold_fire == floor or r.threshold_smoke == floor],
}
print(FLOOR_FACTS)


def figure_expected_cost() -> None:
    means = cost_main.groupby(["model", "ratio"], as_index=False).cost_per_class.mean()
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, 2.8))
        for i, (lab, g) in enumerate(means.groupby("model")):
            g = g.sort_values("ratio")
            ax.plot(g.ratio, g.cost_per_class, LINE_STYLES[i % len(LINE_STYLES)],
                    color=GREYS[i % len(GREYS)], label=lab, markersize=4, linewidth=1.4)
        ax.set_xscale("log")
        ax.set_xticks(list(COST_RATIOS))
        ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("Cost ratio r = cost of a miss : cost of a false alarm")
        ax.set_ylabel("Expected cost (lower is better)")
        ax.legend(frameon=False, loc="upper left")
        save_figure(fig, "Fig3.png")


figure_expected_cost()

## 9. Statistical comparison (Tables 8, 10 and 11)

Two bootstrap procedures are used, both over test images, both with 2,000 resamples, a 95 % percentile interval and seed 42.

**Per-run alarm-rate intervals (Table 11).** The quantity is a mean over images (an image either raised an alarm or did not), so the verified negatives are resampled with replacement and the alarm fraction is recomputed. mAP50 is not a per-image mean, so this simple bootstrap would not be valid for it; its uncertainty is reported through the seed spread instead.

**Paired intervals for Tables 8 and 10.** All 3,257 test images are resampled, and in each resample both detectors, all three seeds of each, are evaluated on the same multiset of images; the three seeds are averaged inside the resample, so the interval describes the three-seed mean the tables report. Thresholds are not re-selected inside a resample: the cost thresholds stay those chosen on validation, and the fixed-recall points are re-derived from each resampled curve, exactly as Section 7 derives them once. The difference between the detectors is formed within each resample, which is what makes the comparison paired; two overlapping per-model intervals would not show that the difference is zero. The two-sided p-value is twice the share of resampled differences on the other side of zero, with a floor of 1/2000.

Resampling is implemented with count weights (how often each image was drawn), which gives the same numbers as copying rows, much faster.

In [ ]:
def bootstrap_ci(values: np.ndarray, resamples: int = BOOTSTRAP_RESAMPLES, level: float = CI_LEVEL,
                 seed: int = RNG_SEED) -> dict[str, float]:
    """Percentile interval for the mean of a per-image quantity."""
    x = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(resamples, dtype=np.float64)
    for i in range(resamples):
        means[i] = x[rng.integers(0, len(x), len(x))].mean()
    tail = (1.0 - level) / 2.0
    return {"mean": float(x.mean()), "low": float(np.quantile(means, tail)),
            "high": float(np.quantile(means, 1.0 - tail)), "n": int(len(x))}


def paired_difference(a: np.ndarray, b: np.ndarray, resamples: int = BOOTSTRAP_RESAMPLES,
                      level: float = CI_LEVEL, seed: int = RNG_SEED) -> dict[str, Any]:
    """Bootstrap of the mean of a - b over the same images."""
    diff = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(resamples, dtype=np.float64)
    for i in range(resamples):
        means[i] = diff[rng.integers(0, len(diff), len(diff))].mean()
    tail = (1.0 - level) / 2.0
    mean = float(diff.mean())
    flipped = float((np.sign(means) != np.sign(mean)).mean()) if mean != 0 else 1.0
    low, high = float(np.quantile(means, tail)), float(np.quantile(means, 1.0 - tail))
    return {"difference": mean, "low": low, "high": high, "p_two_sided": min(1.0, 2.0 * flipped),
            "zero_inside": bool(low <= 0.0 <= high)}


gt_test = GT[TEST_SPLIT]
NEGATIVES_SORTED = sorted(sid for i, sid in enumerate(gt_test.ids) if gt_test.negative[i])
neg_positions = np.array([gt_test.index[s] for s in NEGATIVES_SORTED])
RUN_THRESHOLDS = RUN_METRICS.set_index("run_id").threshold
ALARM_VECTORS = {
    run.run_id: (MAX_SCORE[(run.run_id, TEST_SPLIT)][neg_positions] >= float(RUN_THRESHOLDS[run.run_id])).astype(np.float64)
    for run in RUNS
}

ci_rows = []
for run in RUNS:
    s = bootstrap_ci(ALARM_VECTORS[run.run_id])
    ci_rows.append({"run_id": run.run_id, "model": run.model, "seed": run.seed,
                    "threshold": float(RUN_THRESHOLDS[run.run_id]),
                    "far_per_1000": round(1000 * s["mean"], 3), "ci_low": round(1000 * s["low"], 3),
                    "ci_high": round(1000 * s["high"], 3), "negatives": s["n"]})
TABLE_11 = pd.DataFrame(ci_rows)
save_table(TABLE_11, "table11_far_bootstrap")

# Paired comparisons of the alarm rate on the same negatives (architecture-level vectors are the
# per-image mean over seeds, so architecture is compared with architecture, not seed with seed).
arch_vectors = {}
for name in ("YOLO11n", "ResNet18+FPN+FCOS", "No-negatives", "Size-matched"):
    arch_vectors[name] = np.mean([ALARM_VECTORS[r.run_id] for r in RUNS if r.model == name], axis=0)
FAR_PAIRS = [("Size-matched", "No-negatives"), ("Size-matched", "ResNet18+FPN+FCOS"),
             ("Size-matched", "YOLO11n"), ("No-negatives", "ResNet18+FPN+FCOS"),
             ("No-negatives", "YOLO11n"), ("ResNet18+FPN+FCOS", "YOLO11n")]
pair_rows = []
for a, b in FAR_PAIRS:
    u = paired_difference(arch_vectors[a], arch_vectors[b])
    pair_rows.append({"comparison": f"{a} - {b}", "difference_per_1000": round(1000 * u["difference"], 3),
                      "ci_low": round(1000 * u["low"], 3), "ci_high": round(1000 * u["high"], 3),
                      "p_two_sided": round(u["p_two_sided"], 4), "zero_inside": u["zero_inside"]})
FAR_PAIRED = pd.DataFrame(pair_rows)
save_table(FAR_PAIRED, "far_paired_differences")
print(TABLE_11.to_string(index=False))
print()
print(FAR_PAIRED.to_string(index=False))

In [ ]:
def per_image_inputs(gt: GroundTruth, match50: dict[str, ClassMatch], sweep: np.ndarray) -> dict[str, Any]:
    """Everything a resample needs, computed once per run: per image and threshold, the number of
    true positives that survive; per image, the ground-truth count and whether it fires."""
    n = len(gt.ids)
    tp, ngt, fires = {}, {}, {}
    for cls in CLASSES:
        m = match50[cls]
        counts = np.zeros((n, len(sweep)), dtype=np.float64)
        top = np.full(n, -1.0)
        if len(m.img):
            starts = np.flatnonzero(np.r_[True, m.img[1:] != m.img[:-1]])
            survives = (m.score[:, None] >= sweep[None, :]) & (m.tp[:, :1] > 0)
            counts[m.img[starts]] = np.add.reduceat(survives.astype(np.float64), starts, axis=0)
            top[m.img[starts]] = np.maximum.reduceat(m.score.astype(np.float64), starts)
        tp[cls] = counts
        ngt[cls] = m.n_gt_per_image.astype(np.float64)
        fires[cls] = top[:, None] >= sweep[None, :]
    fires_any = np.zeros_like(fires[CLASSES[0]])
    for cls in CLASSES:
        fires_any |= fires[cls]
    return {"n": n, "tp": tp, "ngt": ngt, "negative": gt.negative.copy(), "fires": fires,
            "fires_any": fires_any, "sweep": np.asarray(sweep, dtype=np.float64)}


def resampled_curve(b: dict[str, Any], weights: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Class-mean recall and alarm fraction at every sweep threshold, for one resample."""
    recalls = []
    for cls in CLASSES:
        n_gt = weights @ b["ngt"][cls]
        if n_gt <= 0:
            continue
        recalls.append((weights @ b["tp"][cls]) / n_gt)
    recall = np.mean(recalls, axis=0) if recalls else np.zeros(len(b["sweep"]))
    neg_weights = weights * b["negative"]
    total_neg = neg_weights.sum()
    if total_neg <= 0:
        return recall, np.zeros(len(b["sweep"]))
    return recall, (neg_weights @ b["fires_any"]) / total_neg


def resampled_recall_far(b: dict[str, Any], weights: np.ndarray, thr: dict[str, float]) -> tuple[float, float]:
    """Class-mean recall and alarm fraction at one pair of per-class thresholds, for one resample."""
    recalls = []
    for cls in CLASSES:
        n_gt = weights @ b["ngt"][cls]
        if n_gt <= 0:
            continue
        j = int(np.argmin(np.abs(b["sweep"] - thr[cls])))
        recalls.append(float((weights @ b["tp"][cls][:, j]) / n_gt))
    neg_weights = weights * b["negative"]
    total_neg = neg_weights.sum()
    if total_neg <= 0:
        return (float(np.mean(recalls)) if recalls else 0.0), 0.0
    fired = np.zeros(b["n"], dtype=bool)
    for cls in CLASSES:
        j = int(np.argmin(np.abs(b["sweep"] - thr[cls])))
        fired |= b["fires"][cls][:, j]
    return (float(np.mean(recalls)) if recalls else 0.0), float((neg_weights @ fired) / total_neg)


PER_IMAGE = {run.run_id: per_image_inputs(gt_test, MATCH50[(run.run_id, TEST_SPLIT)], THRESHOLD_SWEEP)
             for run in MAIN_RUNS}
COST_THRESHOLDS = {
    run.run_id: [(float(r.ratio), {"fire": float(r.threshold_fire), "smoke": float(r.threshold_smoke)})
                 for r in COSTS[COSTS.run_id == run.run_id].itertuples()]
    for run in MAIN_RUNS
}

t0 = time.time()
n_images = len(gt_test.ids)
rng = np.random.default_rng(RNG_SEED)
FAR_SAMPLES: dict[tuple[str, float], list[list[float]]] = {}
COST_SAMPLES: dict[tuple[str, int], list[list[float]]] = {}
for it in range(BOOTSTRAP_RESAMPLES):
    weights = np.bincount(rng.integers(0, n_images, n_images), minlength=n_images).astype(np.float64)
    far_now: dict[tuple[str, float], list[float]] = {}
    cost_now: dict[tuple[str, int], list[float]] = {}
    for run in MAIN_RUNS:   # the same resample for every run, so seed means form inside it
        b = PER_IMAGE[run.run_id]
        recall, far = resampled_curve(b, weights)
        order = np.argsort(recall)
        for rt in FIXED_RECALLS:
            far_now.setdefault((run.model, rt), []).append(
                float(np.interp(rt, recall[order], 1000.0 * far[order])))
        for ratio, thr in COST_THRESHOLDS[run.run_id]:
            rr, ff = resampled_recall_far(b, weights, thr)
            cost_now.setdefault((run.model, int(ratio)), []).append(ratio * (1.0 - rr) + ff)
    for k, v in far_now.items():
        FAR_SAMPLES.setdefault(k, []).append(v)
    for k, v in cost_now.items():
        COST_SAMPLES.setdefault(k, []).append(v)
print(f"paired bootstrap: {BOOTSTRAP_RESAMPLES} resamples of {n_images} test images in {time.time() - t0:.1f} s")


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    tail = (1.0 - CI_LEVEL) / 2.0
    return float(np.quantile(values, tail)), float(np.quantile(values, 1.0 - tail))


def seed_mean_samples(samples: dict, key: tuple) -> np.ndarray:
    return np.asarray([np.mean(v) for v in samples[key]])


seedmean_far_rows = []
for (model, rt) in sorted(FAR_SAMPLES):
    s = seed_mean_samples(FAR_SAMPLES, (model, rt))
    low, high = percentile_interval(s)
    seedmean_far_rows.append({"model": model, "recall": rt, "far_per_1000": round(float(s.mean()), 3),
                              "ci_low": round(low, 3), "ci_high": round(high, 3)})
seedmean_cost_rows = []
for (model, ratio) in sorted(COST_SAMPLES):
    s = seed_mean_samples(COST_SAMPLES, (model, ratio))
    low, high = percentile_interval(s)
    seedmean_cost_rows.append({"model": model, "ratio": ratio, "cost": round(float(s.mean()), 4),
                               "ci_low": round(low, 4), "ci_high": round(high, 4)})
SEEDMEAN_FAR_CI = pd.DataFrame(seedmean_far_rows)
SEEDMEAN_COST_CI = pd.DataFrame(seedmean_cost_rows)
save_table(SEEDMEAN_FAR_CI, "far_at_fixed_recall_seed_mean_intervals")
save_table(SEEDMEAN_COST_CI, "expected_cost_seed_mean_intervals")


def paired_from_samples(samples: dict, key_a: tuple, key_b: tuple) -> dict[str, float]:
    d = seed_mean_samples(samples, key_a) - seed_mean_samples(samples, key_b)
    low, high = percentile_interval(d)
    side = min((d <= 0).mean(), (d >= 0).mean())
    return {"difference": float(d.mean()), "low": low, "high": high,
            "p_two_sided": min(1.0, 2.0 * max(side, 1.0 / BOOTSTRAP_RESAMPLES)),
            "zero_inside": bool(low <= 0 <= high)}


paired_rows = []
for rt in FIXED_RECALLS:
    u = paired_from_samples(FAR_SAMPLES, ("YOLO11n", rt), ("ResNet18+FPN+FCOS", rt))
    paired_rows.append({"table": "Table 8", "quantity": f"far_per_1000@recall={rt}", "key": f"far@{rt}",
                        "difference": round(u["difference"], 3), "ci_low": round(u["low"], 3),
                        "ci_high": round(u["high"], 3), "p_two_sided": round(u["p_two_sided"], 4),
                        "zero_inside": u["zero_inside"]})
for ratio in COST_RATIOS:
    u = paired_from_samples(COST_SAMPLES, ("YOLO11n", ratio), ("ResNet18+FPN+FCOS", ratio))
    paired_rows.append({"table": "Table 10", "quantity": f"expected_cost@r={ratio}", "key": f"cost@{ratio}",
                        "difference": round(u["difference"], 4), "ci_low": round(u["low"], 4),
                        "ci_high": round(u["high"], 4), "p_two_sided": round(u["p_two_sided"], 4),
                        "zero_inside": u["zero_inside"]})
PAIRED = pd.DataFrame(paired_rows)
save_table(PAIRED, "paired_differences_yolo_minus_resnet")


def signed(x: float, dec: int) -> str:
    return f"{x:+.{dec}f}"


paired_by_key = PAIRED.set_index("key")
TABLE_8 = pd.DataFrame([{
    **p,
    "paired_difference": paired_by_key.loc[f"far@{p['recall']}", "difference"],
    "ci_low": paired_by_key.loc[f"far@{p['recall']}", "ci_low"],
    "ci_high": paired_by_key.loc[f"far@{p['recall']}", "ci_high"],
    "p_two_sided": paired_by_key.loc[f"far@{p['recall']}", "p_two_sided"],
} for p in table8_points])
TABLE_8["as_printed"] = [f"{signed(r.paired_difference, 2)} [{signed(r.ci_low, 2)}, {signed(r.ci_high, 2)}]"
                         for r in TABLE_8.itertuples()]
save_table(TABLE_8, "table08_far_at_fixed_recall")

TABLE_10 = table10_points.copy()
TABLE_10["paired_difference"] = [paired_by_key.loc[f"cost@{r}", "difference"] for r in TABLE_10.r]
TABLE_10["ci_low"] = [paired_by_key.loc[f"cost@{r}", "ci_low"] for r in TABLE_10.r]
TABLE_10["ci_high"] = [paired_by_key.loc[f"cost@{r}", "ci_high"] for r in TABLE_10.r]
TABLE_10["as_printed"] = [f"{signed(r.paired_difference, 4)} [{signed(r.ci_low, 4)}, {signed(r.ci_high, 4)}]"
                          for r in TABLE_10.itertuples()]
save_table(TABLE_10, "table10_expected_cost")
print(TABLE_8.to_string(index=False))
print()
print(TABLE_10.to_string(index=False))

## 10. Error analysis (Table 12 and Figure 5)

Two kinds of error are grouped by number rather than by eye.

**False alarms on verified negatives**, grouped by the confidence of the strongest box. An alarm raised at 0.75 cannot be removed by nudging the threshold up a little; one raised at 0.27 can.

**Missed objects**, grouped by the fraction of the image the ground-truth box covers. Section 4 suggested that fire is harder because fire boxes are small; if so, the objects a detector misses should be systematically smaller than those it catches. A ground-truth box counts as caught when any retained detection, of either class, overlaps it at IoU 0.50 or more; the question here is about size, not about the label.

Visual categories such as sunset, haze or headlights need a person to look at the images and are not claimed. The error listing written below names the images for that inspection: up to 400 error instances, alarms first, then missed objects smaller than 1 % of the image, taken in manifest order across runs (in practice all from YOLO11n seed 42, the first run).

In [ ]:
SCORE_BANDS = [(0.0, 0.3, "near threshold (< 0.30)"), (0.3, 0.5, "moderate (0.30-0.50)"),
               (0.5, 0.7, "high (0.50-0.70)"), (0.7, 1.01, "very confident (>= 0.70)")]
SIZE_BANDS = [(0.0, 0.001, "very small (< 0.1% of image)"), (0.001, 0.01, "small (0.1-1%)"),
              (0.01, 0.1, "medium (1-10%)"), (0.1, 1.01, "large (>= 10%)")]
ERROR_LISTING_LIMIT = 400


def band(value: float, bands: list[tuple[float, float, str]]) -> str:
    for low, high, name in bands:
        if low <= value < high:
            return name
    return bands[-1][2]


error_rows, error_listing = [], []
neg_idx = np.flatnonzero(gt_test.negative)
for run in RUNS:
    thr = float(RUN_THRESHOLDS[run.run_id])
    pred = PRED[(run.run_id, TEST_SPLIT)]
    max_s = MAX_SCORE[(run.run_id, TEST_SPLIT)]
    base = {"run_id": run.run_id, "model": run.model, "seed": run.seed, "threshold": thr}

    # False alarms on verified negatives, by the confidence of the strongest box.
    alarm_counts = {name: 0 for *_, name in SCORE_BANDS}
    for i in neg_idx:
        if max_s[i] < thr:
            continue
        alarm_counts[band(float(max_s[i]), SCORE_BANDS)] += 1
        if len(error_listing) < ERROR_LISTING_LIMIT:
            sl = pred.image_slice(int(i))
            scores = pred.score[sl]
            above = scores >= thr
            k = int(np.argmax(np.where(above, scores, -np.inf)))
            error_listing.append({**base, "kind": "false alarm", "sample_id": gt_test.ids[i],
                                  "top_score": round(float(scores[k]), 4),
                                  "class_of_top_box": CLASSES[int(pred.cls[sl][k])],
                                  "boxes_above_threshold": int(above.sum()), "missed_object_area": None})
    for name, count in alarm_counts.items():
        error_rows.append({**base, "kind": "false alarm on a verified negative", "category": name,
                           "count": count, "out_of": len(neg_idx),
                           "pct": round(100 * count / max(1, len(neg_idx)), 3)})

    # Missed objects, by size.
    missed = {name: 0 for *_, name in SIZE_BANDS}
    caught = {name: 0 for *_, name in SIZE_BANDS}
    missed_areas, caught_areas = [], []
    for i, objs in enumerate(gt_test.objects):
        if not objs:
            continue
        gt_boxes = np.asarray([bb for bb, _ in objs], dtype=np.float32).reshape(-1, 4)
        image_area = max(1.0, int(gt_test.width[i]) * int(gt_test.height[i]))
        sl = pred.image_slice(i)
        keep = pred.score[sl] >= thr
        if keep.any():
            hit = (iou_matrix(pred.box[sl][keep], gt_boxes) >= IOU_MAIN).any(axis=0)
        else:
            hit = np.zeros(len(gt_boxes), dtype=bool)
        for j, b in enumerate(gt_boxes):
            area = float((b[2] - b[0]) * (b[3] - b[1])) / image_area
            name = band(area, SIZE_BANDS)
            if hit[j]:
                caught[name] += 1
                caught_areas.append(area)
            else:
                missed[name] += 1
                missed_areas.append(area)
                if len(error_listing) < ERROR_LISTING_LIMIT and area < 0.01:
                    error_listing.append({**base, "kind": "missed object", "sample_id": gt_test.ids[i],
                                          "top_score": None, "class_of_top_box": None,
                                          "boxes_above_threshold": None,
                                          "missed_object_area": round(area, 6)})
    for name in missed:
        total = missed[name] + caught[name]
        error_rows.append({**base, "kind": "missed object, by size", "category": name,
                           "count": missed[name], "out_of": total,
                           "pct": round(100 * missed[name] / max(1, total), 3)})
    med_missed = float(np.median(missed_areas)) if missed_areas else float("nan")
    med_caught = float(np.median(caught_areas)) if caught_areas else float("nan")
    error_rows.append({**base, "kind": "size summary", "category": "median area: missed vs caught",
                       "count": round(med_missed, 6), "out_of": round(med_caught, 6),
                       "pct": round(med_caught / max(1e-9, med_missed), 2)})
    print(f"  {run.run_id:22s} thr {thr:.2f} | {sum(alarm_counts.values()):3d} alarms on "
          f"{len(neg_idx)} negatives | median area missed {med_missed:.5f} vs caught {med_caught:.5f}")

ERRORS = pd.DataFrame(error_rows)
ERROR_LISTING = pd.DataFrame(error_listing)
save_table(ERRORS, "error_analysis")
save_table(ERROR_LISTING, "error_listing")

SIZE_SUMMARY = ERRORS[ERRORS.kind == "size summary"].rename(
    columns={"count": "median_missed", "out_of": "median_caught", "pct": "caught_over_missed"})
t12 = []
for arch in ARCHITECTURES:
    g = SIZE_SUMMARY[SIZE_SUMMARY.model == arch]
    t12.append({
        "model": arch,
        "seeds": " / ".join(str(s) for s in g.seed),
        "median_area_missed": " / ".join(f"{v:.4f}" for v in g.median_missed),
        "median_area_caught": " / ".join(f"{v:.4f}" for v in g.median_caught),
        "caught_over_missed": " / ".join(f"{v:.1f}x" for v in g.caught_over_missed),
    })
TABLE_12 = pd.DataFrame(t12)
save_table(TABLE_12, "table12_missed_vs_caught")
print()
print(TABLE_12.to_string(index=False))
print(f"\nerror listing: {len(ERROR_LISTING)} instances over {ERROR_LISTING.sample_id.nunique()} distinct images "
      f"(runs: {', '.join(ERROR_LISTING.run_id.unique())})")
assert (SIZE_SUMMARY[SIZE_SUMMARY.model.isin(ARCHITECTURES)].caught_over_missed > 1).all()


def figure_missed_object_size() -> None:
    d = SIZE_SUMMARY[SIZE_SUMMARY.model.isin(ARCHITECTURES)]
    means = d.groupby("model", as_index=False).agg(missed=("median_missed", "mean"),
                                                   caught=("median_caught", "mean"))
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, 2.6))
        x = range(len(means))
        width = 0.36
        ax.bar([i - width / 2 for i in x], means.missed * 100, width, label="objects missed",
               color="0.30", edgecolor="black", linewidth=0.7)
        ax.bar([i + width / 2 for i in x], means.caught * 100, width, label="objects caught",
               color="0.78", edgecolor="black", linewidth=0.7)
        ax.set_xticks(list(x))
        ax.set_xticklabels(list(means.model))
        ax.set_ylabel("Median box area (% of image)")
        top = float(max(means.missed.max(), means.caught.max())) * 100
        ax.set_ylim(0, top * 1.35)
        ax.legend(frameon=False, loc="upper right")
        for i, r in enumerate(means.itertuples(index=False)):
            ax.text(i + width / 2, r.caught * 100, f"{r.caught / max(1e-9, r.missed):.1f}\u00d7 larger",
                    ha="center", va="bottom")
        save_figure(fig, "Fig5.png")
    return means


FIG5_DATA = figure_missed_object_size()

## 11. Confidence calibration (Figure 6)

Comparing two detectors at one fixed threshold, such as 0.25, assumes that a given score means the same thing in both. Calibration tests that assumption. Every stored detection carries a score and a correct/incorrect label at IoU 0.50 (the same matching as everywhere else); detections are grouped into ten score bins, and in each bin the mean score is compared with the fraction of detections that are correct:

$$\mathrm{ECE} = \sum_b \frac{n_b}{N}\,\bigl|\,\text{fraction correct}_b - \text{mean score}_b\,\bigr|$$

The maximum calibration error (MCE) is the largest of those gaps. Detections below 0.05 were never stored, so the lowest bin is incomplete and the ECE describes the observable range only.

In [ ]:
def ece_and_bins(scores: np.ndarray, correct: np.ndarray, n_bins: int = CALIBRATION_BINS):
    if len(scores) == 0:
        return float("nan"), float("nan"), []
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    # right=True puts a score that sits exactly on an edge into the lower bin, so 1.0 stays in bin 9.
    member = np.clip(np.digitize(scores, edges[1:-1], right=True), 0, n_bins - 1)
    bins, ece, mce, n = [], 0.0, 0.0, len(scores)
    for b in range(n_bins):
        sel = member == b
        n_b = int(sel.sum())
        if n_b == 0:
            bins.append({"bin": b, "from": round(float(edges[b]), 2), "to": round(float(edges[b + 1]), 2),
                         "n": 0, "mean_score": None, "fraction_correct": None})
            continue
        mean_score = float(scores[sel].mean())
        frac = float(correct[sel].mean())
        gap = abs(frac - mean_score)
        ece += (n_b / n) * gap
        mce = max(mce, gap)
        bins.append({"bin": b, "from": round(float(edges[b]), 2), "to": round(float(edges[b + 1]), 2),
                     "n": n_b, "mean_score": round(mean_score, 5), "fraction_correct": round(frac, 5)})
    return ece, mce, bins


calibration_rows, CALIBRATION_BINS_BY_RUN = [], {}
for run in MAIN_RUNS:
    match50 = MATCH50[(run.run_id, TEST_SPLIT)]
    scores_all = np.concatenate([match50[c].score for c in CLASSES])
    correct_all = np.concatenate([match50[c].tp[:, 0] for c in CLASSES])
    ece, mce, bins = ece_and_bins(scores_all, correct_all)
    CALIBRATION_BINS_BY_RUN[run.run_id] = bins
    calibration_rows.append({"run_id": run.run_id, "model": run.model, "seed": run.seed, "class": "all",
                             "detections": int(len(scores_all)),
                             "mean_score": round(float(scores_all.mean()), 5),
                             "fraction_correct": round(float(correct_all.mean()), 5),
                             "ece": round(ece, 5), "mce": round(mce, 5)})
    for cls in CLASSES:
        e, mc, _ = ece_and_bins(match50[cls].score, match50[cls].tp[:, 0])
        calibration_rows.append({"run_id": run.run_id, "model": run.model, "seed": run.seed, "class": cls,
                                 "detections": int(len(match50[cls].score)),
                                 "mean_score": round(float(match50[cls].score.mean()), 5),
                                 "fraction_correct": round(float(match50[cls].tp[:, 0].mean()), 5),
                                 "ece": round(e, 5), "mce": round(mc, 5)})
CALIBRATION = pd.DataFrame(calibration_rows)
save_table(CALIBRATION, "calibration")
CALIBRATION_BIN_TABLE = pd.DataFrame([{"run_id": r, **b} for r, bins in CALIBRATION_BINS_BY_RUN.items()
                                      for b in bins])
save_table(CALIBRATION_BIN_TABLE, "calibration_bins")
print(CALIBRATION[CALIBRATION["class"] == "all"].to_string(index=False))


def figure_calibration() -> None:
    colours = {"YOLO11n": "#c0392b", "ResNet18+FPN+FCOS": "#2471a3"}
    # Seeds differ by marker and dash pattern, so each panel also reads in greyscale.
    seed_style = {42: ("o", "-"), 1337: ("s", "--"), 2024: ("^", ":")}
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, 2.9), sharey=True)
        for k, (ax, label) in enumerate(zip(axes, sorted({r.model for r in MAIN_RUNS}))):
            ax.plot([0, 1], [0, 1], color="0.45", lw=0.9, ls=(0, (2, 2)), label="perfect calibration")
            for run in MAIN_RUNS:
                if run.model != label:
                    continue
                marker, line = seed_style[run.seed]
                populated = [b for b in CALIBRATION_BINS_BY_RUN[run.run_id] if b["n"] > 0]
                ax.plot([b["mean_score"] for b in populated], [b["fraction_correct"] for b in populated],
                        marker=marker, ls=line, ms=3.5, lw=1.1, mfc="none", color=colours.get(label, "#333"),
                        label=f"seed {run.seed}")
            sub = CALIBRATION[(CALIBRATION.model == label) & (CALIBRATION["class"] == "all")]
            ax.set_title(f"({'ab'[k]}) {label}\nECE {sub.ece.mean():.3f} \u00b1 {sub.ece.std(ddof=1):.3f}")
            ax.set_xlabel("Mean confidence in bin")
            if k == 0:
                ax.set_ylabel("Fraction correct (IoU 0.50)")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        # One legend for both panels, keyed by marker and dash pattern only (colour just tells the
        # detectors apart), in the lower right of panel (b), which lies below the diagonal at high
        # confidence where YOLO11n has no points.
        from matplotlib.lines import Line2D
        handles = [Line2D([], [], color="0.45", lw=0.9, ls=(0, (2, 2)))] + [
            Line2D([], [], color="black", marker=m, ls=ls, ms=3.5, lw=1.1, mfc="none")
            for m, ls in seed_style.values()]
        axes[1].legend(handles, ["perfect calibration"] + [f"seed {s}" for s in seed_style],
                       loc="lower right", frameon=False, handlelength=2.6)
        save_figure(fig, "Fig6.png")


figure_calibration()

## 12. Qualitative examples (Figure 7)

Three images, one for each behaviour the paper measures, all from YOLO11n seed 42 at its validation-selected threshold of 0.25. Picking examples by eye invites picking the most convincing ones, so each panel is chosen by a rule fixed in advance, and two of the three rules deliberately pick the worst case:

* (a) **fire detected**: the fire box with the highest IoU among those covering at least 2 % of the image;
* (b) **fire missed**: the smallest missed fire box that is the image's only annotation and covers between 0.05 % and 1 % of it (the lower bound only keeps the box visible in print);
* (c) **false alarm on a verified negative**: the highest-scoring alarm.

Human annotations are drawn with a dashed green line and detections with a solid red line (the figure carries no caption of its own; the paper's caption states this), so the two remain distinguishable for readers with red-green colour-vision deficiency and in greyscale print. The figure in the original analysis used colour alone; the selected images, boxes and numbers are unchanged.

The selection needs only the manifest and the predictions, so it is always computed and checked. Drawing the figure needs the D-Fire images (`FIRE_DFIRE_IMAGES`); without them the figure is skipped. The curated copy used by the authors capped the long image side at 1,280 pixels, so some original D-Fire images are larger than the manifest says; such images are resized to the manifest size before drawing, which is exact because D-Fire boxes are stored in normalised coordinates.

In [ ]:
QUALITATIVE_RUN = "yolo11n_seed42"
QUALITATIVE_THRESHOLD = 0.25
MISS_AREA_WINDOW = (0.0005, 0.01)
GREEN, RED = "#27ae60", "#e74c3c"
FIG7_HEIGHT = 1.36   # three photographs side by side at the full text width


def select_examples(gt: GroundTruth, pred: Predictions) -> dict[str, Any]:
    caught, missed, alarms = [], [], []
    for i, sid in enumerate(gt.ids):
        if not gt.known[i]:
            continue
        image_area = float(gt.width[i]) * float(gt.height[i])
        if image_area <= 0:
            continue
        sl = pred.image_slice(i)
        dets = [{"bbox": pred.box[k].tolist(), "box32": pred.box[k], "score": float(pred.score[k]),
                 "cls": CLASSES[int(pred.cls[k])]}
                for k in range(sl.start, sl.stop) if pred.score[k] >= QUALITATIVE_THRESHOLD]
        objs = gt.objects[i]
        if not objs:                                  # (c) verified negative that raised an alarm
            if dets:
                top = max(dets, key=lambda d: d["score"])
                alarms.append((top["score"], sid, top))
            continue
        fires = [bb for bb, c in objs if c == "fire"]
        if not fires:
            continue
        fire_dets = [d for d in dets if d["cls"] == "fire"]
        only_annotation = len(objs) == 1
        for bbox in fires:
            frac = ((bbox[2] - bbox[0]) * (bbox[3] - bbox[1])) / image_area
            if fire_dets:
                iou = iou_matrix(np.asarray([d["box32"] for d in fire_dets], dtype=np.float32),
                                 np.asarray([bbox], dtype=np.float32))[:, 0]
                k = int(np.argmax(iou))
                if iou[k] >= 0.5:
                    if frac >= 0.02:                  # (a) large enough to be seen in print
                        caught.append((float(iou[k]), sid, bbox, fire_dets[k]))
                    continue
            if only_annotation and MISS_AREA_WINDOW[0] <= frac <= MISS_AREA_WINDOW[1]:
                missed.append((frac, sid, bbox))      # (b)
    return {"caught": max(caught, key=lambda x: x[0]), "missed": min(missed, key=lambda x: x[0]),
            "alarm": max(alarms, key=lambda x: x[0])}


EXAMPLES = select_examples(gt_test, PRED[(QUALITATIVE_RUN, TEST_SPLIT)])
iou_a, sid_a, gt_a, det_a = EXAMPLES["caught"]
frac_b, sid_b, gt_b = EXAMPLES["missed"]
score_c, sid_c, det_c = EXAMPLES["alarm"]
QUALITATIVE = pd.DataFrame([
    {"panel": "a. fire detected", "sample_id": sid_a, "rule": "highest IoU, box >= 2% of image",
     "value": round(iou_a, 4), "score": round(det_a["score"], 4)},
    {"panel": "b. fire missed", "sample_id": sid_b,
     "rule": f"smallest missed fire box in {MISS_AREA_WINDOW[0]}-{MISS_AREA_WINDOW[1]} of the image, sole annotation",
     "value": round(frac_b, 6), "score": None},
    {"panel": "c. false alarm", "sample_id": sid_c, "rule": "highest-scoring alarm on a verified negative",
     "value": round(score_c, 4), "score": round(score_c, 4)},
])
save_table(QUALITATIVE, "figure7_selected_examples")
print(QUALITATIVE.to_string(index=False))


def load_image_at_manifest_size(sid: str) -> np.ndarray | None:
    if DFIRE_IMAGE_DIR is None:
        return None
    i = gt_test.index[sid]
    path = DFIRE_IMAGE_DIR / gt_test.image_path[i]
    if not path.is_file():
        return None
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert("RGB")
        size = (int(gt_test.width[i]), int(gt_test.height[i]))
        if im.size != size:
            im = im.resize(size, Image.BILINEAR)
        return np.asarray(im)


ANNOTATION_STYLE = {"colour": GREEN, "linestyle": (0, (3, 1.5))}   # dashed: human annotation
DETECTION_STYLE = {"colour": RED, "linestyle": "-"}                 # solid: detection


def draw_box(ax, bbox, colour, label, with_label=True, linestyle="-", label_below=False) -> None:
    x1, y1, x2, y2 = bbox
    ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.5, edgecolor=colour,
                                    linestyle=linestyle))
    if with_label:
        # Annotation labels go under the box and detection labels above it, so the two never cover
        # each other when the boxes nearly coincide. A box touching the image edge gets its label on
        # the inside, so the label never runs into the panel title.
        height = abs(ax.get_ylim()[0] - ax.get_ylim()[1])
        if label_below:
            y, va = (y2 + 4, "top") if y2 < 0.85 * height else (y2 - 4, "bottom")
        else:
            y, va = (y1 - 4, "baseline") if y1 > 0.15 * height else (y1 + 4, "top")
        ax.text(x1, y, label, color="white", va=va,
                bbox=dict(facecolor=colour, edgecolor="none", pad=1.0))


def zoom_inset(ax, image, bbox, colour, factor=8.0, linestyle="-", interpolation="antialiased") -> None:
    """Magnified inset for a box too small to see on the printed page."""
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    half = max(x2 - x1, y2 - y1, 8.0) * factor / 2.0
    h, w = image.shape[0], image.shape[1]
    a, b = max(0, int(cx - half)), min(w, int(cx + half))
    c, d = max(0, int(cy - half)), min(h, int(cy + half))
    inset = ax.inset_axes([0.60, 0.50, 0.38, 0.46])
    inset.imshow(image[c:d, a:b], interpolation=interpolation)
    inset.add_patch(mpatches.Rectangle((x1 - a, y1 - c), x2 - x1, y2 - y1, fill=False, lw=1.6, edgecolor=colour,
                                       linestyle=linestyle))
    inset.set_xticks([])
    inset.set_yticks([])
    for spine in inset.spines.values():
        spine.set_edgecolor(colour)
        spine.set_linewidth(1.2)
    inset.text(0.96, 0.05, f"{factor:.0f}\u00d7", transform=inset.transAxes, ha="right", va="bottom",
               color="white", bbox=dict(facecolor=colour, edgecolor="none", pad=1.0))


def draw_qualitative(images: dict[str, np.ndarray], interpolation: str) -> plt.Figure:
    """The three panels. `interpolation="none"` hands each photograph to a vector backend unresampled,
    so the PDF embeds it at its native resolution; the PNG uses matplotlib's default smoothing."""
    fig, axes = plt.subplots(1, 3, figsize=(FIGURE_WIDTH, FIG7_HEIGHT))
    titles = [
        f"(a) Fire detected\nIoU {iou_a:.2f}, conf. {det_a['score']:.2f}",
        f"(b) Fire missed\nbox = {100 * frac_b:.2f}% of image",
        f"(c) False alarm\n'{det_c['cls']}', conf. {score_c:.2f}",
    ]
    panels = [
        (sid_a, [(gt_a, "ground truth")], [(det_a["bbox"], f"detected {det_a['score']:.2f}")]),
        (sid_b, [(gt_b, "ground truth, not detected")], []),
        (sid_c, [], [(det_c["bbox"], f"{det_c['cls']} {score_c:.2f}")]),
    ]
    ann, det = ANNOTATION_STYLE, DETECTION_STYLE
    for n, (ax, title, (sid, gt_boxes, det_boxes)) in enumerate(zip(axes, titles, panels)):
        image = images[sid]
        ax.imshow(image, interpolation=interpolation)
        small = n == 1   # panel (b): the box is too small to carry a label
        # Detections first, annotations on top: where the two boxes nearly coincide (panel a), the
        # green dashes stay visible over the solid red line.
        for bbox, text in det_boxes:
            draw_box(ax, bbox, det["colour"], text, linestyle=det["linestyle"])
        for bbox, text in gt_boxes:
            draw_box(ax, bbox, ann["colour"], text, with_label=not small, linestyle=ann["linestyle"],
                     label_below=True)
            if small:
                zoom_inset(ax, image, bbox, ann["colour"], linestyle=ann["linestyle"], interpolation=interpolation)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


def figure_qualitative() -> dict[str, Any] | None:
    images = {sid: load_image_at_manifest_size(sid) for sid in (sid_a, sid_b, sid_c)}
    if any(img is None for img in images.values()):
        print("Figure 7 skipped: set FIRE_DFIRE_IMAGES to the D-Fire root (the folder containing "
              "train/images and test/images) to draw it. The selected examples above are still checked.")
        return None
    with plt.rc_context(PHOTO_STYLE):
        fig = draw_qualitative(images, interpolation="antialiased")
    # Record how the boxes were actually drawn, for the check in Section 15.
    all_axes = [a for ax in fig.axes for a in (ax, *ax.child_axes)]   # include the zoom inset
    boxes = [p for a in all_axes for p in a.patches if isinstance(p, mpatches.Rectangle) and not p.get_fill()]
    drawn = {
        "annotation_boxes": sum(p.get_edgecolor()[:3] == matplotlib.colors.to_rgb(GREEN) for p in boxes),
        "annotation_boxes_dashed": sum(p.get_edgecolor()[:3] == matplotlib.colors.to_rgb(GREEN)
                                       and p.get_linestyle() != "-" for p in boxes),
        "detection_boxes": sum(p.get_edgecolor()[:3] == matplotlib.colors.to_rgb(RED) for p in boxes),
        "detection_boxes_solid": sum(p.get_edgecolor()[:3] == matplotlib.colors.to_rgb(RED)
                                     and p.get_linestyle() == "-" for p in boxes),
        "image_sizes": [img.shape[:2] for img in images.values()],
    }
    with plt.rc_context(PHOTO_STYLE):
        save_figure(fig, "Fig7.png", formats=("png",))
        save_figure(draw_qualitative(images, interpolation="none"), "Fig7.pdf", formats=("pdf",))
    return drawn


FIG7_DRAWN = figure_qualitative()
if FIG7_DRAWN:
    print({k: v for k, v in FIG7_DRAWN.items()})

## 13. What the negative images buy (Tables 13 and 14, Figure 8)

The three ablation conditions share architecture, hyperparameters, seed 42 and the frozen test set; only the list of training images differs (Table 5). Two comparisons follow, and together they form a double dissociation:

* **Full versus Size-matched** halves the data while keeping its composition, so it measures the effect of data volume;
* **Size-matched versus No-negatives** keeps 7,938 training images in both arms and differs only in whether verified negatives are present, so it measures the effect of composition. This is the comparison that carries the claim.

The noise floor is the seed standard deviation of mAP50 for the same architecture (Table 6).

In [ ]:
ABLATION_ORDER = [("Full", "resnet18fpn_seed42"), ("Size-matched", "size_matched_seed42"),
                  ("No-negatives", "no_negatives_seed42")]
metrics_by_run = RUN_METRICS.set_index("run_id")
conditions = TABLE_5.set_index("condition")
TABLE_13 = pd.DataFrame([{
    "condition": cond,
    "training_images": int(conditions.loc[cond, "training_images"]),
    "composition": conditions.loc[cond, "composition"],
    "map50": metrics_by_run.loc[run_id, "map50"],
    "ap50_fire": metrics_by_run.loc[run_id, "ap50_fire"],
    "ap50_smoke": metrics_by_run.loc[run_id, "ap50_smoke"],
    "recall": metrics_by_run.loc[run_id, "recall"],
    "far_per_1000": metrics_by_run.loc[run_id, "far_per_1000"],
    "negatives_fired": f"{int(metrics_by_run.loc[run_id, 'negatives_fired'])} of "
                       f"{int(metrics_by_run.loc[run_id, 'negatives']):,}",
} for cond, run_id in ABLATION_ORDER])
save_table(TABLE_13, "table13_negative_ablation")

full, matched, no_neg = (metrics_by_run.loc[r] for _, r in ABLATION_ORDER)
noise_floor = SEED_STATS[("ResNet18+FPN+FCOS", "map50")][1]
resnet_far = MAIN_METRICS[MAIN_METRICS.model == "ResNet18+FPN+FCOS"].far_per_1000
far_ratios = FAR_AT_RECALL.set_index("run_id")
ABLATION = {
    "delta_map50_halving": matched.map50 - full.map50,
    "delta_map50_removing_negatives": no_neg.map50 - matched.map50,
    "noise_floor": noise_floor,
    "halving_over_noise_floor": abs(matched.map50 - full.map50) / noise_floor,
    "far_full": full.far_per_1000, "far_size_matched": matched.far_per_1000, "far_no_negatives": no_neg.far_per_1000,
    "resnet_seed_far_min": float(resnet_far.min()), "resnet_seed_far_max": float(resnet_far.max()),
    "far_ratio_no_neg_over_size_matched": no_neg.far_per_1000 / matched.far_per_1000,
    "pct_negatives_alarmed_no_neg": 100 * no_neg.negatives_fired / no_neg.negatives,
    "pct_negatives_alarmed_size_matched": 100 * matched.negatives_fired / matched.negatives,
    "equal_recall_ratio_0.5": far_ratios.loc["no_negatives_seed42", "far_at_recall_0.5"]
                              / far_ratios.loc["size_matched_seed42", "far_at_recall_0.5"],
    "equal_recall_ratio_0.8": far_ratios.loc["no_negatives_seed42", "far_at_recall_0.8"]
                              / far_ratios.loc["size_matched_seed42", "far_at_recall_0.8"],
}
TABLE_14 = pd.DataFrame([
    {"change": "Halving the data, composition preserved",
     "effect_on_map50": round(ABLATION["delta_map50_halving"], 4),
     "relative_to_noise_floor": f"{ABLATION['halving_over_noise_floor']:.1f}x the noise floor",
     "effect_on_far": f"{full.far_per_1000:.2f} -> {matched.far_per_1000:.2f}",
     "resnet_seed_far_range": f"{ABLATION['resnet_seed_far_min']:.2f}-{ABLATION['resnet_seed_far_max']:.2f}"},
    {"change": "Removing the negatives, volume preserved",
     "effect_on_map50": round(ABLATION["delta_map50_removing_negatives"], 4),
     "relative_to_noise_floor": "inside the noise floor" if abs(ABLATION["delta_map50_removing_negatives"]) <= noise_floor
     else "outside the noise floor",
     "effect_on_far": f"{matched.far_per_1000:.2f} -> {no_neg.far_per_1000:.2f}",
     "resnet_seed_far_range": ""},
])
save_table(TABLE_14, "table14_double_dissociation")
print(TABLE_13.to_string(index=False))
print()
print(TABLE_14.to_string(index=False))
print(f"\nWithout negatives the alarm rate rises {ABLATION['far_ratio_no_neg_over_size_matched']:.1f}-fold "
      f"({ABLATION['pct_negatives_alarmed_size_matched']:.1f}% -> {ABLATION['pct_negatives_alarmed_no_neg']:.1f}% "
      f"of fire-free images); at equal recall the gap is {ABLATION['equal_recall_ratio_0.5']:.1f}x (recall 0.5) "
      f"and {ABLATION['equal_recall_ratio_0.8']:.1f}x (recall 0.8).")


def figure_ablation() -> None:
    names = {"resnet18fpn_seed42": "Full\n14,852", "size_matched_seed42": "Size-\nmatched\n7,938",
             "no_negatives_seed42": "No\nnegatives\n7,938"}
    sub = RUN_METRICS.set_index("run_id").loc[[r for _, r in ABLATION_ORDER]]
    with plt.rc_context(PAPER_STYLE):
        fig, (left, right) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, 2.7))
        x = range(len(sub))
        shades = ["0.78", "0.78", "0.30"]
        for ax, column, fmt in ((left, "map50", "{:.4f}"), (right, "far_per_1000", "{:.2f}")):
            ax.bar(x, sub[column], width=0.55, color=shades, edgecolor="black", linewidth=0.7)
            ax.set_xticks(list(x))
            ax.set_xticklabels([names[r] for r in sub.index])
            ax.set_xlabel("Training set (images)")
            for i, v in enumerate(sub[column]):
                ax.text(i, v, fmt.format(v), ha="center", va="bottom")
        left.set_ylabel("mAP50")
        left.set_ylim(0, max(sub.map50) * 1.2)
        left.set_title("(a) mAP50")
        right.set_ylabel("False alarms per 1,000 (log scale)")
        right.set_yscale("log")
        right.yaxis.set_major_formatter(PLAIN_NUMBER)
        right.set_ylim(top=max(sub.far_per_1000) * 2.5)
        right.set_title("(b) False-alarm rate")
        save_figure(fig, "Fig8.png")


figure_ablation()

## 14. Computational cost (Table 15, recorded on the GPU)

Latency, throughput and peak GPU memory were measured on the locked hardware (NVIDIA GeForce RTX 4050 Laptop GPU, batch 1, 640 x 640 input, bf16 autocast, 20 warm-up and 100 timed iterations, decoding, letterboxing and NMS excluded). These numbers characterise the GPU, not the analysis, and cannot be recomputed on a CPU, so they are read from `data/recorded/gpu_efficiency.csv`, which copies the two benchmark records of the original project (the source of each row is named in the file). Parameter counts are the exception: Appendix B rebuilds the ResNet18+FPN+FCOS detector and its smoke test checks the 12.17 M figure.

In [ ]:
GPU = pd.read_csv(RECORDED_DIR / "gpu_efficiency.csv")
TABLE_15 = pd.DataFrame({
    "model": GPU.model,
    "params": [f"{v:.2f} M" for v in GPU.params_m],
    "model_size": [f"{v:.2f} MB" if v < 10 else f"{v:.1f} MB" for v in GPU.model_size_mb],
    "latency_mean": [f"{v:.2f} ms" for v in GPU.latency_mean_ms],
    "fps": [f"{v:.1f}" for v in GPU.fps],
    "peak_vram": [f"{v:.3f} GB" for v in GPU.peak_vram_gb],
    "gpu": GPU.gpu,
})
save_table(TABLE_15, "table15_compute_cost")
TABLE_15

## 15. Comparison with the manuscript and with the original analysis

Every computed number is checked against two references:

* **the manuscript**: the values printed in Tables 2-15, in the figure captions and in the text of `main.tex`, transcribed below;
* **the original analysis**: the full-precision output files of the authors' analysis pipeline, from which the manuscript was written. Those values were copied verbatim into the `EXPECTED_ARTIFACTS` cell further down; nothing in it was edited by hand.

The tolerance is derived from how a value is written: a number printed with *d* decimals passes when the computed value lies within half a unit of its last digit, which is exactly the condition for it to round to the printed value. Integers, hashes, sample ids and labels must match exactly.

A separate list checks the qualitative **statements** of the paper (for example "fire AP is below smoke AP in all six runs"). A statement that the data do not support is reported as `DOES NOT HOLD`; such a finding concerns the manuscript text, not the computation, and is listed rather than hidden.

In [ ]:
CHECKS: list[dict[str, Any]] = []


def _clean(text: str) -> str:
    return text.strip().replace(",", "").replace("−", "-").replace("+", "")


def _is_number(text: str) -> bool:
    try:
        float(_clean(text))
    except ValueError:
        return False
    return _clean(text).lower() not in ("nan", "inf", "-inf")


def _decimals(text: str) -> int:
    t = _clean(text)
    return len(t.split(".")[1]) if "." in t else 0


def check(group: str, item: str, computed: Any, expected: str, source: str = "manuscript") -> bool:
    """Record one comparison. Numbers: within half a unit of the printed last digit."""
    if isinstance(computed, (np.floating, np.integer)):
        computed = computed.item()
    if _is_number(expected):
        ok = (computed is not None and not (isinstance(computed, float) and math.isnan(computed))
              and abs(float(computed) - float(_clean(expected))) <= 0.5 * 10 ** -_decimals(expected) + 1e-9)
        shown = f"{float(computed):.{max(_decimals(expected) + 2, 2)}f}" if ok or computed is not None else "None"
    elif expected.lower() == "nan":
        ok = computed is None or (isinstance(computed, float) and math.isnan(computed))
        shown = str(computed)
    else:
        ok = str(computed) == expected
        shown = str(computed)
    CHECKS.append({"group": group, "item": item, "source": source, "expected": expected,
                   "computed": shown, "status": "PASS" if ok else "FAIL"})
    return ok

### Values printed in the manuscript

Transcribed from `main.tex`. Signs and thousands separators are kept as printed.

In [ ]:
PAPER_TABLES: dict[str, Any] = {
    # split: images, with objects, verified negative, % negative, fire boxes, smoke boxes, hash, frozen
    "Table 2": {
        "train": ["14,852", "7,938", "6,914", "46.6", "10,256", "8,059", "063f26816180", "no"],
        "val": ["3,418", "1,927", "1,491", "43.6", "2,162", "2,005", "9fe02989d64c", "no"],
        "test": ["3,257", "1,824", "1,433", "44.0", "2,267", "1,790", "6246e1bcca79", "yes"],
    },
    # (split, class): boxes, median area, % boxes below 1 % of image
    "Table 3": {
        ("train", "fire"): ["10,256", "0.00519", "65.3"],
        ("train", "smoke"): ["8,059", "0.15273", "19.3"],
        ("test", "fire"): ["2,267", "0.00625", "61.6"],
        ("test", "smoke"): ["1,790", "0.15179", "17.2"],
    },
    # condition: training images, positives, negatives, train manifest hash
    "Table 5": {
        "Full": ["14,852", "7,938", "6,914", "063f26816180"],
        "No-negatives": ["7,938", "7,938", "0", "a8d7800d9ed2"],
        "Size-matched": ["7,938", "4,243", "3,695", "5737f2a6dde1"],
    },
    # model: seeds, threshold label, then (mean, sd) per quantity
    "Table 6": {
        "YOLO11n": {"seeds": "3", "thr": "0.25", "map50": ("0.7114", "0.0051"),
                    "map50_95": ("0.3959", "0.0021"), "ap50_fire": ("0.6462", "0.0081"),
                    "ap50_smoke": ("0.7767", "0.0049"), "recall": ("0.6819", "0.0022"),
                    "precision": ("0.7186", "0.0070"), "far_per_1000": ("30.01", "3.89")},
        "ResNet18+FPN+FCOS": {"seeds": "3", "thr": "0.45–0.50", "map50": ("0.6646", "0.0033"),
                              "map50_95": ("0.3171", "0.0013"), "ap50_fire": ("0.5803", "0.0041"),
                              "ap50_smoke": ("0.7489", "0.0033"), "recall": ("0.6091", "0.0410"),
                              "precision": ("0.7214", "0.0666"), "far_per_1000": ("8.61", "2.82")},
    },
    # (model, seed): mAP50, FAR, threshold
    "Table 7": {
        ("YOLO11n", 42): ("0.7100", "25.82", "0.25"), ("YOLO11n", 1337): ("0.7072", "33.50", "0.25"),
        ("YOLO11n", 2024): ("0.7171", "30.70", "0.25"),
        ("ResNet18+FPN+FCOS", 42): ("0.6611", "11.86", "0.45"),
        ("ResNet18+FPN+FCOS", 1337): ("0.6676", "6.98", "0.50"),
        ("ResNet18+FPN+FCOS", 2024): ("0.6651", "6.98", "0.50"),
    },
    # recall: YOLO11n mean, ResNet mean, ratio, paired difference, low, high
    "Table 8": {
        0.5: ("5.59", "4.04", "1.4", "+1.78", "−1.94", "+5.58"),
        0.6: ("15.04", "7.88", "1.9", "+7.26", "+1.25", "+13.29"),
        0.7: ("36.50", "16.18", "2.3", "+20.33", "+11.52", "+29.47"),
        0.8: ("97.35", "48.29", "2.0", "+48.71", "+33.17", "+64.34"),
    },
    "Table 9": {
        "YOLO11n": (("0.7186", "0.0070"), ("30.01", "3.89")),
        "ResNet18+FPN+FCOS": (("0.7214", "0.0666"), ("8.61", "2.82")),
    },
    # r: YOLO11n mean, ResNet, lower cost, mAP50 winner, paired difference, low, high
    "Table 10": {
        1: ("0.2673", "0.2200", "ResNet", "YOLO11n", "+0.0474", "+0.0310", "+0.0623"),
        2: ("0.4560", "0.3843", "ResNet", "YOLO11n", "+0.0717", "+0.0459", "+0.0970"),
        5: ("0.9413", "0.7866", "ResNet", "YOLO11n", "+0.1549", "+0.0975", "+0.2101"),
        10: ("1.7433", "1.3861", "ResNet", "YOLO11n", "+0.3579", "+0.2442", "+0.4692"),
        25: ("4.1493", "3.0598", "ResNet", "YOLO11n", "+1.0920", "+0.8156", "+1.3651"),
        50: ("8.1592", "5.6287", "ResNet", "YOLO11n", "+2.5355", "+1.9715", "+3.1125"),
    },
    # run: FAR, interval low, interval high
    "Table 11": {
        "yolo11n_seed42": ("25.82", "17.45", "34.19"), "yolo11n_seed1337": ("33.50", "24.42", "42.57"),
        "yolo11n_seed2024": ("30.70", "22.33", "39.78"), "resnet18fpn_seed42": ("11.86", "6.98", "17.45"),
        "resnet18fpn_seed1337": ("6.98", "3.47", "11.86"), "resnet18fpn_seed2024": ("6.98", "2.79", "11.86"),
    },
    # model: medians missed (seeds 42/1337/2024), medians caught, caught/missed ratios
    "Table 12": {
        "YOLO11n": (("0.0063", "0.0058", "0.0062"), ("0.0281", "0.0282", "0.0282"), ("4.4", "4.8", "4.5")),
        "ResNet18+FPN+FCOS": (("0.0033", "0.0044", "0.0046"), ("0.0402", "0.0469", "0.0435"),
                              ("12.1", "10.6", "9.4")),
    },
    # condition: training images, mAP50, AP fire, AP smoke, FAR, negatives fired, negatives
    "Table 13": {
        "Full": ("14,852", "0.6611", "0.5755", "0.7466", "11.86", "17", "1,433"),
        "Size-matched": ("7,938", "0.6370", "0.5595", "0.7144", "6.28", "9", "1,433"),
        "No-negatives": ("7,938", "0.6338", "0.5778", "0.6898", "306.35", "439", "1,433"),
    },
    "Table 14": {
        "halving": {"delta_map50": "−0.0241", "times_noise_floor": "7", "far_from": "11.86",
                    "far_to": "6.28", "seed_range_low": "6.98", "seed_range_high": "11.86"},
        "removing": {"delta_map50": "−0.0032", "far_from": "6.28", "far_to": "306.35"},
    },
    # model: params (M), size (MB), latency (ms), FPS, peak VRAM (GB)
    "Table 15": {
        "YOLO11n": ("2.59", "5.45", "4.69", "213.4", "0.067"),
        "ResNet18+FPN+FCOS": ("12.17", "46.5", "5.86", "170.7", "0.095"),
    },
}

# Numbers quoted in the running text and captions: (location, description, printed value).
PAPER_TEXT = {
    "images_total": ("III-B", "D-Fire images", "21,527"),
    "negatives_total": ("Abstract, III-B", "verified-negative images in D-Fire", "9,838"),
    "fire_box_shortfall": ("III-B", "fire boxes short of the published count", "7"),
    "smoke_box_shortfall": ("III-B", "smoke boxes short of the published count", "11"),
    "median_fire_area_train": ("I", "median fire box area, train", "0.005"),
    "median_smoke_area_train": ("I", "median smoke box area, train", "0.153"),
    "area_ratio_train": ("I, III-B, IV-G", "median smoke / fire box area, train", "29"),
    "area_ratio_test": ("IV-B, Fig. 1, IV-G", "median smoke / fire box area, test", "24"),
    "official_test_in_train": ("V-4", "official D-Fire test images in the training split", "2,990"),
    "yolo_map50_pct": ("Table 1", "YOLO11n mAP50 (%)", "71.14"),
    "yolo_map50_sd_pct": ("Table 1", "YOLO11n mAP50 sd (%)", "0.51"),
    "yolo_far_1dp": ("Abstract", "YOLO11n alarms per 1,000", "30.0"),
    "yolo_far_sd_1dp": ("Abstract", "YOLO11n alarm sd", "3.9"),
    "resnet_far_1dp": ("Abstract", "ResNet alarms per 1,000", "8.6"),
    "resnet_far_sd_1dp": ("Abstract", "ResNet alarm sd", "2.8"),
    "map50_gain_points": ("Abstract, IV-A", "mAP50 gain of YOLO11n (points)", "4.7"),
    "far_ratio": ("Abstract, IV-A, IV-D", "alarm-rate ratio YOLO11n / ResNet", "3.5"),
    "matched_recall_ratio_min": ("Abstract, IV-C", "lowest alarm ratio at recall 0.60-0.80", "1.9"),
    "matched_recall_ratio_max": ("Abstract, I, IV-C", "highest alarm ratio at recall 0.60-0.80", "2.3"),
    "cv_far_yolo": ("IV-A", "CV of FAR, YOLO11n (%)", "12.95"),
    "cv_far_resnet": ("IV-A", "CV of FAR, ResNet (%)", "32.77"),
    "cv_map_yolo": ("IV-A", "CV of mAP50, YOLO11n (%)", "0.72"),
    "cv_map_resnet": ("IV-A", "CV of mAP50, ResNet (%)", "0.50"),
    "gap_yolo": ("IV-B", "smoke - fire AP gap, YOLO11n", "0.131"),
    "gap_resnet": ("IV-B", "smoke - fire AP gap, ResNet", "0.169"),
    "gap_noise_yolo": ("IV-B", "gap / noise floor, YOLO11n", "26"),
    "gap_noise_resnet": ("IV-B", "gap / noise floor, ResNet", "51"),
    "p_recall_05": ("IV-C", "paired p-value at recall 0.50", "0.352"),
    "caught_missed_min": ("IV-G", "smallest caught/missed factor", "4.4"),
    "caught_missed_max": ("IV-G", "largest caught/missed factor", "12.1"),
    "listing_instances": ("IV-G", "error instances in the released listing", "400"),
    "listing_images": ("IV-G", "distinct images in the listing", "261"),
    "fig7_missed_area_pct": ("Fig. 7", "missed fire box (% of image)", "0.05"),
    "ablation_delta_map": ("Abstract, IV-H", "mAP50 change when removing negatives", "−0.0032"),
    "noise_floor": ("Abstract, IV-H", "seed noise floor (ResNet mAP50 sd)", "0.0033"),
    "ablation_far_from_1dp": ("Abstract", "alarm rate, Size-matched", "6.3"),
    "ablation_far_to_1dp": ("Abstract", "alarm rate, No-negatives", "306.4"),
    "ablation_far_ratio": ("I, IV-H", "alarm-rate multiplication without negatives", "48.8"),
    "pct_alarmed_no_neg": ("IV-H", "fire-free images alarmed, No-negatives (%)", "30.6"),
    "pct_alarmed_matched": ("IV-H", "fire-free images alarmed, Size-matched (%)", "0.6"),
    "equal_recall_ratio_05": ("IV-H", "No-neg / Size-matched alarms at recall 0.5", "62.3"),
    "equal_recall_ratio_08": ("IV-H", "No-neg / Size-matched alarms at recall 0.8", "10.4"),
    "dense_boxes_resnet": ("V-4", "ResNet seed 42 test detections at the 0.05 floor", "726,609"),
    "dense_boxes_yolo": ("V-4", "YOLO11n seed 42 test detections at the 0.05 floor", "8,412"),
    "dense_factor": ("V-4", "detection-density factor", "86"),
}

# The eight quantities of Section III-H, plus the threshold and the alarm count.
PAPER_REPRODUCTION = {"map50": "0.66107", "map50_95": "0.31743", "ap50_fire": "0.57554",
                      "ap50_smoke": "0.74659", "precision": "0.64461", "recall": "0.65638",
                      "f1": "0.64704", "far_per_1000": "11.863", "negatives_fired": "17",
                      "negatives": "1,433", "threshold": "0.45"}

### Values from the original analysis outputs

Copied verbatim from the result files of the original analysis; do not edit.

In [ ]:
EXPECTED_ARTIFACTS = {
    'run_metrics': {
        'yolo11n_seed42': {
            'threshold': '0.25', 'map50': '0.70996', 'map50_95': '0.39525',
            'ap50_fire': '0.64831', 'ap50_smoke': '0.77162', 'precision': '0.71951',
            'recall': '0.68361', 'f1': '0.70098', 'recall_fire': '0.62197',
            'recall_smoke': '0.74525', 'far_per_1000': '25.820', 'negatives': '1433',
            'negatives_fired': '37',
        },
        'yolo11n_seed1337': {
            'threshold': '0.25', 'map50': '0.70723', 'map50_95': '0.39429',
            'ap50_fire': '0.63723', 'ap50_smoke': '0.77723', 'precision': '0.71120',
            'recall': '0.68280', 'f1': '0.69662', 'recall_fire': '0.61756',
            'recall_smoke': '0.74804', 'far_per_1000': '33.496', 'negatives': '1433',
            'negatives_fired': '48',
        },
        'yolo11n_seed2024': {
            'threshold': '0.25', 'map50': '0.71711', 'map50_95': '0.39828',
            'ap50_fire': '0.65294', 'ap50_smoke': '0.78129', 'precision': '0.72520',
            'recall': '0.67942', 'f1': '0.70148', 'recall_fire': '0.62197',
            'recall_smoke': '0.73687', 'far_per_1000': '30.705', 'negatives': '1433',
            'negatives_fired': '44',
        },
        'resnet18fpn_seed42': {
            'threshold': '0.45', 'map50': '0.66107', 'map50_95': '0.31743',
            'ap50_fire': '0.57554', 'ap50_smoke': '0.74659', 'precision': '0.64461',
            'recall': '0.65638', 'f1': '0.64704', 'recall_fire': '0.54963',
            'recall_smoke': '0.76313', 'far_per_1000': '11.863', 'negatives': '1433',
            'negatives_fired': '17',
        },
        'resnet18fpn_seed1337': {
            'threshold': '0.50', 'map50': '0.66763', 'map50_95': '0.31814',
            'ap50_fire': '0.58260', 'ap50_smoke': '0.75265', 'precision': '0.76327',
            'recall': '0.58754', 'f1': '0.65880', 'recall_fire': '0.47508',
            'recall_smoke': '0.70000', 'far_per_1000': '6.978', 'negatives': '1433',
            'negatives_fired': '10',
        },
        'resnet18fpn_seed2024': {
            'threshold': '0.50', 'map50': '0.66514', 'map50_95': '0.31568',
            'ap50_fire': '0.58275', 'ap50_smoke': '0.74754', 'precision': '0.75641',
            'recall': '0.58325', 'f1': '0.65405', 'recall_fire': '0.47375',
            'recall_smoke': '0.69274', 'far_per_1000': '6.978', 'negatives': '1433',
            'negatives_fired': '10',
        },
        'no_negatives_seed42': {
            'threshold': '0.50', 'map50': '0.63378', 'map50_95': '0.31344',
            'ap50_fire': '0.57775', 'ap50_smoke': '0.68981', 'precision': '0.64842',
            'recall': '0.59558', 'f1': '0.60660', 'recall_fire': '0.46714',
            'recall_smoke': '0.72402', 'far_per_1000': '306.350', 'negatives': '1433',
            'negatives_fired': '439',
        },
        'size_matched_seed42': {
            'threshold': '0.50', 'map50': '0.63696', 'map50_95': '0.29286',
            'ap50_fire': '0.55950', 'ap50_smoke': '0.71442', 'precision': '0.71110',
            'recall': '0.56717', 'f1': '0.62649', 'recall_fire': '0.46228',
            'recall_smoke': '0.67207', 'far_per_1000': '6.281', 'negatives': '1433',
            'negatives_fired': '9',
        },
    },
    'val_selection': {
        'yolo11n_seed42': {'threshold': '0.25', 'f1': '0.70202', 'recall': '0.67531', 'precision': '0.73102'},
        'yolo11n_seed1337': {'threshold': '0.25', 'f1': '0.70058', 'recall': '0.67903', 'precision': '0.72372'},
        'yolo11n_seed2024': {'threshold': '0.25', 'f1': '0.70327', 'recall': '0.67323', 'precision': '0.73615'},
        'resnet18fpn_seed42': {'threshold': '0.45', 'f1': '0.63655', 'recall': '0.62821', 'precision': '0.64899'},
        'resnet18fpn_seed1337': {'threshold': '0.50', 'f1': '0.65107', 'recall': '0.57412', 'precision': '0.76102'},
        'resnet18fpn_seed2024': {'threshold': '0.50', 'f1': '0.64211', 'recall': '0.56838', 'precision': '0.74307'},
        'no_negatives_seed42': {'threshold': '0.50', 'f1': '0.59240', 'recall': '0.57343', 'precision': '0.63702'},
        'size_matched_seed42': {'threshold': '0.50', 'f1': '0.61850', 'recall': '0.55401', 'precision': '0.70619'},
    },
    'predictions': {
        'yolo11n_seed42': {
            'val': {'images': '3418', 'boxes': '8296'},
            'test': {'images': '3257', 'boxes': '8412'},
        },
        'yolo11n_seed1337': {
            'val': {'images': '3418', 'boxes': '8763'},
            'test': {'images': '3257', 'boxes': '8955'},
        },
        'yolo11n_seed2024': {
            'val': {'images': '3418', 'boxes': '8533'},
            'test': {'images': '3257', 'boxes': '8739'},
        },
        'resnet18fpn_seed42': {
            'val': {'images': '3418', 'boxes': '770884'},
            'test': {'images': '3257', 'boxes': '726609'},
        },
        'resnet18fpn_seed1337': {
            'val': {'images': '3418', 'boxes': '511107'},
            'test': {'images': '3257', 'boxes': '477276'},
        },
        'resnet18fpn_seed2024': {
            'val': {'images': '3418', 'boxes': '536146'},
            'test': {'images': '3257', 'boxes': '502937'},
        },
        'no_negatives_seed42': {
            'val': {'images': '3418', 'boxes': '889053'},
            'test': {'images': '3257', 'boxes': '836828'},
        },
        'size_matched_seed42': {
            'val': {'images': '3418', 'boxes': '906885'},
            'test': {'images': '3257', 'boxes': '854510'},
        },
    },
    'dataset': {
        'train': {
            'images': '14852', 'with_objects': '7938', 'verified_negative': '6914',
            'pct_negative': '46.6', 'fire_boxes': '10256', 'smoke_boxes': '8059',
            'images_with_fire': '4067', 'images_with_smoke': '7139', 'median_width': '1027',
            'median_height': '660', 'manifest_hash': '063f26816180', 'frozen': 'False',
        },
        'val': {
            'images': '3418', 'with_objects': '1927', 'verified_negative': '1491',
            'pct_negative': '43.6', 'fire_boxes': '2162', 'smoke_boxes': '2005',
            'images_with_fire': '849', 'images_with_smoke': '1765', 'median_width': '1280',
            'median_height': '720', 'manifest_hash': '9fe02989d64c', 'frozen': 'False',
        },
        'test': {
            'images': '3257', 'with_objects': '1824', 'verified_negative': '1433',
            'pct_negative': '44.0', 'fire_boxes': '2267', 'smoke_boxes': '1790',
            'images_with_fire': '906', 'images_with_smoke': '1621', 'median_width': '1024',
            'median_height': '664', 'manifest_hash': '6246e1bcca79', 'frozen': 'True',
        },
    },
    'object_scale': {
        'train/fire': {
            'boxes': '10256', 'median_area': '0.00519', 'mean_area': '0.02418',
            'pct_below_1pct': '65.3',
        },
        'train/smoke': {
            'boxes': '8059', 'median_area': '0.15273', 'mean_area': '0.23035',
            'pct_below_1pct': '19.3',
        },
        'test/fire': {
            'boxes': '2267', 'median_area': '0.00625', 'mean_area': '0.02623',
            'pct_below_1pct': '61.6',
        },
        'test/smoke': {
            'boxes': '1790', 'median_area': '0.15179', 'mean_area': '0.23715',
            'pct_below_1pct': '17.2',
        },
    },
    'curve': {
        'yolo11n_seed42': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.83502', '0.78509', '0.74858', '0.71722', '0.68361', '0.65379', '0.61952',
                '0.58270', '0.54411', '0.50317', '0.45905', '0.40930', '0.36126', '0.30125',
                '0.23288', '0.15451', '0.07236', '0.01619', '0.00072',
            ],
            'precision': [
                '0.41425', '0.53481', '0.60921', '0.67155', '0.71951', '0.76521', '0.79989',
                '0.82933', '0.85753', '0.88317', '0.90013', '0.91729', '0.93345', '0.94784',
                '0.96299', '0.98032', '0.99474', '0.98750', '1.00000',
            ],
            'far_per_1000': [
                '129.100', '76.762', '53.733', '36.985', '25.820', '18.842', '13.957', '9.770',
                '6.978', '4.885', '4.885', '4.187', '2.094', '2.094', '0.000', '0.000', '0.000',
                '0.000', '0.000',
            ],
            'negatives_fired': [
                '185', '110', '77', '53', '37', '27', '20', '14', '10', '7', '7', '6', '3', '3',
                '0', '0', '0', '0', '0',
            ],
        },
        'yolo11n_seed1337': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.84047', '0.79194', '0.75420', '0.72190', '0.68280', '0.64541', '0.61217',
                '0.57870', '0.53677', '0.49540', '0.45464', '0.40530', '0.34717', '0.28858',
                '0.22956', '0.15605', '0.08045', '0.01545', '0.00000',
            ],
            'precision': [
                '0.39438', '0.51142', '0.59539', '0.65961', '0.71120', '0.75196', '0.78280',
                '0.81732', '0.84533', '0.87645', '0.89992', '0.91136', '0.92851', '0.94777',
                '0.96665', '0.97961', '1.00000', '1.00000', '0.00000',
            ],
            'far_per_1000': [
                '139.567', '87.927', '60.014', '46.755', '33.496', '26.518', '21.633', '15.352',
                '12.561', '7.676', '6.281', '4.187', '3.489', '2.094', '0.698', '0.698', '0.000',
                '0.000', '0.000',
            ],
            'negatives_fired': [
                '200', '126', '86', '67', '48', '38', '31', '22', '18', '11', '9', '6', '5', '3',
                '1', '1', '0', '0', '0',
            ],
        },
        'yolo11n_seed2024': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.84331', '0.79162', '0.75096', '0.71450', '0.67942', '0.64559', '0.60917',
                '0.57512', '0.53837', '0.49803', '0.45285', '0.40618', '0.35618', '0.30378',
                '0.24097', '0.16805', '0.09270', '0.01047', '0.00022',
            ],
            'precision': [
                '0.40156', '0.52453', '0.60270', '0.67084', '0.72521', '0.76829', '0.80731',
                '0.83688', '0.86733', '0.89543', '0.91149', '0.93228', '0.94753', '0.95698',
                '0.97285', '0.98681', '0.99618', '1.00000', '0.50000',
            ],
            'far_per_1000': [
                '149.337', '94.208', '66.294', '44.662', '30.705', '21.633', '15.352', '10.468',
                '6.978', '3.489', '2.791', '2.094', '1.396', '0.698', '0.698', '0.000', '0.000',
                '0.000', '0.000',
            ],
            'negatives_fired': [
                '214', '135', '95', '64', '44', '31', '22', '15', '10', '5', '4', '3', '2', '1',
                '1', '0', '0', '0', '0',
            ],
        },
        'resnet18fpn_seed42': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.90327', '0.89730', '0.88460', '0.86614', '0.84108', '0.80436', '0.75820',
                '0.71539', '0.65638', '0.58783', '0.49213', '0.38064', '0.26557', '0.13389',
                '0.05131', '0.01117', '0.00223', '0.00000', '0.00000',
            ],
            'precision': [
                '0.00539', '0.02074', '0.04772', '0.09797', '0.17379', '0.27313', '0.39289',
                '0.51679', '0.64461', '0.74689', '0.82771', '0.89282', '0.92724', '0.96130',
                '0.98148', '1.00000', '0.50000', '0.00000', '0.00000',
            ],
            'far_per_1000': [
                '1000.000', '997.906', '232.380', '136.776', '87.927', '55.129', '36.288',
                '20.935', '11.863', '7.676', '4.187', '0.000', '0.000', '0.000', '0.000',
                '0.000', '0.000', '0.000', '0.000',
            ],
            'negatives_fired': [
                '1433', '1430', '333', '196', '126', '79', '52', '30', '17', '11', '6', '0', '0',
                '0', '0', '0', '0', '0', '0',
            ],
        },
        'resnet18fpn_seed1337': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.90612', '0.90015', '0.88805', '0.86567', '0.83994', '0.80954', '0.76613',
                '0.71866', '0.66279', '0.58754', '0.49074', '0.37716', '0.22556', '0.10309',
                '0.04164', '0.01314', '0.00194', '0.00000', '0.00000',
            ],
            'precision': [
                '0.00795', '0.02243', '0.04833', '0.09651', '0.16923', '0.26872', '0.38859',
                '0.51919', '0.65236', '0.76328', '0.83999', '0.90488', '0.93526', '0.96445',
                '0.99451', '1.00000', '1.00000', '0.00000', '0.00000',
            ],
            'far_per_1000': [
                '1000.000', '544.313', '262.387', '158.409', '92.114', '55.129', '32.100',
                '18.842', '9.770', '6.978', '4.187', '0.698', '0.000', '0.000', '0.000', '0.000',
                '0.000', '0.000', '0.000',
            ],
            'negatives_fired': [
                '1433', '780', '376', '227', '132', '79', '46', '27', '14', '10', '6', '1', '0',
                '0', '0', '0', '0', '0', '0',
            ],
        },
        'resnet18fpn_seed2024': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.90594', '0.90193', '0.88996', '0.86851', '0.84406', '0.81074', '0.76702',
                '0.71384', '0.65677', '0.58325', '0.49150', '0.37416', '0.22352', '0.10580',
                '0.04384', '0.00998', '0.00184', '0.00000', '0.00000',
            ],
            'precision': [
                '0.00726', '0.02100', '0.04626', '0.09461', '0.16906', '0.26935', '0.39318',
                '0.51494', '0.64591', '0.75641', '0.83997', '0.90404', '0.94623', '0.97211',
                '0.99390', '1.00000', '1.00000', '0.00000', '0.00000',
            ],
            'far_per_1000': [
                '1000.000', '609.211', '262.387', '145.150', '89.323', '47.453', '23.029',
                '15.352', '10.468', '6.978', '2.791', '1.396', '0.698', '0.000', '0.000',
                '0.000', '0.000', '0.000', '0.000',
            ],
            'negatives_fired': [
                '1433', '873', '376', '208', '128', '68', '33', '22', '15', '10', '4', '2', '1',
                '0', '0', '0', '0', '0', '0',
            ],
        },
        'no_negatives_seed42': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.91898', '0.91591', '0.90493', '0.88846', '0.86310', '0.82351', '0.78295',
                '0.72955', '0.66743', '0.59558', '0.50992', '0.39646', '0.25217', '0.11863',
                '0.04321', '0.01095', '0.00050', '0.00000', '0.00000',
            ],
            'precision': [
                '0.00466', '0.01235', '0.03142', '0.06808', '0.12651', '0.20464', '0.30593',
                '0.41511', '0.53645', '0.64842', '0.76233', '0.84857', '0.91132', '0.96356',
                '0.98558', '0.98077', '1.00000', '0.00000', '0.00000',
            ],
            'far_per_1000': [
                '1000.000', '993.719', '969.993', '908.583', '815.073', '717.376', '611.305',
                '512.910', '404.745', '306.350', '189.812', '100.488', '29.309', '2.791',
                '1.396', '0.000', '0.000', '0.000', '0.000',
            ],
            'negatives_fired': [
                '1433', '1424', '1390', '1302', '1168', '1028', '876', '735', '580', '439',
                '272', '144', '42', '4', '2', '0', '0', '0', '0',
            ],
        },
        'size_matched_seed42': {
            'threshold': [
                '0.05', '0.10', '0.15', '0.20', '0.25', '0.30', '0.35', '0.40', '0.45', '0.50',
                '0.55', '0.60', '0.65', '0.70', '0.75', '0.80', '0.85', '0.90', '0.95',
            ],
            'recall': [
                '0.90399', '0.90069', '0.88873', '0.86641', '0.83446', '0.80219', '0.75989',
                '0.70619', '0.64171', '0.56718', '0.46951', '0.34161', '0.19661', '0.08389',
                '0.02616', '0.00532', '0.00028', '0.00000', '0.00000',
            ],
            'precision': [
                '0.00473', '0.01624', '0.03978', '0.08251', '0.14656', '0.23511', '0.34161',
                '0.46031', '0.59559', '0.71110', '0.81083', '0.89714', '0.95110', '0.95970',
                '0.98129', '1.00000', '0.50000', '0.00000', '0.00000',
            ],
            'far_per_1000': [
                '1000.000', '704.815', '344.033', '184.927', '106.769', '64.201', '45.359',
                '28.611', '12.561', '6.281', '1.396', '0.698', '0.000', '0.000', '0.000',
                '0.000', '0.000', '0.000', '0.000',
            ],
            'negatives_fired': [
                '1433', '1010', '493', '265', '153', '92', '65', '41', '18', '9', '2', '1', '0',
                '0', '0', '0', '0', '0', '0',
            ],
        },
    },
    'far_at_recall': {
        'yolo11n_seed42': {'0.5': '4.885', '0.6': '11.737', '0.7': '31.265', '0.8': '92.391'},
        'yolo11n_seed1337': {'0.5': '8.219', '0.6': '19.349', '0.7': '39.329', '0.8': '96.504'},
        'yolo11n_seed2024': {'0.5': '3.659', '0.6': '14.037', '0.7': '38.893', '0.8': '103.146'},
        'resnet18fpn_seed42': {'0.5': '4.474', '0.6': '8.419', '0.7': '18.569', '0.8': '53.349'},
        'resnet18fpn_seed1337': {'0.5': '4.454', '0.6': '7.440', '0.7': '15.812', '0.8': '50.068'},
        'resnet18fpn_seed2024': {'0.5': '3.179', '0.6': '7.773', '0.7': '14.168', '0.8': '41.453'},
        'no_negatives_seed42': {'0.5': '182.002', '0.6': '312.403', '0.7': '461.457', '0.8': '655.894'},
        'size_matched_seed42': {'0.5': '2.921', '0.6': '9.046', '0.7': '27.070', '0.8': '63.225'},
    },
    'cost': {
        'yolo11n_seed42': {
            '1': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '0.34221', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '0.29408',
                'threshold_fire': '0.05', 'threshold_smoke': '0.20',
                'recall_per_class': '0.79089', 'far_per_class': '50.244',
                'cost_per_class': '0.25936', 'saving_pct_vs_literature': '24.21',
            },
            '2': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '0.65860', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '0.45906',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.83502', 'far_per_class': '129.100',
                'cost_per_class': '0.45906', 'saving_pct_vs_literature': '30.30',
            },
            '5': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '1.60777', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '0.95400',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.83502', 'far_per_class': '129.100',
                'cost_per_class': '0.95400', 'saving_pct_vs_literature': '40.66',
            },
            '10': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '3.18973', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '1.77890',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.83502', 'far_per_class': '129.100',
                'cost_per_class': '1.77890', 'saving_pct_vs_literature': '44.23',
            },
            '25': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '7.93559', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '4.25359',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.83502', 'far_per_class': '129.100',
                'cost_per_class': '4.25359', 'saving_pct_vs_literature': '46.40',
            },
            '50': {
                'recall_literature': '0.68361', 'far_literature': '25.820',
                'cost_literature': '15.84535', 'threshold_single': '0.05',
                'recall_single': '0.83502', 'far_single': '129.100', 'cost_single': '8.37809',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.83502', 'far_per_class': '129.100',
                'cost_per_class': '8.37809', 'saving_pct_vs_literature': '47.13',
            },
        },
        'yolo11n_seed1337': {
            '1': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '0.35070', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '0.29909',
                'threshold_fire': '0.05', 'threshold_smoke': '0.10',
                'recall_per_class': '0.82260', 'far_per_class': '91.417',
                'cost_per_class': '0.26882', 'saving_pct_vs_literature': '23.35',
            },
            '2': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '0.66790', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '0.45862',
                'threshold_fire': '0.05', 'threshold_smoke': '0.10',
                'recall_per_class': '0.82260', 'far_per_class': '91.417',
                'cost_per_class': '0.44622', 'saving_pct_vs_literature': '33.19',
            },
            '5': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '1.61949', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '0.93719',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84047', 'far_per_class': '139.567',
                'cost_per_class': '0.93719', 'saving_pct_vs_literature': '42.13',
            },
            '10': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '3.20549', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '1.73482',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84047', 'far_per_class': '139.567',
                'cost_per_class': '1.73482', 'saving_pct_vs_literature': '45.88',
            },
            '25': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '7.96348', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '4.12770',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84047', 'far_per_class': '139.567',
                'cost_per_class': '4.12770', 'saving_pct_vs_literature': '48.17',
            },
            '50': {
                'recall_literature': '0.68280', 'far_literature': '33.496',
                'cost_literature': '15.89347', 'threshold_single': '0.05',
                'recall_single': '0.84047', 'far_single': '139.567', 'cost_single': '8.11583',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84047', 'far_per_class': '139.567',
                'cost_per_class': '8.11583', 'saving_pct_vs_literature': '48.94',
            },
        },
        'yolo11n_seed2024': {
            '1': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '0.35129', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '0.30602',
                'threshold_fire': '0.05', 'threshold_smoke': '0.15',
                'recall_per_class': '0.80588', 'far_per_class': '79.553',
                'cost_per_class': '0.27367', 'saving_pct_vs_literature': '22.09',
            },
            '2': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '0.67187', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '0.46271',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84331', 'far_per_class': '149.337',
                'cost_per_class': '0.46271', 'saving_pct_vs_literature': '31.13',
            },
            '5': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '1.63361', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '0.93278',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84331', 'far_per_class': '149.337',
                'cost_per_class': '0.93278', 'saving_pct_vs_literature': '42.90',
            },
            '10': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '3.23651', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '1.71621',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84331', 'far_per_class': '149.337',
                'cost_per_class': '1.71621', 'saving_pct_vs_literature': '46.97',
            },
            '25': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '8.04522', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '4.06653',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84331', 'far_per_class': '149.337',
                'cost_per_class': '4.06653', 'saving_pct_vs_literature': '49.45',
            },
            '50': {
                'recall_literature': '0.67942', 'far_literature': '30.705',
                'cost_literature': '16.05973', 'threshold_single': '0.05',
                'recall_single': '0.84331', 'far_single': '149.337', 'cost_single': '7.98373',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.84331', 'far_per_class': '149.337',
                'cost_per_class': '7.98373', 'saving_pct_vs_literature': '50.29',
            },
        },
        'resnet18fpn_seed42': {
            '1': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '0.24685', 'threshold_single': '0.25',
                'recall_single': '0.84108', 'far_single': '87.927', 'cost_single': '0.24685',
                'threshold_fire': '0.20', 'threshold_smoke': '0.30',
                'recall_per_class': '0.84296', 'far_per_class': '68.388',
                'cost_per_class': '0.22543', 'saving_pct_vs_literature': '8.68',
            },
            '2': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '0.40578', 'threshold_single': '0.20',
                'recall_single': '0.86614', 'far_single': '136.776', 'cost_single': '0.40449',
                'threshold_fire': '0.20', 'threshold_smoke': '0.25',
                'recall_per_class': '0.85497', 'far_per_class': '93.510',
                'cost_per_class': '0.38357', 'saving_pct_vs_literature': '5.47',
            },
            '5': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '0.88255', 'threshold_single': '0.20',
                'recall_single': '0.86614', 'far_single': '136.776', 'cost_single': '0.80606',
                'threshold_fire': '0.15', 'threshold_smoke': '0.20',
                'recall_per_class': '0.87342', 'far_per_class': '149.337',
                'cost_per_class': '0.78223', 'saving_pct_vs_literature': '11.37',
            },
            '10': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '1.67717', 'threshold_single': '0.15',
                'recall_single': '0.88460', 'far_single': '232.380', 'cost_single': '1.38642',
                'threshold_fire': '0.15', 'threshold_smoke': '0.15',
                'recall_per_class': '0.88460', 'far_per_class': '232.380',
                'cost_per_class': '1.38642', 'saving_pct_vs_literature': '17.34',
            },
            '25': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '4.06103', 'threshold_single': '0.15',
                'recall_single': '0.88460', 'far_single': '232.380', 'cost_single': '3.11749',
                'threshold_fire': '0.15', 'threshold_smoke': '0.15',
                'recall_per_class': '0.88460', 'far_per_class': '232.380',
                'cost_per_class': '3.11749', 'saving_pct_vs_literature': '23.23',
            },
            '50': {
                'recall_literature': '0.84108', 'far_literature': '87.927',
                'cost_literature': '8.03414', 'threshold_single': '0.05',
                'recall_single': '0.90327', 'far_single': '1000.000', 'cost_single': '5.83665',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.90327', 'far_per_class': '1000.000',
                'cost_per_class': '5.83665', 'saving_pct_vs_literature': '27.35',
            },
        },
        'resnet18fpn_seed1337': {
            '1': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '0.25217', 'threshold_single': '0.25',
                'recall_single': '0.83994', 'far_single': '92.114', 'cost_single': '0.25217',
                'threshold_fire': '0.20', 'threshold_smoke': '0.30',
                'recall_per_class': '0.84416', 'far_per_class': '69.784',
                'cost_per_class': '0.22562', 'saving_pct_vs_literature': '10.53',
            },
            '2': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '0.41223', 'threshold_single': '0.20',
                'recall_single': '0.86567', 'far_single': '158.409', 'cost_single': '0.42706',
                'threshold_fire': '0.20', 'threshold_smoke': '0.25',
                'recall_per_class': '0.85450', 'far_per_class': '102.582',
                'cost_per_class': '0.39358', 'saving_pct_vs_literature': '4.52',
            },
            '5': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '0.89240', 'threshold_single': '0.20',
                'recall_single': '0.86567', 'far_single': '158.409', 'cost_single': '0.83004',
                'threshold_fire': '0.15', 'threshold_smoke': '0.20',
                'recall_per_class': '0.87604', 'far_per_class': '182.135',
                'cost_per_class': '0.80194', 'saving_pct_vs_literature': '10.14',
            },
            '10': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '1.69268', 'threshold_single': '0.15',
                'recall_single': '0.88805', 'far_single': '262.387', 'cost_single': '1.38188',
                'threshold_fire': '0.15', 'threshold_smoke': '0.15',
                'recall_per_class': '0.88805', 'far_per_class': '262.387',
                'cost_per_class': '1.38188', 'saving_pct_vs_literature': '18.36',
            },
            '25': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '4.09353', 'threshold_single': '0.10',
                'recall_single': '0.90015', 'far_single': '544.313', 'cost_single': '3.04055',
                'threshold_fire': '0.10', 'threshold_smoke': '0.10',
                'recall_per_class': '0.90015', 'far_per_class': '544.313',
                'cost_per_class': '3.04055', 'saving_pct_vs_literature': '25.72',
            },
            '50': {
                'recall_literature': '0.83994', 'far_literature': '92.114',
                'cost_literature': '8.09495', 'threshold_single': '0.10',
                'recall_single': '0.90015', 'far_single': '544.313', 'cost_single': '5.53678',
                'threshold_fire': '0.10', 'threshold_smoke': '0.10',
                'recall_per_class': '0.90015', 'far_per_class': '544.313',
                'cost_per_class': '5.53678', 'saving_pct_vs_literature': '31.60',
            },
        },
        'resnet18fpn_seed2024': {
            '1': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '0.24526', 'threshold_single': '0.25',
                'recall_single': '0.84406', 'far_single': '89.323', 'cost_single': '0.24526',
                'threshold_fire': '0.20', 'threshold_smoke': '0.35',
                'recall_per_class': '0.83164', 'far_per_class': '40.475',
                'cost_per_class': '0.20883', 'saving_pct_vs_literature': '14.85',
            },
            '2': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '0.40120', 'threshold_single': '0.25',
                'recall_single': '0.84406', 'far_single': '89.323', 'cost_single': '0.40120',
                'threshold_fire': '0.20', 'threshold_smoke': '0.25',
                'recall_per_class': '0.85818', 'far_per_class': '92.114',
                'cost_per_class': '0.37576', 'saving_pct_vs_literature': '6.34',
            },
            '5': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '0.86902', 'threshold_single': '0.20',
                'recall_single': '0.86851', 'far_single': '145.150', 'cost_single': '0.80259',
                'threshold_fire': '0.15', 'threshold_smoke': '0.20',
                'recall_per_class': '0.87711', 'far_per_class': '161.200',
                'cost_per_class': '0.77564', 'saving_pct_vs_literature': '10.75',
            },
            '10': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '1.64872', 'threshold_single': '0.15',
                'recall_single': '0.88996', 'far_single': '262.387', 'cost_single': '1.36277',
                'threshold_fire': '0.15', 'threshold_smoke': '0.20',
                'recall_per_class': '0.87711', 'far_per_class': '161.200',
                'cost_per_class': '1.39007', 'saving_pct_vs_literature': '15.69',
            },
            '25': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '3.98781', 'threshold_single': '0.10',
                'recall_single': '0.90193', 'far_single': '609.211', 'cost_single': '3.06097',
                'threshold_fire': '0.10', 'threshold_smoke': '0.15',
                'recall_per_class': '0.89327', 'far_per_class': '353.105',
                'cost_per_class': '3.02134', 'saving_pct_vs_literature': '24.24',
            },
            '50': {
                'recall_literature': '0.84406', 'far_literature': '89.323',
                'cost_literature': '7.88630', 'threshold_single': '0.10',
                'recall_single': '0.90193', 'far_single': '609.211', 'cost_single': '5.51273',
                'threshold_fire': '0.10', 'threshold_smoke': '0.10',
                'recall_per_class': '0.90193', 'far_per_class': '609.211',
                'cost_per_class': '5.51273', 'saving_pct_vs_literature': '30.10',
            },
        },
        'no_negatives_seed42': {
            '1': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '0.95197', 'threshold_single': '0.60',
                'recall_single': '0.39646', 'far_single': '100.488', 'cost_single': '0.70403',
                'threshold_fire': '0.40', 'threshold_smoke': '0.65',
                'recall_per_class': '0.49324', 'far_per_class': '111.654',
                'cost_per_class': '0.61841', 'saving_pct_vs_literature': '35.04',
            },
            '2': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '1.08887', 'threshold_single': '0.35',
                'recall_single': '0.78295', 'far_single': '611.305', 'cost_single': '1.04540',
                'threshold_fire': '0.25', 'threshold_smoke': '0.45',
                'recall_per_class': '0.79690', 'far_per_class': '482.205',
                'cost_per_class': '0.88841', 'saving_pct_vs_literature': '18.41',
            },
            '5': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '1.49957', 'threshold_single': '0.05',
                'recall_single': '0.91898', 'far_single': '1000.000', 'cost_single': '1.40508',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.91898', 'far_per_class': '1000.000',
                'cost_per_class': '1.40508', 'saving_pct_vs_literature': '6.30',
            },
            '10': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '2.18407', 'threshold_single': '0.05',
                'recall_single': '0.91898', 'far_single': '1000.000', 'cost_single': '1.81015',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.91898', 'far_per_class': '1000.000',
                'cost_per_class': '1.81015', 'saving_pct_vs_literature': '17.12',
            },
            '25': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '4.23756', 'threshold_single': '0.05',
                'recall_single': '0.91898', 'far_single': '1000.000', 'cost_single': '3.02539',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.91898', 'far_per_class': '1000.000',
                'cost_per_class': '3.02539', 'saving_pct_vs_literature': '28.61',
            },
            '50': {
                'recall_literature': '0.86310', 'far_literature': '815.073',
                'cost_literature': '7.66005', 'threshold_single': '0.05',
                'recall_single': '0.91898', 'far_single': '1000.000', 'cost_single': '5.05077',
                'threshold_fire': '0.05', 'threshold_smoke': '0.05',
                'recall_per_class': '0.91898', 'far_per_class': '1000.000',
                'cost_per_class': '5.05077', 'saving_pct_vs_literature': '34.06',
            },
        },
        'size_matched_seed42': {
            '1': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '0.27231', 'threshold_single': '0.30',
                'recall_single': '0.80219', 'far_single': '64.201', 'cost_single': '0.26201',
                'threshold_fire': '0.25', 'threshold_smoke': '0.35',
                'recall_per_class': '0.80290', 'far_per_class': '56.525',
                'cost_per_class': '0.25363', 'saving_pct_vs_literature': '6.86',
            },
            '2': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '0.43785', 'threshold_single': '0.25',
                'recall_single': '0.83446', 'far_single': '106.769', 'cost_single': '0.43785',
                'threshold_fire': '0.20', 'threshold_smoke': '0.30',
                'recall_per_class': '0.83792', 'far_per_class': '88.625',
                'cost_per_class': '0.41279', 'saving_pct_vs_literature': '5.72',
            },
            '5': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '0.93447', 'threshold_single': '0.20',
                'recall_single': '0.86641', 'far_single': '184.927', 'cost_single': '0.85288',
                'threshold_fire': '0.15', 'threshold_smoke': '0.20',
                'recall_per_class': '0.87700', 'far_per_class': '220.516',
                'cost_per_class': '0.83554', 'saving_pct_vs_literature': '10.59',
            },
            '10': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '1.76217', 'threshold_single': '0.15',
                'recall_single': '0.88873', 'far_single': '344.033', 'cost_single': '1.45676',
                'threshold_fire': '0.15', 'threshold_smoke': '0.15',
                'recall_per_class': '0.88873', 'far_per_class': '344.033',
                'cost_per_class': '1.45676', 'saving_pct_vs_literature': '17.33',
            },
            '25': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '4.24528', 'threshold_single': '0.10',
                'recall_single': '0.90069', 'far_single': '704.815', 'cost_single': '3.18744',
                'threshold_fire': '0.10', 'threshold_smoke': '0.10',
                'recall_per_class': '0.90069', 'far_per_class': '704.815',
                'cost_per_class': '3.18744', 'saving_pct_vs_literature': '24.92',
            },
            '50': {
                'recall_literature': '0.83446', 'far_literature': '106.769',
                'cost_literature': '8.38378', 'threshold_single': '0.10',
                'recall_single': '0.90069', 'far_single': '704.815', 'cost_single': '5.67007',
                'threshold_fire': '0.10', 'threshold_smoke': '0.10',
                'recall_per_class': '0.90069', 'far_per_class': '704.815',
                'cost_per_class': '5.67007', 'saving_pct_vs_literature': '32.37',
            },
        },
    },
    'far_ci': {
        'yolo11n_seed42': ['25.820', '17.446', '34.194'],
        'yolo11n_seed1337': ['33.496', '24.424', '42.568'],
        'yolo11n_seed2024': ['30.705', '22.331', '39.777'],
        'resnet18fpn_seed42': ['11.863', '6.978', '17.446'],
        'resnet18fpn_seed1337': ['6.978', '3.472', '11.863'],
        'resnet18fpn_seed2024': ['6.978', '2.791', '11.863'],
        'no_negatives_seed42': ['306.350', '281.228', '330.077'],
        'size_matched_seed42': ['6.281', '2.094', '10.468'],
    },
    'far_paired': {
        'Size-matched - No-negatives': ['-300.070', '-323.098', '-275.645', '0.0000'],
        'Size-matched - ResNet18+FPN+FCOS': ['-2.326', '-7.444', '2.559', '0.3590'],
        'Size-matched - YOLO11n': ['-23.726', '-30.472', '-17.679', '0.0000'],
        'No-negatives - ResNet18+FPN+FCOS': ['297.744', '273.319', '321.011', '0.0000'],
        'No-negatives - YOLO11n': ['276.343', '251.448', '301.000', '0.0000'],
        'ResNet18+FPN+FCOS - YOLO11n': ['-21.400', '-28.611', '-14.887', '0.0000'],
    },
    'seed_spread': {
        'ResNet18+FPN+FCOS|map50': ['0.66461', '0.00331'],
        'ResNet18+FPN+FCOS|map50_95': ['0.31708', '0.00127'],
        'ResNet18+FPN+FCOS|ap50_fire': ['0.58030', '0.00412'],
        'ResNet18+FPN+FCOS|ap50_smoke': ['0.74893', '0.00326'],
        'ResNet18+FPN+FCOS|precision': ['0.72143', '0.06662'],
        'ResNet18+FPN+FCOS|recall': ['0.60906', '0.04104'],
        'ResNet18+FPN+FCOS|far_per_1000': ['8.60633', '2.82036'],
        'YOLO11n|map50': ['0.71143', '0.00510'],
        'YOLO11n|map50_95': ['0.39594', '0.00208'],
        'YOLO11n|ap50_fire': ['0.64616', '0.00807'],
        'YOLO11n|ap50_smoke': ['0.77671', '0.00486'],
        'YOLO11n|precision': ['0.71864', '0.00704'],
        'YOLO11n|recall': ['0.68194', '0.00222'],
        'YOLO11n|far_per_1000': ['30.00700', '3.88531'],
    },
    'seedmean_far_ci': {
        'ResNet18+FPN+FCOS|0.5': ['4.015', '1.616', '6.973'],
        'ResNet18+FPN+FCOS|0.6': ['7.847', '4.031', '12.233'],
        'ResNet18+FPN+FCOS|0.7': ['16.182', '10.512', '22.414'],
        'ResNet18+FPN+FCOS|0.8': ['48.725', '37.121', '63.888'],
        'YOLO11n|0.5': ['5.799', '2.673', '9.265'],
        'YOLO11n|0.6': ['15.107', '9.912', '21.195'],
        'YOLO11n|0.7': ['36.515', '27.140', '46.890'],
        'YOLO11n|0.8': ['97.437', '79.245', '118.161'],
    },
    'seedmean_cost_ci': {
        'ResNet18+FPN+FCOS|1': ['0.2200', '0.2030', '0.2378'],
        'ResNet18+FPN+FCOS|2': ['0.3845', '0.3555', '0.4164'],
        'ResNet18+FPN+FCOS|5': ['0.7868', '0.7236', '0.8568'],
        'ResNet18+FPN+FCOS|10': ['1.3862', '1.2650', '1.5185'],
        'ResNet18+FPN+FCOS|25': ['3.0594', '2.7754', '3.3798'],
        'ResNet18+FPN+FCOS|50': ['5.6279', '5.0657', '6.2406'],
        'YOLO11n|1': ['0.2674', '0.2496', '0.2862'],
        'YOLO11n|2': ['0.4561', '0.4254', '0.4880'],
        'YOLO11n|5': ['0.9418', '0.8749', '1.0126'],
        'YOLO11n|10': ['1.7442', '1.6150', '1.8829'],
        'YOLO11n|25': ['4.1514', '3.8292', '4.4960'],
        'YOLO11n|50': ['8.1634', '7.5175', '8.8478'],
    },
    'paired_bootstrap': {
        'far@0.5': ['1.783', '-1.937', '5.583', '0.3520'],
        'far@0.6': ['7.260', '1.251', '13.293', '0.0180'],
        'far@0.7': ['20.333', '11.517', '29.472', '0.0010'],
        'far@0.8': ['48.712', '33.168', '64.339', '0.0010'],
        'cost@1': ['0.0474', '0.0310', '0.0623', '0.0010'],
        'cost@2': ['0.0717', '0.0459', '0.0970', '0.0010'],
        'cost@5': ['0.1549', '0.0975', '0.2101', '0.0010'],
        'cost@10': ['0.3579', '0.2442', '0.4692', '0.0010'],
        'cost@25': ['1.0920', '0.8156', '1.3651', '0.0010'],
        'cost@50': ['2.5355', '1.9715', '3.1125', '0.0010'],
    },
    'errors': {
        'yolo11n_seed42': {
            'threshold': '0.25',
            'alarm_bands': ['10', '20', '4', '3'],
            'missed_by_size': ['219', '524', '366', '184'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.006319',
            'median_caught': '0.028062',
            'caught_over_missed': '4.44',
        },
        'yolo11n_seed1337': {
            'threshold': '0.25',
            'alarm_bands': ['10', '27', '8', '3'],
            'missed_by_size': ['227', '524', '368', '179'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.005832',
            'median_caught': '0.028182',
            'caught_over_missed': '4.83',
        },
        'yolo11n_seed2024': {
            'threshold': '0.25',
            'alarm_bands': ['13', '26', '4', '1'],
            'missed_by_size': ['229', '527', '375', '182'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.006228',
            'median_caught': '0.028173',
            'caught_over_missed': '4.52',
        },
        'resnet18fpn_seed42': {
            'threshold': '0.45',
            'alarm_bands': ['0', '6', '11', '0'],
            'missed_by_size': ['318', '627', '332', '139'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.003313',
            'median_caught': '0.040173',
            'caught_over_missed': '12.13',
        },
        'resnet18fpn_seed1337': {
            'threshold': '0.50',
            'alarm_bands': ['0', '0', '10', '0'],
            'missed_by_size': ['323', '740', '427', '205'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.004403',
            'median_caught': '0.046866',
            'caught_over_missed': '10.64',
        },
        'resnet18fpn_seed2024': {
            'threshold': '0.50',
            'alarm_bands': ['0', '0', '10', '0'],
            'missed_by_size': ['329', '722', '448', '214'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.004621',
            'median_caught': '0.043479',
            'caught_over_missed': '9.41',
        },
        'no_negatives_seed42': {
            'threshold': '0.50',
            'alarm_bands': ['0', '0', '435', '4'],
            'missed_by_size': ['328', '746', '434', '164'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.004170',
            'median_caught': '0.050781',
            'caught_over_missed': '12.18',
        },
        'size_matched_seed42': {
            'threshold': '0.50',
            'alarm_bands': ['0', '0', '9', '0'],
            'missed_by_size': ['328', '707', '512', '227'],
            'total_by_size': ['339', '1366', '1168', '1184'],
            'median_missed': '0.005496',
            'median_caught': '0.045911',
            'caught_over_missed': '8.35',
        },
    },
    'calibration': {
        'yolo11n_seed42': {
            'all': {
                'n': '8412', 'mean_score': '0.32125', 'fraction_correct': '0.40157',
                'ece': '0.08032', 'mce': '0.21631',
            },
            'fire': {
                'n': '5157', 'mean_score': '0.28008', 'fraction_correct': '0.35815',
                'ece': '0.07807', 'mce': '0.20659',
            },
            'smoke': {
                'n': '3255', 'mean_score': '0.38647', 'fraction_correct': '0.47035',
                'ece': '0.08463', 'mce': '0.23313',
            },
        },
        'yolo11n_seed1337': {
            'all': {
                'n': '8955', 'mean_score': '0.30925', 'fraction_correct': '0.37956',
                'ece': '0.07031', 'mce': '0.20193',
            },
            'fire': {
                'n': '5563', 'mean_score': '0.26946', 'fraction_correct': '0.33327',
                'ece': '0.06381', 'mce': '0.20618',
            },
            'smoke': {
                'n': '3392', 'mean_score': '0.37451', 'fraction_correct': '0.45548',
                'ece': '0.08097', 'mce': '0.20354',
            },
        },
        'yolo11n_seed2024': {
            'all': {
                'n': '8739', 'mean_score': '0.31081', 'fraction_correct': '0.39020',
                'ece': '0.07939', 'mce': '0.21829',
            },
            'fire': {
                'n': '5319', 'mean_score': '0.27176', 'fraction_correct': '0.34931',
                'ece': '0.07755', 'mce': '0.23543',
            },
            'smoke': {
                'n': '3420', 'mean_score': '0.37155', 'fraction_correct': '0.45380',
                'ece': '0.08225', 'mce': '0.21034',
            },
        },
        'resnet18fpn_seed42': {
            'all': {
                'n': '726609', 'mean_score': '0.09381', 'fraction_correct': '0.00500',
                'ece': '0.08993', 'mce': '0.28187',
            },
            'fire': {
                'n': '469422', 'mean_score': '0.08893', 'fraction_correct': '0.00407',
                'ece': '0.08568', 'mce': '0.26970',
            },
            'smoke': {
                'n': '257187', 'mean_score': '0.10271', 'fraction_correct': '0.00670',
                'ece': '0.09771', 'mce': '0.29817',
            },
        },
        'resnet18fpn_seed1337': {
            'all': {
                'n': '477276', 'mean_score': '0.10331', 'fraction_correct': '0.00765',
                'ece': '0.09757', 'mce': '0.28346',
            },
            'fire': {
                'n': '198289', 'mean_score': '0.11103', 'fraction_correct': '0.00973',
                'ece': '0.10338', 'mce': '0.27161',
            },
            'smoke': {
                'n': '278987', 'mean_score': '0.09782', 'fraction_correct': '0.00617',
                'ece': '0.09345', 'mce': '0.29829',
            },
        },
        'resnet18fpn_seed2024': {
            'all': {
                'n': '502937', 'mean_score': '0.10443', 'fraction_correct': '0.00726',
                'ece': '0.09895', 'mce': '0.27884',
            },
            'fire': {
                'n': '249024', 'mean_score': '0.10635', 'fraction_correct': '0.00776',
                'ece': '0.10019', 'mce': '0.26469',
            },
            'smoke': {
                'n': '253913', 'mean_score': '0.10255', 'fraction_correct': '0.00676',
                'ece': '0.09773', 'mce': '0.29883',
            },
        },
    },
    'qualitative': {
        'a': {'sample_id': 'dfire/train/images/WEB04445', 'value': '0.9734', 'score': '0.8542'},
        'b': {'sample_id': 'dfire/test/images/WEB10030', 'value': '0.000527', 'score': 'nan'},
        'c': {'sample_id': 'dfire/train/images/WEB09048', 'value': '0.7426', 'score': '0.7426'},
    },
}

In [ ]:
# ---------------------------------------------------------------- manuscript: tables
t2 = TABLE_2.set_index("split")
for split, exp in PAPER_TABLES["Table 2"].items():
    r = t2.loc[split]
    for col, e in zip(["images", "with_objects", "verified_negative", "pct_negative",
                       "fire_boxes", "smoke_boxes"], exp[:6]):
        check("Table 2", f"{split} {col}", r[col], e)
    check("Table 2", f"{split} manifest hash", r.manifest_hash, exp[6])
    check("Table 2", f"{split} frozen", "yes" if r.frozen else "no", exp[7])

t3 = TABLE_3.set_index(["split", "class"])
for key, exp in PAPER_TABLES["Table 3"].items():
    r = t3.loc[key]
    check("Table 3", f"{key} boxes", r.boxes, exp[0])
    check("Table 3", f"{key} median area", r.median_area_fraction, exp[1])
    check("Table 3", f"{key} % below 1%", r.pct_boxes_below_1pct_of_image, exp[2])

t5 = TABLE_5.set_index("condition")
for cond, exp in PAPER_TABLES["Table 5"].items():
    r = t5.loc[cond]
    for col, e in zip(["training_images", "positives", "negatives", "train_manifest_hash"], exp):
        check("Table 5", f"{cond} {col}", r[col], e)
    check("Table 5", f"{cond} hash matches the run record", str(bool(r.matches_training_run_record)), "True")

for arch, exp in PAPER_TABLES["Table 6"].items():
    g = MAIN_METRICS[MAIN_METRICS.model == arch]
    check("Table 6", f"{arch} seeds", len(g), exp["seeds"])
    check("Table 6", f"{arch} threshold", threshold_label(g.threshold), exp["thr"])
    for key in ("map50", "map50_95", "ap50_fire", "ap50_smoke", "recall", "precision", "far_per_1000"):
        mean, sd = SEED_STATS[(arch, key)]
        check("Table 6", f"{arch} {key} mean", mean, exp[key][0])
        check("Table 6", f"{arch} {key} sd", sd, exp[key][1])

t7 = TABLE_7.set_index(["model", "seed"])
for key, (m, f, t) in PAPER_TABLES["Table 7"].items():
    check("Table 7", f"{key} mAP50", t7.loc[key, "map50"], m)
    check("Table 7", f"{key} FAR", t7.loc[key, "far_per_1000"], f)
    check("Table 7", f"{key} threshold", t7.loc[key, "threshold"], t)

t8 = TABLE_8.set_index("recall")
for rt, exp in PAPER_TABLES["Table 8"].items():
    r = t8.loc[rt]
    check("Table 8", f"recall {rt} YOLO11n mean", r.yolo11n_mean, exp[0])
    check("Table 8", f"recall {rt} ResNet mean", r.resnet_mean, exp[1])
    check("Table 8", f"recall {rt} ratio", r.yolo11n_mean / r.resnet_mean, exp[2])
    check("Table 8", f"recall {rt} paired difference", r.paired_difference, exp[3])
    check("Table 8", f"recall {rt} interval low", r.ci_low, exp[4])
    check("Table 8", f"recall {rt} interval high", r.ci_high, exp[5])

for arch, (prec, far) in PAPER_TABLES["Table 9"].items():
    for key, (m, s) in (("precision", prec), ("far_per_1000", far)):
        check("Table 9", f"{arch} {key} mean", SEED_STATS[(arch, key)][0], m)
        check("Table 9", f"{arch} {key} sd", SEED_STATS[(arch, key)][1], s)

t10 = TABLE_10.set_index("r")
for ratio, exp in PAPER_TABLES["Table 10"].items():
    r = t10.loc[ratio]
    check("Table 10", f"r={ratio} YOLO11n mean", r.yolo11n_mean, exp[0])
    check("Table 10", f"r={ratio} ResNet mean", r.resnet_mean, exp[1])
    check("Table 10", f"r={ratio} lower cost", r.lower_cost.split("18")[0], exp[2])
    check("Table 10", f"r={ratio} mAP50 winner", r.map50_winner, exp[3])
    check("Table 10", f"r={ratio} paired difference", r.paired_difference, exp[4])
    check("Table 10", f"r={ratio} interval low", r.ci_low, exp[5])
    check("Table 10", f"r={ratio} interval high", r.ci_high, exp[6])

t11 = TABLE_11.set_index("run_id")
for run_id, (f, lo, hi) in PAPER_TABLES["Table 11"].items():
    check("Table 11", f"{run_id} FAR", t11.loc[run_id, "far_per_1000"], f)
    check("Table 11", f"{run_id} low", t11.loc[run_id, "ci_low"], lo)
    check("Table 11", f"{run_id} high", t11.loc[run_id, "ci_high"], hi)

for arch, (missed, caught, ratios) in PAPER_TABLES["Table 12"].items():
    g = SIZE_SUMMARY[SIZE_SUMMARY.model == arch].reset_index(drop=True)
    for i, seed in enumerate(g.seed):
        check("Table 12", f"{arch} seed {seed} median missed", g.median_missed[i], missed[i])
        check("Table 12", f"{arch} seed {seed} median caught", g.median_caught[i], caught[i])
        check("Table 12", f"{arch} seed {seed} caught/missed", g.caught_over_missed[i], ratios[i])

t13 = TABLE_13.set_index("condition")
for cond, exp in PAPER_TABLES["Table 13"].items():
    r = t13.loc[cond]
    fired, total = r.negatives_fired.split(" of ")
    for label, value, e in zip(["training images", "mAP50", "AP fire", "AP smoke", "FAR", "negatives fired",
                                "negatives"],
                               [r.training_images, r.map50, r.ap50_fire, r.ap50_smoke, r.far_per_1000,
                                int(fired), int(total.replace(",", ""))], exp):
        check("Table 13", f"{cond} {label}", value, e)

t14 = PAPER_TABLES["Table 14"]
check("Table 14", "halving: delta mAP50", ABLATION["delta_map50_halving"], t14["halving"]["delta_map50"])
check("Table 14", "halving: multiple of the noise floor", ABLATION["halving_over_noise_floor"],
      t14["halving"]["times_noise_floor"])
check("Table 14", "halving: FAR before", ABLATION["far_full"], t14["halving"]["far_from"])
check("Table 14", "halving: FAR after", ABLATION["far_size_matched"], t14["halving"]["far_to"])
check("Table 14", "ResNet seed FAR range, low", ABLATION["resnet_seed_far_min"], t14["halving"]["seed_range_low"])
check("Table 14", "ResNet seed FAR range, high", ABLATION["resnet_seed_far_max"], t14["halving"]["seed_range_high"])
check("Table 14", "removing negatives: delta mAP50", ABLATION["delta_map50_removing_negatives"],
      t14["removing"]["delta_map50"])
check("Table 14", "removing negatives: FAR before", ABLATION["far_size_matched"], t14["removing"]["far_from"])
check("Table 14", "removing negatives: FAR after", ABLATION["far_no_negatives"], t14["removing"]["far_to"])

gpu = GPU.set_index("model")
for model, exp in PAPER_TABLES["Table 15"].items():
    for col, e in zip(["params_m", "model_size_mb", "latency_mean_ms", "fps", "peak_vram_gb"], exp):
        check("Table 15", f"{model} {col} (recorded)", gpu.loc[model, col], e)

# ---------------------------------------------------------------- manuscript: Section III-H
rep = RUN_METRICS.set_index("run_id").loc["resnet18fpn_seed42"]
for key, e in PAPER_REPRODUCTION.items():
    check("Reproduction (III-H)", key, rep[key], e)

# ---------------------------------------------------------------- manuscript: running text
pred_counts = PREDICTIONS_TABLE.set_index(["run_id", "split"]).boxes
t8_ratios = [r.yolo11n_mean / r.resnet_mean for r in TABLE_8.itertuples() if r.recall >= 0.6]
size_ratios = SIZE_SUMMARY[SIZE_SUMMARY.model.isin(ARCHITECTURES)].caught_over_missed
text_computed = {
    "images_total": DATASET_AUDIT["images"],
    "negatives_total": DATASET_AUDIT["negative_images"],
    "fire_box_shortfall": DFIRE_PUBLISHED["fire_boxes"] - DATASET_AUDIT["fire_boxes"],
    "smoke_box_shortfall": DFIRE_PUBLISHED["smoke_boxes"] - DATASET_AUDIT["smoke_boxes"],
    "median_fire_area_train": train_scale.loc["fire", "median_area_fraction"],
    "median_smoke_area_train": train_scale.loc["smoke", "median_area_fraction"],
    "area_ratio_train": TEXT_VALUES["smoke_over_fire_median_area_train"],
    "area_ratio_test": TEXT_VALUES["smoke_over_fire_median_area_test"],
    "official_test_in_train": DATASET_AUDIT["official_test_images_in_train"],
    "yolo_map50_pct": 100 * y_map[0], "yolo_map50_sd_pct": 100 * y_map[1],
    "yolo_far_1dp": y_far[0], "yolo_far_sd_1dp": y_far[1],
    "resnet_far_1dp": r_far[0], "resnet_far_sd_1dp": r_far[1],
    "map50_gain_points": TEXT_VALUES["map50_gain_points"],
    "far_ratio": TEXT_VALUES["far_ratio_yolo_over_resnet"],
    "matched_recall_ratio_min": min(t8_ratios), "matched_recall_ratio_max": max(t8_ratios),
    "cv_far_yolo": TEXT_VALUES["cv_far_yolo_pct"], "cv_far_resnet": TEXT_VALUES["cv_far_resnet_pct"],
    "cv_map_yolo": TEXT_VALUES["cv_map50_yolo_pct"], "cv_map_resnet": TEXT_VALUES["cv_map50_resnet_pct"],
    "gap_yolo": FIRE_SMOKE.set_index("model").loc["YOLO11n", "gap_smoke_minus_fire"],
    "gap_resnet": FIRE_SMOKE.set_index("model").loc["ResNet18+FPN+FCOS", "gap_smoke_minus_fire"],
    "gap_noise_yolo": FIRE_SMOKE.set_index("model").loc["YOLO11n", "gap_over_noise_floor"],
    "gap_noise_resnet": FIRE_SMOKE.set_index("model").loc["ResNet18+FPN+FCOS", "gap_over_noise_floor"],
    "p_recall_05": TABLE_8.set_index("recall").loc[0.5, "p_two_sided"],
    "caught_missed_min": size_ratios.min(), "caught_missed_max": size_ratios.max(),
    "listing_instances": len(ERROR_LISTING), "listing_images": ERROR_LISTING.sample_id.nunique(),
    "fig7_missed_area_pct": 100 * frac_b,
    "ablation_delta_map": ABLATION["delta_map50_removing_negatives"],
    "noise_floor": ABLATION["noise_floor"],
    "ablation_far_from_1dp": ABLATION["far_size_matched"], "ablation_far_to_1dp": ABLATION["far_no_negatives"],
    "ablation_far_ratio": ABLATION["far_ratio_no_neg_over_size_matched"],
    "pct_alarmed_no_neg": ABLATION["pct_negatives_alarmed_no_neg"],
    "pct_alarmed_matched": ABLATION["pct_negatives_alarmed_size_matched"],
    "equal_recall_ratio_05": ABLATION["equal_recall_ratio_0.5"],
    "equal_recall_ratio_08": ABLATION["equal_recall_ratio_0.8"],
    "dense_boxes_resnet": pred_counts.loc[("resnet18fpn_seed42", TEST_SPLIT)],
    "dense_boxes_yolo": pred_counts.loc[("yolo11n_seed42", TEST_SPLIT)],
    "dense_factor": pred_counts.loc[("resnet18fpn_seed42", TEST_SPLIT)] / pred_counts.loc[("yolo11n_seed42", TEST_SPLIT)],
}
for key, (where, what, printed) in PAPER_TEXT.items():
    check("Text and captions", f"{where}: {what}", text_computed[key], printed)
print(f"{len(CHECKS)} comparisons against the manuscript recorded")

In [ ]:
# ---------------------------------------------------------------- figures: the plotted data
A = EXPECTED_ARTIFACTS
SRC = "original analysis"
fs = FIRE_SMOKE.set_index("model")
for arch in ARCHITECTURES:
    runs = [r.run_id for r in RUNS if r.model == arch]
    check("Figure 1", f"{arch} mean AP fire", fs.loc[arch, "ap_fire"],
          f"{np.mean([float(A['run_metrics'][r]['ap50_fire']) for r in runs]):.6f}", SRC)
    check("Figure 1", f"{arch} mean AP smoke", fs.loc[arch, "ap_smoke"],
          f"{np.mean([float(A['run_metrics'][r]['ap50_smoke']) for r in runs]):.6f}", SRC)
for cls in CLASSES:
    check("Figure 1", f"test median area {cls}", test_scale.loc[cls, "median_area_fraction"],
          A["object_scale"][f"test/{cls}"]["median_area"], SRC)
check("Figure 1", "caption: test median fire box smaller by", TEXT_VALUES["smoke_over_fire_median_area_test"], "24")

fig2 = FIG2_DATA.set_index(["model", "threshold"])
for arch in ARCHITECTURES:
    runs = [r.run_id for r in RUNS if r.model == arch]
    for j, t in enumerate(THRESHOLD_SWEEP):
        exp_recall = np.mean([float(A["curve"][r]["recall"][j]) for r in runs])
        exp_far = np.mean([float(A["curve"][r]["far_per_1000"][j]) for r in runs])
        check("Figure 2", f"{arch} t={t:.2f} recall", fig2.loc[(arch, round(float(t), 2)), "recall"],
              f"{exp_recall:.6f}", SRC)
        check("Figure 2", f"{arch} t={t:.2f} FAR", fig2.loc[(arch, round(float(t), 2)), "far"],
              f"{exp_far:.4f}", SRC)

for arch in ARCHITECTURES:
    runs = [r.run_id for r in RUNS if r.model == arch]
    for ratio in COST_RATIOS:
        exp_cost = np.mean([float(A["cost"][r][str(ratio)]["cost_per_class"]) for r in runs])
        check("Figure 3", f"{arch} r={ratio} mean cost", COST_MEAN.loc[ratio, arch], f"{exp_cost:.6f}", SRC)

for run in MAIN_RUNS:
    row = RUN_METRICS.set_index("run_id").loc[run.run_id]
    check("Figure 4", f"{run.run_id} mAP50", row.map50, A["run_metrics"][run.run_id]["map50"], SRC)
    check("Figure 4", f"{run.run_id} FAR", row.far_per_1000, A["run_metrics"][run.run_id]["far_per_1000"], SRC)

f5 = FIG5_DATA.set_index("model")
for arch in ARCHITECTURES:
    runs = [r.run_id for r in RUNS if r.model == arch]
    check("Figure 5", f"{arch} mean median missed", f5.loc[arch, "missed"],
          f"{np.mean([float(A['errors'][r]['median_missed']) for r in runs]):.8f}", SRC)
    check("Figure 5", f"{arch} mean median caught", f5.loc[arch, "caught"],
          f"{np.mean([float(A['errors'][r]['median_caught']) for r in runs]):.8f}", SRC)

cal = CALIBRATION.set_index(["run_id", "class"])
for run in MAIN_RUNS:
    for cls in ("all",) + CLASSES:
        exp = A["calibration"][run.run_id][cls]
        check("Figure 6", f"{run.run_id} {cls} ECE", cal.loc[(run.run_id, cls), "ece"], exp["ece"], SRC)
        check("Figure 6", f"{run.run_id} {cls} MCE", cal.loc[(run.run_id, cls), "mce"], exp["mce"], SRC)
        check("Figure 6", f"{run.run_id} {cls} detections", cal.loc[(run.run_id, cls), "detections"], exp["n"], SRC)

q = A["qualitative"]
check("Figure 7", "panel (a) sample", sid_a, q["a"]["sample_id"], SRC)
check("Figure 7", "panel (a) IoU", iou_a, q["a"]["value"], SRC)
check("Figure 7", "panel (a) confidence", det_a["score"], q["a"]["score"], SRC)
check("Figure 7", "panel (b) sample", sid_b, q["b"]["sample_id"], SRC)
check("Figure 7", "panel (b) area fraction", frac_b, q["b"]["value"], SRC)
check("Figure 7", "panel (c) sample", sid_c, q["c"]["sample_id"], SRC)
check("Figure 7", "panel (c) confidence", score_c, q["c"]["value"], SRC)
check("Figure 7", "caption: missed box (% of image)", 100 * frac_b, "0.05")
# The line styles of Figure 7 are new in this release (dashed annotation, solid detection), so they
# are checked against the figure as drawn rather than against the earlier published image.
if FIG7_DRAWN:
    check("Figure 7", "annotation boxes drawn (two panels plus the zoom inset)", FIG7_DRAWN["annotation_boxes"], "3",
          "figure as drawn")
    check("Figure 7", "annotation boxes dashed", FIG7_DRAWN["annotation_boxes_dashed"], "3", "figure as drawn")
    check("Figure 7", "detection boxes drawn", FIG7_DRAWN["detection_boxes"], "2", "figure as drawn")
    check("Figure 7", "detection boxes solid", FIG7_DRAWN["detection_boxes_solid"], "2", "figure as drawn")

for cond, run_id in ABLATION_ORDER:
    row = RUN_METRICS.set_index("run_id").loc[run_id]
    check("Figure 8", f"{cond} mAP50", row.map50, A["run_metrics"][run_id]["map50"], SRC)
    check("Figure 8", f"{cond} FAR", row.far_per_1000, A["run_metrics"][run_id]["far_per_1000"], SRC)

# ---------------------------------------------------------------- original analysis, full precision
rm = RUN_METRICS.set_index("run_id")
vs = VAL_SELECTION.set_index("run_id")
for run in RUNS:
    for key, e in A["run_metrics"][run.run_id].items():
        check("Per-run metrics", f"{run.run_id} {key}", rm.loc[run.run_id, key], e, SRC)
    for key, e in A["val_selection"][run.run_id].items():
        check("Validation threshold selection", f"{run.run_id} {key}",
              vs.loc[run.run_id, "threshold" if key == "threshold" else f"val_{key}"], e, SRC)
    for split in (SELECTION_SPLIT, TEST_SPLIT):
        p = PREDICTIONS_TABLE.set_index(["run_id", "split"]).loc[(run.run_id, split)]
        check("Stored predictions", f"{run.run_id} {split} images", p.images_in_file,
              A["predictions"][run.run_id][split]["images"], SRC)
        check("Stored predictions", f"{run.run_id} {split} boxes", p.boxes,
              A["predictions"][run.run_id][split]["boxes"], SRC)
    curve = CURVES[run.run_id]
    for col in ("recall", "precision", "far_per_1000", "negatives_fired"):
        for j, e in enumerate(A["curve"][run.run_id][col]):
            check("False-alarm curves", f"{run.run_id} {col} @ {THRESHOLD_SWEEP[j]:.2f}", curve[col].iloc[j], e, SRC)
    far_r = FAR_AT_RECALL.set_index("run_id")
    for rt, e in A["far_at_recall"][run.run_id].items():
        check("FAR at fixed recall", f"{run.run_id} recall {rt}", far_r.loc[run.run_id, f"far_at_recall_{rt}"], e, SRC)
    costs = COSTS[COSTS.run_id == run.run_id].set_index("ratio")
    names = {"recall_literature": "recall_literature", "far_literature": "far_per_1000_literature",
             "cost_literature": "cost_literature", "threshold_single": "threshold_single",
             "recall_single": "recall_single", "far_single": "far_per_1000_single", "cost_single": "cost_single",
             "threshold_fire": "threshold_fire", "threshold_smoke": "threshold_smoke",
             "recall_per_class": "recall_per_class", "far_per_class": "far_per_1000_per_class",
             "cost_per_class": "cost_per_class", "saving_pct_vs_literature": "saving_pct_vs_literature"}
    for ratio, values in A["cost"][run.run_id].items():
        for key, e in values.items():
            check("Expected cost (all policies)", f"{run.run_id} r={ratio} {key}",
                  costs.loc[int(ratio), names[key]], e, SRC)
    ci = TABLE_11.set_index("run_id").loc[run.run_id]
    for label, value, e in zip(("FAR", "low", "high"), (ci.far_per_1000, ci.ci_low, ci.ci_high),
                               A["far_ci"][run.run_id]):
        check("FAR bootstrap per run", f"{run.run_id} {label}", value, e, SRC)
    err = ERRORS[ERRORS.run_id == run.run_id]
    exp_err = A["errors"][run.run_id]
    for i, e in enumerate(exp_err["alarm_bands"]):
        check("Error analysis", f"{run.run_id} alarm band {i}",
              err[err.kind.str.startswith("false alarm")]["count"].iloc[i], e, SRC)
    miss_rows = err[err.kind.str.startswith("missed")]
    for i, (e_m, e_t) in enumerate(zip(exp_err["missed_by_size"], exp_err["total_by_size"])):
        check("Error analysis", f"{run.run_id} size band {i} missed", miss_rows["count"].iloc[i], e_m, SRC)
        check("Error analysis", f"{run.run_id} size band {i} total", miss_rows["out_of"].iloc[i], e_t, SRC)
    srow = SIZE_SUMMARY.set_index("run_id").loc[run.run_id]
    check("Error analysis", f"{run.run_id} median missed", srow.median_missed, exp_err["median_missed"], SRC)
    check("Error analysis", f"{run.run_id} median caught", srow.median_caught, exp_err["median_caught"], SRC)
    check("Error analysis", f"{run.run_id} caught/missed", srow.caught_over_missed, exp_err["caught_over_missed"], SRC)

fp = FAR_PAIRED.set_index("comparison")
for comparison, (d, lo, hi, p) in A["far_paired"].items():
    r = fp.loc[comparison]
    for label, value, e in zip(("difference", "low", "high", "p"),
                               (r.difference_per_1000, r.ci_low, r.ci_high, r.p_two_sided), (d, lo, hi, p)):
        check("FAR paired comparisons", f"{comparison} {label}", value, e, SRC)

for key, (m, s) in A["seed_spread"].items():
    arch, metric = key.split("|")
    check("Seed spread", f"{arch} {metric} mean", round(SEED_STATS[(arch, metric)][0], 5), m, SRC)
    check("Seed spread", f"{arch} {metric} sd", round(SEED_STATS[(arch, metric)][1], 5), s, SRC)

sf = SEEDMEAN_FAR_CI.set_index(["model", "recall"])
for key, (m, lo, hi) in A["seedmean_far_ci"].items():
    model, rt = key.split("|")
    r = sf.loc[(model, float(rt))]
    for label, value, e in zip(("mean", "low", "high"), (r.far_per_1000, r.ci_low, r.ci_high), (m, lo, hi)):
        check("Seed-mean intervals", f"FAR {model} recall {rt} {label}", value, e, SRC)
sc = SEEDMEAN_COST_CI.set_index(["model", "ratio"])
for key, (m, lo, hi) in A["seedmean_cost_ci"].items():
    model, ratio = key.split("|")
    r = sc.loc[(model, int(ratio))]
    for label, value, e in zip(("mean", "low", "high"), (r.cost, r.ci_low, r.ci_high), (m, lo, hi)):
        check("Seed-mean intervals", f"cost {model} r={ratio} {label}", value, e, SRC)
pb = PAIRED.set_index("key")
for key, (d, lo, hi, p) in A["paired_bootstrap"].items():
    r = pb.loc[key]
    for label, value, e in zip(("difference", "low", "high", "p"),
                               (r.difference, r.ci_low, r.ci_high, r.p_two_sided), (d, lo, hi, p)):
        check("Paired bootstrap (Tables 8, 10)", f"{key} {label}", value, e, SRC)

for split, values in A["dataset"].items():
    r = TABLE_2.set_index("split").loc[split]
    for key, e in values.items():
        check("Dataset composition", f"{split} {key}", r[key], e, SRC)
for key, values in A["object_scale"].items():
    split, cls = key.split("/")
    r = TABLE_3.set_index(["split", "class"]).loc[(split, cls)]
    for k, col in (("boxes", "boxes"), ("median_area", "median_area_fraction"),
                   ("mean_area", "mean_area_fraction"), ("pct_below_1pct", "pct_boxes_below_1pct_of_image")):
        check("Object scale", f"{key} {k}", r[col], values[k], SRC)
for run_id, classes in A["calibration"].items():
    for cls, values in classes.items():
        r = cal.loc[(run_id, cls)]
        check("Calibration", f"{run_id} {cls} mean score", r.mean_score, values["mean_score"], SRC)
        check("Calibration", f"{run_id} {cls} fraction correct", r.fraction_correct, values["fraction_correct"], SRC)

print(f"{len(CHECKS)} comparisons recorded in total")

### Statements in the text

In [ ]:
populated_bins = CALIBRATION_BIN_TABLE[CALIBRATION_BIN_TABLE.n > 0].merge(
    pd.DataFrame([{"run_id": r.run_id, "model": r.model} for r in MAIN_RUNS]), on="run_id")
under_confident = populated_bins.fraction_correct > populated_bins.mean_score
over_bins = populated_bins[~under_confident]
paired10 = PAIRED[PAIRED.table == "Table 10"]
paired8 = PAIRED[PAIRED.table == "Table 8"].set_index("key")

STATEMENTS = [
    ("IV-B", "Fire AP is lower than smoke AP in all six runs",
     bool((MAIN_METRICS.ap50_fire < MAIN_METRICS.ap50_smoke).all())),
    ("IV-E", "ResNet18+FPN+FCOS has the lower expected cost at every cost ratio",
     bool((TABLE_10.lower_cost == "ResNet18+FPN+FCOS").all())),
    ("I, IV-E", "The paired cost intervals exclude zero at every cost ratio",
     bool((~paired10.zero_inside).all())),
    ("IV-C", "At recall 0.60, 0.70 and 0.80 the paired alarm intervals exclude zero; at 0.50 the interval contains it",
     bool((~paired8.loc[["far@0.6", "far@0.7", "far@0.8"], "zero_inside"]).all() and paired8.loc["far@0.5", "zero_inside"])),
    ("IV-E", "YOLO11n's cost-optimal fire threshold sits at the 0.05 floor at every ratio and seed",
     FLOOR_FACTS["yolo_fire_at_floor_everywhere"]),
    ("IV-E", "YOLO11n's smoke threshold sits at the floor on every seed from r = 5 upwards",
     FLOOR_FACTS["yolo_smoke_at_floor_for_r_ge_5"]),
    ("IV-E", "ResNet18+FPN+FCOS reaches the floor only at r = 50, seed 42",
     FLOOR_FACTS["resnet_floor_cases"] == [(42, 50)]),
    ("IV-G", "In all six runs the median missed object is smaller than the median caught object",
     bool((SIZE_SUMMARY[SIZE_SUMMARY.model.isin(ARCHITECTURES)].caught_over_missed > 1).all())),
    ("IV-G", "The released error listing comes from the YOLO11n seed-42 run",
     bool((ERROR_LISTING.run_id == "yolo11n_seed42").all())),
    ("IV-H", "Removing the negatives moves mAP50 by less than the seed noise floor",
     abs(ABLATION["delta_map50_removing_negatives"]) <= ABLATION["noise_floor"]),
    ("III-C", "No image is shared across splits (identical file or perceptual hash)",
     DATASET_AUDIT["cross_split_identical_hashes"] == 0),
    ("III-G", "The validation manifest is identical across the three ablation conditions",
     TRAINING_RUNS.val_manifest_sha256.nunique() == 1),
    ("Table 14", "After halving the data, the alarm rate 6.28 lies inside the ResNet seed range 6.98-11.86",
     ABLATION["resnet_seed_far_min"] <= ABLATION["far_size_matched"] <= ABLATION["resnet_seed_far_max"]),
    ("Fig. 6 caption", "Each detector is under-confident (fraction correct above mean score) in every populated bin",
     bool(under_confident.all())),
]
STATEMENT_TABLE = pd.DataFrame([{"where": w, "statement": s, "status": "HOLDS" if ok else "DOES NOT HOLD"}
                                for w, s, ok in STATEMENTS])
save_table(STATEMENT_TABLE, "manuscript_statement_checks")
print(STATEMENT_TABLE.to_string(index=False))
if not bool(under_confident.all()):
    print("\nBins where the fraction correct is below the mean score (over-confident):")
    print(over_bins[["model", "run_id", "from", "to", "n", "mean_score", "fraction_correct"]].to_string(index=False))

### Summary

In [ ]:
CHECK_TABLE = pd.DataFrame(CHECKS)
save_table(CHECK_TABLE, "comparison_with_manuscript")
paper_groups = [f"Table {n}" for n in (2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)] + \
               ["Reproduction (III-H)", "Text and captions"] + [f"Figure {n}" for n in range(1, 9)]
summary = (CHECK_TABLE.assign(passed=CHECK_TABLE.status == "PASS")
           .groupby("group").agg(checks=("status", "size"), passed=("passed", "sum")))
summary["status"] = np.where(summary.checks == summary.passed, "PASS", "FAIL")
ordered = [g for g in paper_groups if g in summary.index] + [g for g in summary.index if g not in paper_groups]
summary = summary.loc[ordered]
print("Paper tables, figures and text:")
print(summary.loc[[g for g in ordered if g in paper_groups]].to_string())
print("\nOriginal analysis outputs (full precision):")
print(summary.loc[[g for g in ordered if g not in paper_groups]].to_string())

failures = CHECK_TABLE[CHECK_TABLE.status != "PASS"]
if len(failures):
    print("\nFAILED comparisons:")
    print(failures.to_string(index=False))
n_statement_issues = int((STATEMENT_TABLE.status != "HOLDS").sum())
print(f"\n{len(CHECK_TABLE)} comparisons, {len(failures)} failed. "
      f"{len(STATEMENT_TABLE)} statements checked, {n_statement_issues} not supported by the data "
      "(see the statement table above; these concern the manuscript text, not the computation).")
print(f"Total runtime: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
assert len(failures) == 0, "some computed values do not match the manuscript or the original analysis"
print("\nOVERALL: PASS - every computed table and figure matches the manuscript and the original analysis.")

## Appendix: retraining the detectors (optional, off by default)

Nothing above depends on this appendix. It documents how the stored predictions were produced and lets a reader with a GPU regenerate them. Two switches control it:

* `RUN_TRAINING` retrains the detectors and writes new prediction files in the format of Section 2. It needs a CUDA GPU, the D-Fire images (`FIRE_DFIRE_IMAGES`), the packages in `requirements-training.txt`, and many hours: one ResNet18+FPN+FCOS run took 5 to 7 hours and one YOLO11n run 2.5 to 13 hours on the RTX 4050 Laptop GPU of the paper.
* `RUN_SMOKE_TEST` builds the ResNet18+FPN+FCOS detector on the CPU, checks its parameter count against the 12.17 M of Tables 4 and 15, and runs one forward and backward pass and one optimiser step on a dummy batch, followed by decoding and NMS. It takes a few seconds and needs only PyTorch (plus `timm` for the exact backbone).

Retrained models will not reproduce the stored predictions bit for bit: GPU kernels, library versions and data-loading order all differ between machines. They should reproduce the paper's conclusions within the seed spread reported in Table 6.

The training configuration (Table 4 of the paper):

| | Reference track | In-house track |
|---|---|---|
| Model | YOLO11n (Ultralytics) | ResNet18 + FPN + FCOS head |
| Parameters | 2.59 M | 12.17 M |
| Initialisation | COCO-pretrained `yolo11n.pt` | ImageNet-pretrained ResNet18 backbone |
| Optimiser | AdamW | AdamW |
| Learning rate | 0.001, cosine decay to 1e-6 | 0.001 (backbone 0.0001), cosine decay to 1e-6 |
| Weight decay / warm-up | 0.0005 / 3 epochs | 0.0005 / 3 epochs |
| Epochs / batch | 60 / 8, Ultralytics nominal batch `nbs` = 64, i.e. 8-step gradient accumulation (effective batch 64) | 60 / 8 with 2-step gradient accumulation (effective batch 16) |
| Augmentation | h-flip 0.5, scale 0.3, translate 0.1, HSV 0.015/0.5/0.4; mosaic, mixup and copy-paste off | h-flip 0.5, scale 0.3, translate 0.1, HSV 0.015/0.5/0.4 |
| NMS IoU / stored confidence floor | 0.6 / 0.05 | 0.6 / 0.05 |
| Seeds | 42, 1337, 2024 | 42, 1337, 2024 |

The two tracks differ in effective batch size. The YOLO track was trained with batch 8 and Ultralytics' default nominal batch size `nbs` = 64, i.e. 8-step gradient accumulation (effective batch 64); Ultralytics also scales the weight decay by the same factor. The ResNet track used 2-step accumulation (effective batch 16). Appendix A states `nbs` = 64 explicitly so that a future change of the Ultralytics default cannot alter it.

In [ ]:
RUN_TRAINING = False     # retrain both detectors (CUDA GPU, D-Fire images, many hours)
RUN_SMOKE_TEST = False   # build the ResNet18+FPN+FCOS detector on the CPU and run one training step
TRAINING_DIR = OUTPUT_DIR / "training"
TRAINING_SEEDS = (42, 1337, 2024)

if RUN_TRAINING and DFIRE_IMAGE_DIR is None:
    raise RuntimeError("RUN_TRAINING needs the D-Fire images: set FIRE_DFIRE_IMAGES to the D-Fire root.")

### Appendix A. YOLO11n with Ultralytics

The hyperparameters are those of the locked configuration of the original project (`configs/models/ref_yolo11n.yaml` and `configs/training_defaults.yaml`). Mosaic, mixup and copy-paste are disabled because they build composite images whose content no longer matches any human annotation; a composite of background images is no longer a verified negative.

The training set is written in YOLO format from the frozen manifests (class 0 = fire, 1 = smoke), so both tracks see byte-identical splits. Verified negatives get an empty label file, which in YOLO format asserts that the image contains nothing; that is correct here because a person checked them. Detections are exported at confidence 0.05 and NMS IoU 0.6, with boxes rounded to 0.01 pixel and scores to five decimals, in the format of Section 2. Ultralytics' own `model.val()` is not used for any reported number: its AP implementation differs from the COCO convention in small ways, and both tracks must be scored by the same code.

In [ ]:
YOLO_HYPERPARAMETERS = {
    "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0,
    "fliplr": 0.5, "flipud": 0.0, "degrees": 0.0, "scale": 0.3, "translate": 0.1,
    "hsv_h": 0.015, "hsv_s": 0.5, "hsv_v": 0.4,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.001,   # final LR = lr0 * lrf = 1e-6
    "weight_decay": 0.0005, "warmup_epochs": 3.0, "cos_lr": True, "patience": 12, "amp": True,
}
# batch 8 with the nominal batch size nbs = 64 (the Ultralytics default, written out here):
# Ultralytics accumulates gradients over 64 / 8 = 8 steps, an effective batch of 64.
YOLO_TRAIN_ARGS = {"epochs": 60, "imgsz": 640, "batch": 8, "nbs": 64, "deterministic": True,
                   "plots": False, "val": True, "exist_ok": True, "verbose": True}
PREDICTION_CONFIDENCE_FLOOR = 0.05
PREDICTION_NMS_IOU = 0.60


def export_yolo_dataset(out_dir: Path, image_root: Path, manifests: dict[str, pd.DataFrame]) -> Path:
    """Write images (hard links where possible) and YOLO label files; return the data.yaml path."""
    import shutil

    class_index = {c: i for i, c in enumerate(CLASSES)}
    for split, df in manifests.items():
        img_dir = out_dir / "images" / split
        lbl_dir = out_dir / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        for row in df.itertuples(index=False):
            src = image_root / row.image_path
            if not src.is_file():
                raise FileNotFoundError(f"D-Fire image missing: {src}")
            stem = row.sample_id.replace("/", "__")
            dst = img_dir / f"{stem}{src.suffix}"
            if not dst.exists():
                try:
                    os.link(src, dst)
                except OSError:
                    shutil.copyfile(src, dst)
            lines = [f"{class_index[b['cls']]} {b['cx']:.6f} {b['cy']:.6f} {b['w']:.6f} {b['h']:.6f}"
                     for column in ("fire_boxes", "smoke_boxes") for b in parse_boxes(getattr(row, column))]
            (lbl_dir / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    data_yaml = out_dir / "data.yaml"
    data_yaml.write_text(
        f"path: {out_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(CLASSES)}\nnames:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)),
        encoding="utf-8")
    return data_yaml


def train_yolo(seed: int, data_yaml: Path) -> Path:
    from ultralytics import YOLO

    model = YOLO("yolo11n.pt")
    result = model.train(data=str(data_yaml), seed=seed, device=0, project=str(TRAINING_DIR / "yolo"),
                         name=f"seed{seed}", **YOLO_TRAIN_ARGS, **YOLO_HYPERPARAMETERS)
    return Path(result.save_dir) / "weights" / "best.pt"


def write_prediction_file(records: list[dict[str, Any]], out_path: Path, split: str, experiment_id: str,
                          seconds: float) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    meta = {"split": split, "domain": "dfire", "experiment_id": experiment_id, "n_samples": len(records),
            "conf_threshold_used": PREDICTION_CONFIDENCE_FLOOR, "inference_seconds": round(seconds, 3)}
    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write(json.dumps({"_meta": meta}) + "\n")
        for rec in records:
            fh.write(json.dumps(rec) + "\n")


def export_yolo_predictions(weights: Path, split: str, out_path: Path) -> None:
    """Detections of one model on one split, in manifest pixel coordinates."""
    from ultralytics import YOLO

    model = YOLO(str(weights))
    records, start = [], time.time()
    for row in MANIFESTS[split].itertuples(index=False):
        res = model.predict(str(DFIRE_IMAGE_DIR / row.image_path), conf=PREDICTION_CONFIDENCE_FLOOR,
                            iou=PREDICTION_NMS_IOU, imgsz=640, device=0, verbose=False)[0]
        # The curated copy capped the long side at 1,280 px; rescale if the image read here is larger.
        orig_h, orig_w = res.orig_shape
        sx, sy = int(row.width) / orig_w, int(row.height) / orig_h
        dets = []
        for box in res.boxes:
            x1, y1, x2, y2 = (float(v) for v in box.xyxy[0].tolist())
            idx = int(box.cls.item())
            dets.append({"bbox": [round(x1 * sx, 2), round(y1 * sy, 2), round(x2 * sx, 2), round(y2 * sy, 2)],
                         "score": round(float(box.conf.item()), 5), "class_idx": idx, "class": CLASSES[idx]})
        records.append({"sample_id": row.sample_id, "source": "dfire", "detections": dets})
    write_prediction_file(records, out_path, split, "E0", time.time() - start)


if RUN_TRAINING:
    yolo_yaml = export_yolo_dataset(TRAINING_DIR / "yolo_dataset", DFIRE_IMAGE_DIR, MANIFESTS)
    for seed in TRAINING_SEEDS:
        best = train_yolo(seed, yolo_yaml)
        for split in (SELECTION_SPLIT, TEST_SPLIT):
            export_yolo_predictions(best, split, TRAINING_DIR / "predictions" / f"yolo11n_seed{seed}"
                                    / f"{split}_dfire.jsonl")
    print(f"New YOLO11n predictions are in {TRAINING_DIR / 'predictions'}; point FIRE_PRED_DIR there "
          "(together with the other runs) and rerun the notebook to score them.")
else:
    print("Appendix A is switched off (RUN_TRAINING = False).")

### Appendix B. The ResNet18+FPN+FCOS detector

The in-house detector, ported from the original training pipeline. What is kept exactly:

* **architecture**: ImageNet-pretrained ResNet-18 trunk (outputs at strides 8, 16, 32), a top-down FPN with 128 channels, a shared two-layer FCOS tower with classification, box-regression (exp, scaled per level and multiplied by the stride) and centerness branches, and the image-level presence head (512 → 256 → 3) that the E0b configuration trained alongside the detector. The parameter count, 12,170,317, matches the 12.17 M of Tables 4 and 15;
* **targets and loss**: FCOS assignment with centre sampling (radius 1.5 strides), size ranges 0-64, 64-128 and above 128 pixels per level, smallest box wins; sigmoid focal loss (alpha 0.25, gamma 2) normalised by positives, GIoU box loss weighted by centerness, centerness BCE; weights 1 / 2 / 1; presence BCE on the two labels D-Fire defines (fire and smoke present, derived from the boxes), weight 0.5;
* **inference**: score = sqrt(class probability × centerness), top 1,000 candidates above 0.05, class-aware NMS at IoU 0.6, at most 300 detections, boxes mapped back through the letterbox to original pixels;
* **optimisation**: AdamW with separate learning rates for backbone (1e-4) and heads (1e-3), no weight decay on norms and biases, per-step linear warm-up over 3 epochs then cosine decay to 1e-6, batch 8 with 2-step gradient accumulation (effective batch 16), gradient clipping at 10, EMA of the weights (decay 0.9998 with warm-up), bf16 autocast, 60 epochs, model selection on validation mAP50-95 with early stopping after 12 epochs without improvement.

What is simplified relative to the original pipeline:

* the multi-source machinery of the original project (thermal input adapter, domain alignment, per-sample task masks for unlabelled sources, multi-source sampler) is left out; with D-Fire alone every sample has known boxes, so those masks were all ones in the paper's runs anyway;
* checkpoint resumption, run registry, GPU cool-down pauses and telemetry are left out;
* the backbone is built with `timm` (the original library and default ImageNet weights). If `timm` is not installed, the same ResNet-18 trunk is taken from `torchvision`; the architecture and parameter count are identical, but the pretrained weights are torchvision's, not timm's;
* the data loader is a compact rewrite of the original letterbox (pad value 114, ImageNet normalisation) and augmentation; its random stream is seeded but not identical to the original one.

Only the smoke test has been executed for this repository. The training loop follows the original settings line by line but has not been rerun end to end here.

In [ ]:
if RUN_TRAINING or RUN_SMOKE_TEST:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    INF = 1e8
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    PAD_VALUE = 114
    IMAGE_SIZE = 640
    FCOSHEAD_STRIDES = (8, 16, 32)   # ResNet-18 outputs at strides 8, 16 and 32
    FPN_CHANNELS = 128
    SCALE_RANGES = ((0.0, 64.0), (64.0, 128.0), (128.0, INF))
    CENTER_RADIUS = 1.5
    LOSS = {"lambda_det": 1.0, "lambda_cls": 0.5, "w_cls": 1.0, "w_box": 2.0, "w_ctr": 1.0,
            "focal_alpha": 0.25, "focal_gamma": 2.0}

    # ------------------------------------------------------------------ model
    class ConvNormAct(nn.Sequential):
        def __init__(self, cin: int, cout: int, k: int = 3, s: int = 1) -> None:
            super().__init__(nn.Conv2d(cin, cout, k, s, padding=k // 2, bias=False),
                             nn.BatchNorm2d(cout), nn.SiLU(inplace=True))

    class ResNet18Encoder(nn.Module):
        """ResNet-18 trunk returning the stride-8, 16 and 32 feature maps."""

        def __init__(self, pretrained: bool = True) -> None:
            super().__init__()
            try:
                import timm
                self.body = timm.create_model("resnet18", pretrained=pretrained, features_only=True,
                                              out_indices=(2, 3, 4))
                self.channels = list(self.body.feature_info.channels())
                self.strides = list(self.body.feature_info.reduction())
                self.source = "timm"
            except ImportError:
                import torchvision
                weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
                net = torchvision.models.resnet18(weights=weights)
                self.body = None
                self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1)
                self.layer2, self.layer3, self.layer4 = net.layer2, net.layer3, net.layer4
                self.channels, self.strides = [128, 256, 512], [8, 16, 32]
                self.source = "torchvision"

        def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
            if self.body is not None:
                return self.body(x)
            c3 = self.layer2(self.stem(x))
            c4 = self.layer3(c3)
            return [c3, c4, self.layer4(c4)]

    class FPN(nn.Module):
        """Top-down feature pyramid with lateral 1x1 projections and 3x3 smoothing."""

        def __init__(self, in_channels: list[int], out_channels: int = FPN_CHANNELS) -> None:
            super().__init__()
            self.lateral = nn.ModuleList(nn.Conv2d(c, out_channels, kernel_size=1) for c in in_channels)
            self.smooth = nn.ModuleList(ConvNormAct(out_channels, out_channels, 3) for _ in in_channels)
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_uniform_(m.weight, a=1)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def forward(self, feats: list[torch.Tensor]) -> list[torch.Tensor]:
            lat = [conv(f) for conv, f in zip(self.lateral, feats)]
            for i in range(len(lat) - 1, 0, -1):
                lat[i - 1] = lat[i - 1] + F.interpolate(lat[i], size=lat[i - 1].shape[-2:], mode="nearest")
            return [smooth(x) for smooth, x in zip(self.smooth, lat)]

    class Scale(nn.Module):
        """Learnable per-level multiplier on the regression output."""

        def __init__(self, init: float = 1.0) -> None:
            super().__init__()
            self.scale = nn.Parameter(torch.tensor(float(init)))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x * self.scale

    class FCOSHead(nn.Module):
        """Anchor-free head: one shared tower, then class, box (l, t, r, b) and centerness branches."""

        def __init__(self, channels: int = FPN_CHANNELS, num_classes: int = 2, tower_depth: int = 2,
                     strides: tuple[int, ...] = (8, 16, 32), prior_prob: float = 0.01) -> None:
            super().__init__()
            self.strides = strides
            self.tower = nn.Sequential(*[ConvNormAct(channels, channels, 3) for _ in range(tower_depth)])
            self.cls_pred = nn.Conv2d(channels, num_classes, 3, padding=1)
            self.reg_pred = nn.Conv2d(channels, 4, 3, padding=1)
            self.ctr_pred = nn.Conv2d(channels, 1, 3, padding=1)
            self.scales = nn.ModuleList(Scale(1.0) for _ in strides)
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.normal_(m.weight, std=0.01)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
            # Focal-loss prior: start every location at a low foreground probability so the huge
            # background population does not dominate the first iterations.
            nn.init.constant_(self.cls_pred.bias, -math.log((1 - prior_prob) / prior_prob))

        @staticmethod
        def points(feat: torch.Tensor, stride: int) -> torch.Tensor:
            h, w = feat.shape[-2:]
            ys = torch.arange(h, device=feat.device, dtype=torch.float32) * stride + stride / 2
            xs = torch.arange(w, device=feat.device, dtype=torch.float32) * stride + stride / 2
            yy, xx = torch.meshgrid(ys, xs, indexing="ij")
            return torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)

        def forward(self, feats: list[torch.Tensor]) -> dict[str, list[torch.Tensor]]:
            out: dict[str, list[torch.Tensor]] = {"cls": [], "ltrb": [], "ctr": [], "points": []}
            for i, f in enumerate(feats):
                t = self.tower(f)
                out["cls"].append(self.cls_pred(t))
                # exp() keeps distances positive; multiplying by the stride starts each level inside
                # the size band it is responsible for, which keeps the GIoU term trainable early on.
                out["ltrb"].append(torch.exp(self.scales[i](self.reg_pred(t))) * self.strides[i])
                out["ctr"].append(self.ctr_pred(t))
                out["points"].append(self.points(f, self.strides[i]))
            return out

    def flatten_levels(det: dict[str, list[torch.Tensor]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Concatenate all levels into (N, P, C), (N, P, 4) and (N, P, 1) tensors."""
        def _f(tensors: list[torch.Tensor], c: int) -> torch.Tensor:
            return torch.cat([t.permute(0, 2, 3, 1).reshape(t.shape[0], -1, c) for t in tensors], dim=1)
        return _f(det["cls"], det["cls"][0].shape[1]), _f(det["ltrb"], 4), _f(det["ctr"], 1)

    class PresenceHead(nn.Module):
        """Image-level logits for (wildfire, fire, smoke) present, from the pooled stride-32 features."""

        def __init__(self, in_channels: int = 512, hidden: int = 256, outputs: int = 3, dropout: float = 0.2) -> None:
            super().__init__()
            self.mlp = nn.Sequential(nn.Linear(in_channels, hidden), nn.SiLU(inplace=True),
                                     nn.Dropout(dropout), nn.Linear(hidden, outputs))
            for m in self.mlp.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    nn.init.zeros_(m.bias)

        def forward(self, feat: torch.Tensor) -> torch.Tensor:
            return self.mlp(feat.mean(dim=(2, 3)))

    class FireSmokeDetector(nn.Module):
        def __init__(self, pretrained: bool = True) -> None:
            super().__init__()
            self.encoder = ResNet18Encoder(pretrained)
            self.neck = FPN(self.encoder.channels, FPN_CHANNELS)
            self.det_head = FCOSHead(FPN_CHANNELS, strides=tuple(self.encoder.strides))
            self.cls_head = PresenceHead(self.encoder.channels[-1])

        def forward(self, x: torch.Tensor) -> tuple[dict[str, list[torch.Tensor]], torch.Tensor]:
            feats = self.encoder(x)
            return self.det_head(self.neck(feats)), self.cls_head(feats[-1])

        def param_groups(self, base_lr: float = 1e-3, backbone_lr: float = 1e-4,
                         weight_decay: float = 5e-4) -> list[dict[str, Any]]:
            """Backbone and heads at different learning rates; no decay on norms and biases."""
            groups: dict[str, dict[str, Any]] = {
                "backbone_decay": {"params": [], "lr": backbone_lr, "weight_decay": weight_decay},
                "backbone_no_decay": {"params": [], "lr": backbone_lr, "weight_decay": 0.0},
                "head_decay": {"params": [], "lr": base_lr, "weight_decay": weight_decay},
                "head_no_decay": {"params": [], "lr": base_lr, "weight_decay": 0.0},
            }
            for name, p in self.named_parameters():
                if not p.requires_grad:
                    continue
                where = "backbone" if name.startswith("encoder.") else "head"
                kind = "no_decay" if (p.ndim <= 1 or name.endswith(".bias")) else "decay"
                groups[f"{where}_{kind}"]["params"].append(p)
            return [{"name": k, **v} for k, v in groups.items() if v["params"]]

    # ------------------------------------------------------------------ targets and loss
    def compute_centerness(ltrb: torch.Tensor) -> torch.Tensor:
        lr, tb = ltrb[:, [0, 2]], ltrb[:, [1, 3]]
        return torch.sqrt((lr.min(dim=1).values.clamp(min=0) / lr.max(dim=1).values.clamp(min=1e-6))
                          * (tb.min(dim=1).values.clamp(min=0) / tb.max(dim=1).values.clamp(min=1e-6)))

    def assign_fcos_targets(points_per_level: list[torch.Tensor], gt: torch.Tensor | None,
                            strides: tuple[int, ...], num_classes: int = 2) -> dict[str, torch.Tensor]:
        """FCOS assignment for one image. `gt` is (G, 5): x1, y1, x2, y2, class index."""
        points = torch.cat(points_per_level, dim=0)
        n, device = points.shape[0], points.device
        ranges = torch.cat([pts.new_tensor(SCALE_RANGES[lvl]).expand(pts.shape[0], 2)
                            for lvl, pts in enumerate(points_per_level)])
        level_stride = torch.cat([pts.new_full((pts.shape[0],), strides[lvl])
                                  for lvl, pts in enumerate(points_per_level)])
        labels = torch.full((n,), num_classes, dtype=torch.long, device=device)
        if gt is None or gt.numel() == 0:   # verified negative: every location is background
            return {"labels": labels, "ltrb": torch.zeros((n, 4), device=device),
                    "centerness": torch.zeros(n, device=device),
                    "pos": torch.zeros(n, dtype=torch.bool, device=device)}
        gt = gt.to(device).float()
        x, y = points[:, 0:1], points[:, 1:2]
        x1, y1, x2, y2 = (gt[:, i].unsqueeze(0) for i in range(4))
        ltrb = torch.stack([x - x1, y - y1, x2 - x, y2 - y], dim=2)              # (P, G, 4)
        inside = ltrb.min(dim=2).values > 0
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        radius = level_stride.unsqueeze(1) * CENTER_RADIUS
        in_center = ((x > torch.maximum(cx - radius, x1)) & (x < torch.minimum(cx + radius, x2))
                     & (y > torch.maximum(cy - radius, y1)) & (y < torch.minimum(cy + radius, y2)))
        inside = inside & in_center
        max_side = ltrb.max(dim=2).values
        in_range = (max_side >= ranges[:, 0:1]) & (max_side <= ranges[:, 1:2])
        areas = ((x2 - x1) * (y2 - y1)).expand(n, gt.shape[0]).clone()
        areas[~(inside & in_range)] = INF
        min_area, min_idx = areas.min(dim=1)                                      # smallest box wins
        pos = min_area < INF
        labels[pos] = gt[min_idx[pos], 4].long()
        targets = torch.zeros((n, 4), device=device)
        targets[pos] = ltrb[torch.arange(n, device=device), min_idx][pos]
        centerness = torch.zeros(n, device=device)
        if pos.any():
            centerness[pos] = compute_centerness(targets[pos])
        return {"labels": labels, "ltrb": targets, "centerness": centerness, "pos": pos}

    def focal_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float, gamma: float) -> torch.Tensor:
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        return (alpha * targets + (1 - alpha) * (1 - targets)) * ce * ((1 - p_t) ** gamma)

    def giou_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
        """Generalised IoU loss on (l, t, r, b) distances from the same location."""
        pl, pt, pr, pb = pred.unbind(-1)
        tl, tt, tr, tb = target.unbind(-1)
        area_p = (pl + pr).clamp(min=0) * (pt + pb).clamp(min=0)
        area_t = (tl + tr).clamp(min=0) * (tt + tb).clamp(min=0)
        inter = ((torch.min(pl, tl) + torch.min(pr, tr)).clamp(min=0)
                 * (torch.min(pt, tt) + torch.min(pb, tb)).clamp(min=0))
        union = area_p + area_t - inter + eps
        enclose = ((torch.max(pl, tl) + torch.max(pr, tr)).clamp(min=0)
                   * (torch.max(pt, tt) + torch.max(pb, tb)).clamp(min=0)) + eps
        return 1.0 - (inter / union - (enclose - union) / enclose)

    def detection_loss(det: dict[str, list[torch.Tensor]], gt_boxes: list[torch.Tensor | None]) -> tuple[torch.Tensor, dict[str, float]]:
        """Per-image FCOS loss, shape (N,)."""
        cls_flat, box_flat, ctr_flat = flatten_levels(det)
        n, n_points, n_cls = cls_flat.shape
        points = [p.to(cls_flat.device) for p in det["points"]]
        per_image = cls_flat.new_zeros(n)
        stats = {"cls": 0.0, "box": 0.0, "ctr": 0.0, "n_pos": 0.0}
        for i in range(n):
            tgt = assign_fcos_targets(points, gt_boxes[i], FCOSHEAD_STRIDES, n_cls)
            pos = tgt["pos"]
            onehot = cls_flat.new_zeros((n_points, n_cls))
            if pos.any():
                onehot[pos, tgt["labels"][pos]] = 1.0
            n_pos = max(1, int(pos.sum()))   # normalise by positives, not by locations
            l_cls = focal_loss(cls_flat[i], onehot, LOSS["focal_alpha"], LOSS["focal_gamma"]).sum() / n_pos
            if pos.any():
                ctr_t = tgt["centerness"][pos]
                l_box = (giou_loss(box_flat[i][pos], tgt["ltrb"][pos]) * ctr_t).sum() / ctr_t.sum().clamp(min=1e-6)
                l_ctr = F.binary_cross_entropy_with_logits(ctr_flat[i][pos].squeeze(-1), ctr_t)
            else:
                l_box = l_ctr = cls_flat.new_zeros(())
            per_image[i] = LOSS["w_cls"] * l_cls + LOSS["w_box"] * l_box + LOSS["w_ctr"] * l_ctr
            for k, v in (("cls", l_cls), ("box", l_box), ("ctr", l_ctr)):
                stats[k] += float(v.detach()) / n
            stats["n_pos"] += int(pos.sum()) / n
        return per_image, stats

    def total_loss(det, presence_logits, gt_boxes, presence_targets, presence_masks):
        """lambda_det * detection + lambda_cls * presence. Every D-Fire image has known boxes, so the
        detection mask of the original objective is 1 for all of them; the presence term uses only the
        labels D-Fire defines (fire and smoke present), wildfire_present stays masked out."""
        per_image, stats = detection_loss(det, gt_boxes)
        elem = F.binary_cross_entropy_with_logits(presence_logits, presence_targets, reduction="none")
        per_sample = (elem * presence_masks).sum(1) / presence_masks.sum(1).clamp(min=1.0)
        labelled = (presence_masks.sum(1) > 0).float()
        cls_loss = (per_sample * labelled).sum() / labelled.sum().clamp(min=1.0)
        total = LOSS["lambda_det"] * per_image.mean() + LOSS["lambda_cls"] * cls_loss
        stats["presence"] = float(cls_loss.detach())
        return total, stats

    # ------------------------------------------------------------------ inference
    def batched_nms(boxes: torch.Tensor, scores: torch.Tensor, classes: torch.Tensor, iou: float) -> torch.Tensor:
        try:
            from torchvision.ops import batched_nms as tv_nms
            return tv_nms(boxes, scores, classes, iou)
        except ImportError:   # plain-torch fallback, class by class
            keep = []
            for c in classes.unique():
                idx = (classes == c).nonzero(as_tuple=True)[0]
                order = scores[idx].argsort(descending=True)
                while order.numel():
                    i = order[0]
                    keep.append(int(idx[i]))
                    rest = order[1:]
                    b = boxes[idx]
                    xx1 = torch.maximum(b[i, 0], b[rest, 0])
                    yy1 = torch.maximum(b[i, 1], b[rest, 1])
                    xx2 = torch.minimum(b[i, 2], b[rest, 2])
                    yy2 = torch.minimum(b[i, 3], b[rest, 3])
                    inter = (xx2 - xx1).clamp(min=0) * (yy2 - yy1).clamp(min=0)
                    area = lambda z: (z[..., 2] - z[..., 0]) * (z[..., 3] - z[..., 1])  # noqa: E731
                    order = rest[inter / (area(b[i]) + area(b[rest]) - inter + 1e-9) <= iou]
            kept = torch.tensor(keep, dtype=torch.long, device=boxes.device)
            return kept[scores[kept].argsort(descending=True)] if keep else kept

    @torch.no_grad()
    def decode_detections(det: dict[str, list[torch.Tensor]], metas: list[dict[str, float]],
                          conf: float = PREDICTION_CONFIDENCE_FLOOR, iou: float = PREDICTION_NMS_IOU,
                          max_detections: int = 300, topk_before_nms: int = 1000) -> list[list[dict[str, Any]]]:
        """Head outputs -> detections in original-image pixels, in the stored-prediction format."""
        cls_flat, box_flat, ctr_flat = flatten_levels(det)
        points = torch.cat([p.to(cls_flat.device) for p in det["points"]], dim=0)
        # FCOS score: geometric mean of class probability and centerness.
        scores_all = (cls_flat.float().sigmoid() * ctr_flat.float().sigmoid()).sqrt()
        out = []
        for i in range(cls_flat.shape[0]):
            flat = scores_all[i].reshape(-1)
            n_keep = min(topk_before_nms, int((flat > conf).sum()))
            if n_keep == 0:
                out.append([])
                continue
            top_scores, top_idx = flat.topk(n_keep)
            point_idx, class_idx = top_idx // cls_flat.shape[2], top_idx % cls_flat.shape[2]
            ltrb = box_flat[i][point_idx].float()
            pts = points[point_idx]
            boxes = torch.stack([pts[:, 0] - ltrb[:, 0], pts[:, 1] - ltrb[:, 1],
                                 pts[:, 0] + ltrb[:, 2], pts[:, 1] + ltrb[:, 3]], dim=1)
            kept = batched_nms(boxes, top_scores, class_idx, iou)[:max_detections]
            m = metas[i]
            b = boxes[kept].clone()
            b[:, [0, 2]] = ((b[:, [0, 2]] - m["pad_x"]) / m["scale"]).clamp(0, m["orig_w"])
            b[:, [1, 3]] = ((b[:, [1, 3]] - m["pad_y"]) / m["scale"]).clamp(0, m["orig_h"])
            out.append([{"bbox": [round(float(v), 2) for v in bb], "score": round(float(s), 5),
                         "class_idx": int(c), "class": CLASSES[int(c)]}
                        for bb, s, c in zip(b, top_scores[kept], class_idx[kept])])
        return out

    print("Appendix B definitions loaded.")
else:
    print("Appendix B is switched off (RUN_SMOKE_TEST = False, RUN_TRAINING = False).")

In [ ]:
if RUN_SMOKE_TEST:
    torch.manual_seed(0)
    t_smoke = time.time()
    model = FireSmokeDetector(pretrained=False)   # random weights: no download needed
    n_params = sum(p.numel() for p in model.parameters())
    print(f"backbone from {model.encoder.source}; {n_params:,} parameters ({n_params / 1e6:.4f} M)")
    assert abs(n_params / 1e6 - 12.1703) < 5e-5, "parameter count differs from the recorded 12.1703 M"

    # Dummy batch: image 0 holds one fire and one smoke box (letterboxed pixels), image 1 is a
    # verified negative with no boxes.
    images = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    gt_boxes = [torch.tensor([[100.0, 120.0, 180.0, 210.0, 0.0], [300.0, 60.0, 600.0, 420.0, 1.0]]),
                torch.zeros((0, 5))]
    presence_targets = torch.tensor([[0.0, 1.0, 1.0], [0.0, 0.0, 0.0]])
    presence_masks = torch.tensor([[0.0, 1.0, 1.0], [0.0, 1.0, 1.0]])   # wildfire_present is unknown in D-Fire

    optimizer = torch.optim.AdamW(model.param_groups(1e-3, 1e-4, 5e-4), lr=1e-3, betas=(0.9, 0.999))
    model.train()
    det, logits = model(images)
    loss, parts = total_loss(det, logits, gt_boxes, presence_targets, presence_masks)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    assert torch.isfinite(loss) and torch.isfinite(grad_norm)
    assert all(p.grad is not None for p in model.parameters() if p.requires_grad), "a parameter received no gradient"
    print(f"loss {float(loss.detach()):.4f} (focal {parts['cls']:.4f}, GIoU {parts['box']:.4f}, "
          f"centerness {parts['ctr']:.4f}, presence {parts['presence']:.4f}); "
          f"{parts['n_pos']:.1f} positive locations per image; gradient norm {float(grad_norm):.3f}")

    model.eval()
    with torch.no_grad():
        det, _ = model(images)
    identity = {"scale": 1.0, "pad_x": 0.0, "pad_y": 0.0, "orig_w": IMAGE_SIZE, "orig_h": IMAGE_SIZE}
    decoded = decode_detections(det, [identity, identity])
    assert all(len(d) <= 300 for d in decoded)
    assert all(0 <= v <= IMAGE_SIZE for d in decoded for x in d for v in x["bbox"])
    print(f"decoded {[len(d) for d in decoded]} detections per image after NMS "
          f"(untrained weights, so the boxes are meaningless)")
    print(f"smoke test passed in {time.time() - t_smoke:.1f} s")
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

#### Training loop and prediction export

Guarded by `RUN_TRAINING`. It trains the three seeds of the in-house track on the full training split and writes validation and test predictions in the stored format. The two ablation arms are the same call with the training manifest of Table 5 (`ABLATION_MANIFESTS` from Section 4).

In [ ]:
if RUN_TRAINING:
    import copy

    import cv2

    class DFireDataset(torch.utils.data.Dataset):
        """Manifest rows -> letterboxed, normalised CHW tensors plus boxes in letterboxed pixels."""

        def __init__(self, rows: list[dict[str, Any]], train: bool, seed: int = 0) -> None:
            self.rows, self.train, self.seed, self.epoch = rows, train, seed, 0
            self.mean = np.array(IMAGENET_MEAN, dtype=np.float32).reshape(3, 1, 1)
            self.std = np.array(IMAGENET_STD, dtype=np.float32).reshape(3, 1, 1)

        def __len__(self) -> int:
            return len(self.rows)

        def __getitem__(self, i: int):
            row = self.rows[i]
            img = cv2.cvtColor(cv2.imread(str(DFIRE_IMAGE_DIR / row["image_path"])), cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            cls_index = {c: k for k, c in enumerate(CLASSES)}
            boxes = np.array([[(b["cx"] - b["w"] / 2) * w, (b["cy"] - b["h"] / 2) * h,
                               (b["cx"] + b["w"] / 2) * w, (b["cy"] + b["h"] / 2) * h, cls_index[b["cls"]]]
                              for col in ("fire_boxes", "smoke_boxes") for b in parse_boxes(row[col])],
                             dtype=np.float32).reshape(-1, 5)
            presence = np.array([0.0, float((boxes[:, 4] == 0).any()), float((boxes[:, 4] == 1).any())],
                                dtype=np.float32)
            if self.train:
                rng = random.Random(self.seed * 1_000_003 + self.epoch * 100_003 + i)
                img, boxes = self._augment(img, boxes, rng)
            scale = min(IMAGE_SIZE / w, IMAGE_SIZE / h)
            nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
            resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
            canvas = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), PAD_VALUE, dtype=np.uint8)
            pad_x, pad_y = (IMAGE_SIZE - nw) / 2, (IMAGE_SIZE - nh) / 2
            canvas[int(round(pad_y)):int(round(pad_y)) + nh, int(round(pad_x)):int(round(pad_x)) + nw] = resized
            if len(boxes):
                boxes[:, [0, 2]] = boxes[:, [0, 2]] * scale + pad_x
                boxes[:, [1, 3]] = boxes[:, [1, 3]] * scale + pad_y
                boxes = self._clip(boxes)
            chw = (canvas.transpose(2, 0, 1).astype(np.float32) / 255.0 - self.mean) / self.std
            meta = {"scale": scale, "pad_x": pad_x, "pad_y": pad_y, "orig_w": w, "orig_h": h,
                    "sample_id": row["sample_id"]}
            return torch.from_numpy(np.ascontiguousarray(chw)), torch.from_numpy(boxes), torch.from_numpy(presence), meta

        @staticmethod
        def _clip(boxes: np.ndarray, min_side: float = 2.0, min_visibility: float = 0.25) -> np.ndarray:
            before = np.maximum(boxes[:, 2] - boxes[:, 0], 0) * np.maximum(boxes[:, 3] - boxes[:, 1], 0)
            out = boxes.copy()
            out[:, [0, 2]] = out[:, [0, 2]].clip(0, IMAGE_SIZE)
            out[:, [1, 3]] = out[:, [1, 3]].clip(0, IMAGE_SIZE)
            wd, ht = out[:, 2] - out[:, 0], out[:, 3] - out[:, 1]
            vis = np.where(before > 0, np.maximum(wd, 0) * np.maximum(ht, 0) / np.maximum(before, 1e-9), 0.0)
            return out[(wd >= min_side) & (ht >= min_side) & (vis >= min_visibility)]

        @staticmethod
        def _augment(img: np.ndarray, boxes: np.ndarray, rng: random.Random):
            h, w = img.shape[:2]
            boxes = boxes.copy()
            if rng.random() < 0.5:                                    # horizontal flip
                img = np.ascontiguousarray(img[:, ::-1])
                if len(boxes):
                    x1 = boxes[:, 0].copy()
                    boxes[:, 0], boxes[:, 2] = w - boxes[:, 2], w - x1
            s = 1.0 + rng.uniform(-0.3, 0.3)                          # scale and translate
            tx, ty = rng.uniform(-0.1, 0.1) * w, rng.uniform(-0.1, 0.1) * h
            cx, cy = w / 2, h / 2
            M = np.array([[s, 0, tx + cx - s * cx], [0, s, ty + cy - s * cy]], dtype=np.float32)
            img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                                 borderValue=(PAD_VALUE,) * 3)
            if len(boxes):
                boxes[:, [0, 2]] = boxes[:, [0, 2]] * s + M[0, 2]
                boxes[:, [1, 3]] = boxes[:, [1, 3]] * s + M[1, 2]
            gains = np.array([rng.uniform(-0.015, 0.015) * 180, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4)])
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[..., 0] = (hsv[..., 0] + gains[0]) % 180
            hsv[..., 1] = np.clip(hsv[..., 1] * (1 + gains[1]), 0, 255)
            hsv[..., 2] = np.clip(hsv[..., 2] * (1 + gains[2]), 0, 255)
            img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
            if rng.random() < 0.01:
                img = cv2.GaussianBlur(img, (5, 5), sigmaX=rng.uniform(0.1, 1.5))
            return img, boxes

    def collate(batch):
        images, boxes, presence, metas = zip(*batch)
        return torch.stack(images), list(boxes), torch.stack(presence), list(metas)

    class WarmupCosine(torch.optim.lr_scheduler.LRScheduler):
        """Per-step linear warm-up, then cosine decay to min_lr, keeping each group's base LR."""

        def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float = 1e-6) -> None:
            self.warmup_steps, self.total_steps, self.min_lr = max(0, warmup_steps), max(1, total_steps), min_lr
            super().__init__(optimizer)

        def get_lr(self) -> list[float]:
            step = self.last_epoch
            out = []
            for base in self.base_lrs:
                if self.warmup_steps and step < self.warmup_steps:
                    out.append(base * (step + 1) / self.warmup_steps)
                else:
                    progress = min(1.0, max(0.0, (step - self.warmup_steps)
                                            / max(1, self.total_steps - self.warmup_steps)))
                    out.append(self.min_lr + (base - self.min_lr) * 0.5 * (1 + math.cos(math.pi * progress)))
            return out

    class ModelEMA:
        """Exponential moving average of the weights; the EMA copy is what gets evaluated."""

        def __init__(self, model: nn.Module, decay: float = 0.9998, warmup_steps: int = 2000) -> None:
            self.ema = copy.deepcopy(model).eval()
            for p in self.ema.parameters():
                p.requires_grad_(False)
            self.decay, self.warmup_steps, self.updates = decay, warmup_steps, 0

        @torch.no_grad()
        def update(self, model: nn.Module) -> None:
            self.updates += 1
            d = self.decay * (1 - math.exp(-self.updates / self.warmup_steps))
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
                else:
                    v.copy_(msd[k])

    def predict_rows(model: nn.Module, rows: list[dict[str, Any]], device) -> list[dict[str, Any]]:
        """Stored-prediction records, rescaled to the manifest image size."""
        loader = torch.utils.data.DataLoader(DFireDataset(rows, train=False), batch_size=8, shuffle=False,
                                             num_workers=4, collate_fn=collate)
        width = {r["sample_id"]: (int(r["width"]), int(r["height"])) for r in rows}
        records = []
        model.eval()
        for images, _, _, metas in loader:
            with torch.no_grad(), torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                det, _ = model(images.to(device).to(memory_format=torch.channels_last))
            for meta, dets in zip(metas, decode_detections(det, metas)):
                mw, mh = width[meta["sample_id"]]
                sx, sy = mw / meta["orig_w"], mh / meta["orig_h"]
                for d in dets:
                    x1, y1, x2, y2 = d["bbox"]
                    d["bbox"] = [round(x1 * sx, 2), round(y1 * sy, 2), round(x2 * sx, 2), round(y2 * sy, 2)]
                records.append({"sample_id": meta["sample_id"], "source": "dfire", "detections": dets})
        return records

    def records_to_predictions(records: list[dict[str, Any]], gt: GroundTruth, split: str) -> Predictions:
        rows = [(gt.index[r["sample_id"]], CLASSES.index(d["class"]), d["score"], d["bbox"])
                for r in records for d in r["detections"]]
        img = np.array([r[0] for r in rows], dtype=np.int32)
        order = np.argsort(img, kind="stable")
        return Predictions("training", split, Path("."), {}, len(records), img[order],
                           np.array([r[1] for r in rows], dtype=np.int8)[order],
                           np.array([r[2] for r in rows], dtype=np.float64)[order],
                           np.array([r[3] for r in rows], dtype=np.float32).reshape(-1, 4)[order])

    def train_resnet_detector(seed: int, train_rows: list[dict[str, Any]], out_dir: Path, epochs: int = 60,
                              batch_size: int = 8, grad_accum: int = 2) -> Path:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        device = torch.device("cuda")
        model = FireSmokeDetector(pretrained=True).to(device).to(memory_format=torch.channels_last)
        optimizer = torch.optim.AdamW(model.param_groups(1e-3, 1e-4, 5e-4), lr=1e-3, betas=(0.9, 0.999))
        dataset = DFireDataset(train_rows, train=True, seed=seed)
        loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True,
                                             num_workers=6, pin_memory=True, collate_fn=collate,
                                             generator=torch.Generator().manual_seed(seed))
        steps_per_epoch = max(1, len(loader) // grad_accum)
        scheduler = WarmupCosine(optimizer, int(3.0 * steps_per_epoch), steps_per_epoch * epochs, 1e-6)
        ema = ModelEMA(model, 0.9998)
        val_rows = manifest_records(MANIFESTS[SELECTION_SPLIT])
        best, best_epoch, stale = -1.0, -1, 0
        out_dir.mkdir(parents=True, exist_ok=True)
        for epoch in range(epochs):
            dataset.epoch = epoch
            model.train()
            optimizer.zero_grad(set_to_none=True)
            for i, (images, boxes, presence, _) in enumerate(loader):
                images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    det, logits = model(images)
                    loss, _ = total_loss(det, logits.float(), [b.to(device) for b in boxes],
                                         presence.to(device), torch.tensor([[0.0, 1.0, 1.0]], device=device)
                                         .expand(len(boxes), 3))
                (loss / grad_accum).backward()
                if (i + 1) % grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                    ema.update(model)
            # Model selection on validation mAP50-95 of the EMA weights, scored by Section 3's code.
            pred = records_to_predictions(predict_rows(ema.ema, val_rows, device), GT[SELECTION_SPLIT],
                                          SELECTION_SPLIT)
            res = evaluate_detection(GT[SELECTION_SPLIT], match_detections(GT[SELECTION_SPLIT], pred, IOU_RANGE),
                                     image_max_score(GT[SELECTION_SPLIT], pred), LITERATURE_THRESHOLD)
            score = res["mean_ap50_95"] or 0.0
            print(f"seed {seed} epoch {epoch + 1}/{epochs}: val mAP50-95 {score:.5f}")
            if score > best + 1e-4:
                best, best_epoch, stale = score, epoch, 0
                torch.save(ema.ema.state_dict(), out_dir / "best.pt")
            else:
                stale += 1
                if stale >= 12:
                    print(f"early stop: no improvement since epoch {best_epoch + 1}")
                    break
        return out_dir / "best.pt"

    for seed in TRAINING_SEEDS:
        run_dir = TRAINING_DIR / "resnet18fpn" / f"seed{seed}"
        weights = train_resnet_detector(seed, manifest_records(MANIFESTS["train"]), run_dir)
        model = FireSmokeDetector(pretrained=False).to("cuda").to(memory_format=torch.channels_last)
        model.load_state_dict(torch.load(weights, map_location="cuda"))
        for split in (SELECTION_SPLIT, TEST_SPLIT):
            start = time.time()
            recs = predict_rows(model, manifest_records(MANIFESTS[split]), torch.device("cuda"))
            write_prediction_file(recs, TRAINING_DIR / "predictions" / f"resnet18fpn_seed{seed}"
                                  / f"{split}_dfire.jsonl", split, "E0b", time.time() - start)
    print(f"New ResNet18+FPN+FCOS predictions are in {TRAINING_DIR / 'predictions'}.")
else:
    print("Training loop skipped (RUN_TRAINING = False).")